# WIMB voice bot on Kaggle's free GPU

A voice bot for Indian calls (Hindi, Marathi, English and mixes): it talks for any **task** written as a small JSON
file (ask a shop about stock, book a clinic visit, confirm an order, or your own).

Each turn: **ears** (speech to text) → **brain** (rules + a small model read what was said, checked by type) →
the conversation decides what to say (from the task, never made up) → **mouth** (text to speech).

**MODE = "eval"** measures every part and picks the best free one. **MODE = "live"** opens a web page to talk to
the bot (the link is sent to the GitHub job; it needs the password).

Built by `voicebot/build_voicebot.py`; run it from GitHub: **Actions → Voice bot → Run workflow**.

In [ ]:
#@title 1. Settings: mode, models, live page
import itertools, json, os, re, shutil, subprocess, sys, time, urllib.error, urllib.parse, urllib.request

MODE = "eval"  # "eval": measure everything; "live": the web page
BRAINS = ["gemma3:4b", "qwen3:4b-instruct-2507-q4_K_M"]  # eval: all; live: the first that works (eval 3: both 48/48)
ASR = ["faster-whisper", "transformers"]  # speech to text ("indic-conformer" is tried too with an HF token)
TTS = ["kokoro", "parler", "mms"]  # text to speech, in order of preference (parler needs an HF token)
BENCH_ROWS = 60  # real speech per language for the speech-to-text test
LIVE_MINUTES = 120  # live: how long the page stays open
LIVE_PASSWORD = ""  # live: the page's password (user name: wimb)
NOTIFY_TOPIC = ""  # live: the page's link is posted to ntfy.sh/<this> for the GitHub job to print
WORKING_DIR = "/kaggle/working/voicebot"  # everything is saved here (the Output tab)

In [ ]:
#@title 2. Helpers (no need to read: the voicebot package, written to disk)
PACKAGE_FILES = {'__init__.py': '', 'numerals.py': '"""Spoken numbers, prices, times, days and phone numbers in Hindi, Marathi and English, written the way speech to\ntext writes them (Devanagari or Latin letters, words or digits), turned into values; and values back into words for\nthe voice.\n\n    find_numbers("ढाई सौ का है")                    -> [Span(value=250, ...)]\n    money("सोलह ए वाला है, दो सौ बीस रुपये")          -> 220      (16 is skipped: "ए" (amp) follows it)\n    money("rate two thirty hai")                     -> 230      (shop talk: "two thirty" is 230)\n    time_of_day("शाम साढ़े चार बजे")                  -> "16:30"\n    day("उद्या येतो")                                 -> "tomorrow"\n    phone("नौ आठ सात छह पाँच, चार तीन दो एक शून्य")   -> "9876543210"\n    to_words(220, "hi") -> "दो सौ बीस";  money_words(220, "mr") -> "दोनशे वीस रुपये";  time_words("16:30", "hi")\n\nFractions people say: डेढ़ 1.5, ढाई 2.5, सवा +¼, साढ़े +½, पौने −¼ (Marathi दीड, अडीच, सव्वा, साडे, पावणे), so\n"साढ़े तीन सौ" is 350 and "पौने दो हज़ार" is 1,750. No imports beyond the standard library.\n"""\n\nimport re\nimport unicodedata\nfrom dataclasses import dataclass\n\nHI_ONES = [  # 0 to 99\n    "शून्य", "एक", "दो", "तीन", "चार", "पाँच", "छह", "सात", "आठ", "नौ", "दस", "ग्यारह", "बारह", "तेरह", "चौदह",\n    "पंद्रह", "सोलह", "सत्रह", "अठारह", "उन्नीस", "बीस", "इक्कीस", "बाईस", "तेईस", "चौबीस", "पच्चीस", "छब्बीस",\n    "सत्ताईस", "अट्ठाईस", "उनतीस", "तीस", "इकतीस", "बत्तीस", "तैंतीस", "चौंतीस", "पैंतीस", "छत्तीस", "सैंतीस",\n    "अड़तीस", "उनतालीस", "चालीस", "इकतालीस", "बयालीस", "तैंतालीस", "चवालीस", "पैंतालीस", "छियालीस", "सैंतालीस",\n    "अड़तालीस", "उनचास", "पचास", "इक्यावन", "बावन", "तिरपन", "चौवन", "पचपन", "छप्पन", "सत्तावन", "अट्ठावन", "उनसठ",\n    "साठ", "इकसठ", "बासठ", "तिरसठ", "चौंसठ", "पैंसठ", "छियासठ", "सड़सठ", "अड़सठ", "उनहत्तर", "सत्तर", "इकहत्तर",\n    "बहत्तर", "तिहत्तर", "चौहत्तर", "पचहत्तर", "छिहत्तर", "सतहत्तर", "अठहत्तर", "उनासी", "अस्सी", "इक्यासी", "बयासी",\n    "तिरासी", "चौरासी", "पचासी", "छियासी", "सत्तासी", "अट्ठासी", "नवासी", "नब्बे", "इक्यानवे", "बानवे", "तिरानवे",\n    "चौरानवे", "पचानवे", "छियानवे", "सत्तानवे", "अट्ठानवे", "निन्यानवे",\n]\nHI_MORE = {  # other spellings speech to text writes\n    "छः": 6, "छै": 6, "छे": 6, "सतरह": 17, "अट्ठारह": 18, "उन्नीस": 19, "उन्निस": 19, "इकीस": 21, "बाइस": 22,\n    "तेइस": 23, "चौबिस": 24, "पचीस": 25, "पच्चिस": 25, "छबीस": 26, "सताईस": 27, "अठाईस": 28, "अट्ठाइस": 28,\n    "चौतीस": 34, "पैतीस": 35, "सैतीस": 37, "अडतीस": 38, "चालिस": 40, "बियालीस": 42, "चौवालीस": 44, "पैतालीस": 45,\n    "पचास": 50, "इक्कावन": 51, "तिरेपन": 53, "चौअन": 54, "छप्पन": 56, "सत्तावन": 57, "अठावन": 58, "चौसठ": 64,\n    "पैसठ": 65, "अड़सठ": 68, "उन्हत्तर": 69, "पिचहत्तर": 75, "पचत्तर": 75, "उन्यासी": 79, "अस्सि": 80, "नब्बें": 90,\n    "पचानबे": 95, "निन्यानबे": 99, "निनानवे": 99,\n}\nMR_ONES = [  # 0 to 100\n    "शून्य", "एक", "दोन", "तीन", "चार", "पाच", "सहा", "सात", "आठ", "नऊ", "दहा", "अकरा", "बारा", "तेरा", "चौदा",\n    "पंधरा", "सोळा", "सतरा", "अठरा", "एकोणीस", "वीस", "एकवीस", "बावीस", "तेवीस", "चोवीस", "पंचवीस", "सव्वीस",\n    "सत्तावीस", "अठ्ठावीस", "एकोणतीस", "तीस", "एकतीस", "बत्तीस", "तेहेतीस", "चौतीस", "पस्तीस", "छत्तीस", "सदतीस",\n    "अडतीस", "एकोणचाळीस", "चाळीस", "एक्केचाळीस", "बेचाळीस", "त्रेचाळीस", "चव्वेचाळीस", "पंचेचाळीस", "सेहेचाळीस",\n    "सत्तेचाळीस", "अठ्ठेचाळीस", "एकोणपन्नास", "पन्नास", "एक्कावन्न", "बावन्न", "त्रेपन्न", "चोपन्न", "पंचावन्न",\n    "छप्पन्न", "सत्तावन्न", "अठ्ठावन्न", "एकोणसाठ", "साठ", "एकसष्ठ", "बासष्ठ", "त्रेसष्ठ", "चौसष्ठ", "पासष्ठ",\n    "सहासष्ठ", "सदुसष्ठ", "अडुसष्ठ", "एकोणसत्तर", "सत्तर", "एक्काहत्तर", "बाहत्तर", "त्र्याहत्तर", "चौऱ्याहत्तर",\n    "पंचाहत्तर", "शहात्तर", "सत्याहत्तर", "अठ्ठ्याहत्तर", "एकोणऐंशी", "ऐंशी", "एक्क्याऐंशी", "ब्याऐंशी", "त्र्याऐंशी",\n    "चौऱ्याऐंशी", "पंच्याऐंशी", "शहाऐंशी", "सत्त्याऐंशी", "अठ्ठ्याऐंशी", "एकोणनव्वद", "नव्वद", "एक्क्याण्णव",\n    "ब्याण्णव", "त्र्याण्णव", "चौऱ्याण्णव", "पंच्याण्णव", "शहाण्णव", "सत्त्याण्णव", "अठ्ठ्याण्णव", "नव्व्याण्णव", "शंभर",\n]\nMR_MORE = {"नउ": 9, "तेहतीस": 33, "पंच्याहत्तर": 75, "छप्पन": 56, "एक्याऐंशी": 81}\nEN_ONES = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten", "eleven", "twelve",\n           "thirteen", "fourteen", "fifteen", "sixteen", "seventeen", "eighteen", "nineteen"]\nEN_TENS = {"twenty": 20, "thirty": 30, "forty": 40, "fourty": 40, "fifty": 50, "sixty": 60, "seventy": 70,\n           "eighty": 80, "ninety": 90}\nEN_DEVANAGARI = {  # English number words as Hindi speech to text writes them\n    "ज़ीरो": 0, "वन": 1, "टू": 2, "थ्री": 3, "फोर": 4, "फ़ोर": 4, "फाइव": 5, "फ़ाइव": 5, "सिक्स": 6, "सेवन": 7, "एट": 8,\n    "नाइन": 9, "टेन": 10, "इलेवन": 11, "ट्वेल्व": 12, "थर्टीन": 13, "फोर्टीन": 14, "फिफ्टीन": 15, "सिक्सटीन": 16,\n    "सेवेंटीन": 17, "एटीन": 18, "नाइनटीन": 19, "ट्वेंटी": 20, "थर्टी": 30, "फोर्टी": 40, "फ़ोर्टी": 40, "फिफ्टी": 50,\n    "फ़िफ़्टी": 50, "सिक्सटी": 60, "सेवेंटी": 70, "एटी": 80, "नाइंटी": 90, "नाइनटी": 90,\n}\nLATIN_HI = {  # Hindi number words in Latin letters (Hinglish); compared by latin_key(), so spellings may vary\n    0: "shunya", 1: "ek", 2: "do", 3: "teen tin", 4: "char chaar", 5: "paanch panch paach", 6: "chhe chhah che chah",\n    7: "saat", 8: "aath", 9: "nau", 10: "das dus", 11: "gyarah gyara", 12: "barah bara", 13: "terah", 14: "chaudah",\n    15: "pandrah", 16: "solah sola", 17: "satrah", 18: "atharah attharah", 19: "unnis unees", 20: "bees bis",\n    21: "ikkis", 22: "bais baees", 23: "teis teees", 24: "chaubis chobis", 25: "pachchis pachis pacchis",\n    26: "chhabbis chabbis", 27: "sattais", 28: "atthais athais", 29: "untis", 30: "tees", 31: "ikattis iktis",\n    32: "battis", 33: "taintis tetis", 34: "chautis chontis", 35: "paintis pentis", 36: "chhattis chattis",\n    37: "saintis sentis", 38: "adtis artis", 39: "untalis", 40: "chalis", 41: "iktalis", 42: "bayalis byalis",\n    43: "taintalis tetalis", 44: "chavalis chawalis", 45: "paintalis pentalis", 46: "chhiyalis chiyalis",\n    47: "saintalis sentalis", 48: "adtalis artalis", 49: "unchas", 50: "pachas", 51: "ikyavan ekyavan",\n    52: "bavan", 53: "tirpan trepan", 54: "chauvan chovan", 55: "pachpan", 56: "chhappan chappan", 57: "sattavan",\n    58: "atthavan athavan", 59: "unsath", 60: "saath", 61: "iksath", 62: "basath", 63: "tirsath", 64: "chausath chosath",\n    65: "painsath pensath", 66: "chhiyasath", 67: "sadsath", 68: "adsath", 69: "unhattar", 70: "sattar",\n    71: "ikhattar", 72: "bahattar", 73: "tihattar", 74: "chauhattar", 75: "pachhattar pachattar", 76: "chhihattar",\n    77: "sathattar satattar", 78: "athhattar athattar", 79: "unasi", 80: "assi", 81: "ikyasi", 82: "bayasi",\n    83: "tirasi", 84: "chaurasi", 85: "pachasi", 86: "chhiyasi", 87: "sattasi", 88: "atthasi athasi",\n    89: "navasi nawasi", 90: "nabbe", 91: "ikyanve", 92: "banve", 93: "tiranve", 94: "chauranve", 95: "pachanve",\n    96: "chhiyanve", 97: "sattanve", 98: "atthanve", 99: "ninyanve ninanve",\n}\nMULTIPLIERS = {  # word -> value\n    "सौ": 100, "सैकड़ा": 100, "हज़ार": 1000, "हजार": 1000, "लाख": 100_000, "करोड़": 10_000_000, "करोड": 10_000_000,\n    "कोटी": 10_000_000, "हंड्रेड": 100, "थाउजेंड": 1000, "थाउज़ेंड": 1000, "थाउसेंड": 1000, "hundred": 100,\n    "thousand": 1000, "lakh": 100_000, "lac": 100_000, "crore": 10_000_000, "sau": 100, "so": None,\n    "hazaar": 1000, "hazar": 1000, "hajar": 1000, "hajaar": 1000, "karod": 10_000_000, "k": 1000,\n}\nMULTIPLIERS = {k: v for k, v in MULTIPLIERS.items() if v}  # "so" is English too: not a multiplier\nFRACTIONS = {  # a whole value said in one word\n    "डेढ़": 1.5, "डेढ": 1.5, "डेड़": 1.5, "ढाई": 2.5, "अढ़ाई": 2.5, "आधा": 0.5, "आधी": 0.5, "दीड": 1.5, "अडीच": 2.5,\n    "अर्धा": 0.5, "dedh": 1.5, "derh": 1.5, "dhed": 1.5, "dhai": 2.5, "adhai": 2.5, "aadha": 0.5, "adha": 0.5,\n    "deed": 1.5, "adich": 2.5,\n}\nPREFIXES = {  # added to the next value: साढ़े तीन = 3.5\n    "सवा": 0.25, "साढ़े": 0.5, "साढे": 0.5, "साड़े": 0.5, "साडे": 0.5, "पौने": -0.25, "पौन": -0.25, "सव्वा": 0.25,\n    "पावणे": -0.25, "पाउणे": -0.25, "sava": 0.25, "sawa": 0.25, "sadhe": 0.5, "saadhe": 0.5, "sade": 0.5,\n    "saade": 0.5, "sarhe": 0.5, "paune": -0.25, "pone": -0.25, "paun": -0.25,\n}\nMONEY_AFTER = {"रुपये", "रुपए", "रूपये", "रूपए", "रुपया", "रुपय", "रुपयांना", "रुपयाला", "रुपयात", "रु", "रू", "₹",\n               "rs", "rupees", "rupee", "rupaye", "rupay", "rupiya", "rupya", "inr", "/-", "का", "की", "के", "में",\n               "ला", "ना", "मध्ये", "ka", "ki", "ke", "mein", "me", "padega", "पड़ेगा", "पडेगा", "पड़ेंगे", "होगा",\n               "होंगे", "lagega", "लगेगा", "लगेंगे", "लागतील", "पडेल", "पडतील"}\nMONEY_BEFORE = {"₹", "rs", "inr", "रु", "दाम", "रेट", "भाव", "कीमत", "क़ीमत", "किंमत", "दर", "price", "rate", "daam",\n                "bhav", "keemat", "kimat", "cost", "mrp", "एमआरपी", "प्राइस", "चार्ज", "charge", "फीस", "fees", "fee",\n                "बिल", "bill", "total", "टोटल"}\nWEAK_AFTER = {"का", "की", "के", "में", "ला", "ना", "मध्ये", "ka", "ki", "ke", "mein", "me"}  # "ढाई सौ का"\nUNITS = {  # a number right before one of these is a size or a count, not a price\n    "ए", "a", "amp", "amps", "ampere", "एम्प", "एम्पियर", "वाट", "watt", "watts", "w", "किलो", "kg", "kilo", "ग्राम",\n    "gram", "grams", "g", "लीटर", "litre", "liter", "ml", "एमएम", "mm", "इंच", "inch", "फुट", "feet", "foot", "मीटर",\n    "meter", "metre", "नंबर", "number", "no", "साइज", "साइज़", "size", "साल", "वर्ष", "years", "year", "बजे", "बजकर",\n    "baje", "वाजता", "वाजून", "दिन", "दिवस", "din", "days", "day", "घंटे", "घंटा", "तास", "ghante", "hours", "मिनट",\n    "मिनिट", "minute", "minutes", "तारीख", "तारखेला", "%", "परसेंट", "percent", "पीस", "piece", "pieces", "pcs",\n    "जीबी", "gb", "एमबी", "mb", "हफ्ते", "हफ़्ते", "आठवडे", "महीने", "महिने", "पैकेट", "packet", "डिब्बे", "बॉक्स",\n    "box", "बोरी", "बैग", "bag", "बजता", "नग", "नग्स", "लोग", "जण", "लोक", "बार", "वेळा", "times", "टन", "ton",\n    "वोल्ट", "volt", "v", "kw", "किलोवाट", "hp", "एचपी", "एमसीबी", "mcb", "सिलेंडर", "kmph", "km", "किमी", "किलोमीटर",\n}\nNUKTA, CANDRABINDU, ANUSVARA = "़", "ँ", "ं"\nTOKEN = re.compile(r"₹|/-|\\d[\\d,]*(?:\\.\\d+)?|[^\\s\\d₹.,!?।|;:\\"\'()\\[\\]{}<>/\\\\-]+")\nDIGITS = re.compile(r"^\\d[\\d,]*(?:\\.\\d+)?$")\nMR_SUFFIXES = ("ाला", "ाना", "ाचा", "ाची", "ाचे", "ाने", "ात", "ला", "ना", "चा", "ची", "चे", "ने", "त", "ं", "ची")\n\n\ndef deva_key(word: str) -> str:\n    """A Devanagari word without nukta, with chandrabindu as anusvara: "पाँच" and "पांच" match, "हज़ार" and "हजार"."""\n    word = unicodedata.normalize("NFC", word)\n    return word.replace(NUKTA, "").replace(CANDRABINDU, ANUSVARA).replace("ऩ", "न").replace("ड़", "ड").replace("ढ़", "ढ")\n\n\ndef latin_key(word: str) -> str:\n    """A Latin word in a loose spelling: "paanch" and "panch", "pachchis" and "pachis", "hazaar" and "hajar" match."""\n    word = word.lower()\n    for a, b in (("aa", "a"), ("ee", "i"), ("oo", "u"), ("w", "v"), ("z", "j"), ("ph", "f"), ("q", "k")):\n        word = word.replace(a, b)\n    word = re.sub(r"(.)\\1+", r"\\1", word)  # no doubled letters\n    word = re.sub(r"(?<=[aeiou])h$", "", word)  # "pandrah" -> "pandra"\n    return word.replace("chh", "ch").replace("th", "t").replace("dh", "d")\n\n\ndef is_latin(word: str) -> bool:\n    return bool(re.match(r"^[A-Za-z]+$", word))\n\n\ndef key(word: str) -> str:\n    return latin_key(word) if is_latin(word) else deva_key(word)\n\n\ndef _table() -> dict[str, tuple[str, float]]:\n    """Every number word (by key) -> (kind, value). kind: "val" (a value), "mul" (×100, ×1000, ...), "frac" (a whole\n    fraction like डेढ़), "pre" (+/- added to the next value, like साढ़े)."""\n    table: dict[str, tuple[str, float]] = {}\n\n    def add(word, kind, value, keep=False):\n        k = key(word)\n        if keep and k in table:\n            return\n        table[k] = (kind, value)\n\n    for n, w in enumerate(HI_ONES):\n        add(w, "val", n)\n    for w, n in HI_MORE.items():\n        add(w, "val", n)\n    for n, w in enumerate(MR_ONES):  # Marathi and Hindi share many words with the same value\n        add(w, "val", n, keep=True)\n        if "ष्ठ" in w:\n            add(w.replace("ष्ठ", "ष्ट"), "val", n, keep=True)\n    for w, n in MR_MORE.items():\n        add(w, "val", n, keep=True)\n    for n, w in enumerate(EN_ONES):\n        add(w, "val", n)\n    for w, n in {**EN_TENS, **EN_DEVANAGARI}.items():\n        add(w, "val", n)\n    for n, words in LATIN_HI.items():\n        for w in words.split():\n            add(w, "val", n, keep=True)\n    for w, m in MULTIPLIERS.items():\n        add(w, "mul", m)\n    for w, v in FRACTIONS.items():\n        add(w, "frac", v)\n    for w, v in PREFIXES.items():\n        add(w, "pre", v)\n    return table\n\n\nTABLE = _table()\nMR_HUNDREDS = {key(w): n for n, w in enumerate(MR_ONES[1:10], 1)}  # दोनशे = 200: the stem before "शे"\nPREFIX_KEYS = sorted((k for k, (kind, _) in TABLE.items() if kind == "pre" and not k.isascii()), key=len, reverse=True)\n\n\n@dataclass\nclass Span:\n    """One number found in a text: its value, its tokens [start, end), the words, and whether it is clearly a number\n    (has digits, a multiplier, a fraction or several words, or is 10 or more) rather than e.g. "दो" in "बता दो"."""\n    value: float\n    start: int\n    end: int\n    text: str\n    strong: bool\n\n\ndef tokens(text: str) -> list[str]:\n    return TOKEN.findall(unicodedata.normalize("NFC", text or ""))\n\n\ndef classify(token: str) -> list[tuple[str, float]]:\n    """A token as number parts: [] (not a number), [("val", 220)], [("val", 2), ("mul", 100)] for "दोनशे", ..."""\n    if DIGITS.match(token):\n        value = float(token.replace(",", ""))\n        return [("digits", int(value) if value.is_integer() else value)]\n    m = re.match(r"^(\\d+(?:\\.\\d+)?)[kK]$", token)  # 2k\n    if m:\n        return [("digits", float(m.group(1))), ("mul", 1000)]\n    k = key(token)\n    if k in TABLE:\n        return [TABLE[k]]\n    if not is_latin(token):\n        found = classify_key(k)\n        if found:\n            return found\n        for pre in PREFIX_KEYS:  # joined: साडेतीनशे = साडे + तीनशे, सव्वादोनशे\n            if k.startswith(pre) and len(k) > len(pre) + 1:\n                rest = classify_key(k[len(pre):])\n                if rest and rest[0][0] == "val":\n                    return [TABLE[pre], *rest]\n        for suffix in MR_SUFFIXES:  # Marathi joins endings on: शंभराला, दोनशेला, हजाराला\n            if k.endswith(suffix) and len(k) > len(suffix) + 1:\n                stem = k[: -len(suffix)]\n                for candidate in (stem, stem + "र" if suffix.startswith("ा") else stem):\n                    found = classify_key(candidate)\n                    if found:\n                        return found\n    return []\n\n\ndef classify_key(k: str) -> list[tuple[str, float]]:\n    if k in TABLE:\n        return [TABLE[k]]\n    if k.endswith("शे"):  # Marathi hundreds in one word: दोनशे 200, एकशे 100, दीडशे 150, अडीचशे 250\n        stem = k[:-2]\n        if stem in MR_HUNDREDS:\n            return [("val", MR_HUNDREDS[stem]), ("mul", 100)]\n        if TABLE.get(stem, ("",))[0] == "frac":\n            return [("frac", TABLE[stem][1]), ("mul", 100)]\n    return []\n\n\ndef whole(value: float) -> float:\n    return int(value) if float(value).is_integer() else round(value, 2)\n\n\ndef find_numbers(text: str, colloquial: bool = False) -> list[Span]:\n    """Every number in the text. colloquial: two values in a row like "दो बीस" or "two thirty" are one price (220,\n    230), as shopkeepers say them; otherwise they are two numbers."""\n    words = tokens(text)\n    spans: list[Span] = []\n    i = 0\n    while i < len(words):\n        parts = classify(words[i])\n        if not parts or (parts == [parts[0]] and parts[0][0] == "pre"\n                         and not (i + 1 < len(words) and classify(words[i + 1]))):\n            i += 1\n            continue\n        start, total, current, raw, prefix, top, count, strong = i, 0.0, None, True, None, None, 0, False\n        while i < len(words):\n            parts = classify(words[i])\n            if (not parts and words[i].lower() == "and" and current is not None and not raw and i + 1 < len(words)\n                    and classify(words[i + 1])):\n                i += 1  # "two hundred and twenty"\n                continue\n            if not parts:\n                break\n            stop = False\n            for kind, v in parts:\n                if kind == "pre":\n                    if current is not None:\n                        stop = True  # a fraction word starts a new number: "दो सौ, साढ़े तीन सौ"\n                        break\n                    prefix, strong = v, True\n                elif kind in ("val", "digits", "frac"):\n                    x = v + (prefix or 0)\n                    if prefix is not None or kind != "val":\n                        strong = True\n                    if current is None:\n                        current, raw = x, True\n                    elif raw and current in EN_TENS.values() and 1 <= x <= 9:\n                        current += x  # "twenty five"\n                    elif raw and colloquial and 1 <= current <= 9 and float(current).is_integer() and 10 <= x <= 99:\n                        current, raw, strong = current * 100 + x, False, True  # "दो बीस" = 220\n                    elif not raw and top is not None and x < top and (current % top == 0 or (\n                            current % 10 == 0 and current % 100 in EN_TENS.values() and 1 <= x <= 9)):\n                        current += x  # "दो सौ बीस": 200 + 20; "two hundred twenty five"\n                    else:\n                        stop = True\n                        break\n                    prefix = None\n                    count += 1\n                elif kind == "mul":\n                    strong = True\n                    base = current if current is not None else 1 + (prefix or 0)\n                    prefix = None\n                    if v == 100:\n                        if not raw and current is not None and current >= 100:\n                            stop = True  # "दो सौ तीन सौ": two numbers\n                            break\n                        current, raw, top = base * 100, False, 100\n                    else:\n                        if top is not None and top >= v and current is None:\n                            stop = True\n                            break\n                        total += base * v\n                        current, raw, top = None, False, v\n            if stop:\n                break\n            i += 1\n        value = total + (current or 0)\n        if i == start:  # nothing taken (e.g. a lone prefix)\n            i += 1\n            continue\n        if total == 0 and current is None:\n            continue\n        strong = strong or count > 1 or value >= 10\n        spans.append(Span(whole(value), start, i, " ".join(words[start:i]), strong))\n    return spans\n\n\ndef _near(words: list[str], start: int, end: int, cues: set[str], before: int, after: int) -> bool:\n    keys = {key(c) for c in cues}\n    window = words[max(0, start - before):start] if before else []\n    window += words[end:end + after] if after else []\n    return any(key(w) in keys or w.lower() in cues for w in window)\n\n\ndef money(text: str, expected: bool = False) -> float | None:\n    """The price said in the text (rupees), or None. Sizes and counts ("16A", "दो पीस", "5 किलो") are skipped. With\n    expected (the price was just asked), a clear number counts alone; without, it needs a sign that it is money next\n    to it ("रुपये", "₹", "दाम", "का", "पड़ेगा")."""\n    words = tokens(text)\n    spans = find_numbers(text, colloquial=True)\n    units = {key(u) for u in UNITS}\n    best, best_score = None, (0, False, 0)\n    for s in spans:\n        nxt = words[s.end] if s.end < len(words) else ""\n        if key(nxt) in units or nxt.lower() in UNITS or not 0 < s.value < 10_000_000:\n            continue\n        cue = _near(words, s.start, s.end, MONEY_AFTER - WEAK_AFTER, 0, 2) or _near(words, s.start, s.end,\n                                                                                   MONEY_BEFORE, 3, 0)\n        score = 3 * cue + _near(words, s.start, s.end, WEAK_AFTER, 0, 1) + s.strong\n        if expected and score == 0 and s.value >= 1 and len(spans) == 1:\n            score = 1\n        rank = (score, s.strong, s.value)\n        if score and (expected or score >= 2) and (best is None or rank > best_score):\n            best, best_score = s.value, rank\n    return best\n\n\ndef annotate(text: str) -> str:\n    """The text with each number said in words followed by its digits, to help a small model read it:\n    "ढाई सौ का है" -> "ढाई सौ (=250) का है". Plain digits and lone weak words ("बता दो") are left as they are."""\n    words = tokens(text)\n    spans = [s for s in find_numbers(text, colloquial=True)\n             if s.strong and not all(DIGITS.match(w) for w in words[s.start:s.end])]\n    out = text\n    for s in reversed(spans):\n        pos = _find_end(out, words, s.end - 1)\n        if pos is not None:\n            out = out[:pos] + f" (={s.value})" + out[pos:]\n    return out\n\n\ndef _find_end(text: str, words: list[str], index: int) -> int | None:\n    """Where token number index ends in text."""\n    pos = 0\n    for n, w in enumerate(words):\n        found = text.find(w, pos)\n        if found < 0:\n            return None\n        pos = found + len(w)\n        if n == index:\n            return pos\n    return None\n\n\n# --- times and days ---------------------------------------------------------------------------------------------\nCLOCK = {"बजे", "बजकर", "बज", "बजा", "बजेंगे", "baje", "bajkar", "वाजता", "वाजून", "वाजेपर्यंत", "वाजे", "vajta",\n         "o\'clock", "oclock", "am", "pm", "a.m", "p.m"}\nMORNING = {"सुबह", "सवेरे", "सुबे", "सकाळी", "सकाळ", "subah", "subha", "savere", "sakali", "morning", "am"}\nNOON = {"दोपहर", "दुपहर", "दुपारी", "दुपार", "dopahar", "dopehar", "dupari", "afternoon", "noon"}\nEVENING = {"शाम", "सायं", "संध्याकाळी", "संध्याकाळ", "सायंकाळी", "shaam", "sham", "sandhyakali", "evening", "pm"}\nNIGHT = {"रात", "रात्रि", "रात्री", "रात्र", "raat", "rat", "ratri", "night"}\nDAYS = {\n    "today": ["आज", "आजच", "aaj", "aj", "today"],\n    "tomorrow": ["कल", "उद्या", "उद्याला", "kal", "kaal", "udya", "tomorrow", "tmrw"],\n    "day_after": ["परसों", "परसो", "परवा", "परवाला", "parson", "parso", "parva", "parwa"],\n    "monday": ["सोमवार", "सोमवारी", "somvar", "somwar", "monday", "मंडे"],\n    "tuesday": ["मंगलवार", "मंगळवार", "मंगळवारी", "mangalvar", "mangalwar", "tuesday", "ट्यूज़डे", "ट्यूसडे"],\n    "wednesday": ["बुधवार", "बुधवारी", "budhvar", "budhwar", "wednesday", "वेडनेसडे", "वेन्सडे"],\n    "thursday": ["गुरुवार", "गुरूवार", "बृहस्पतिवार", "वीरवार", "गुरुवारी", "guruvar", "guruwar", "thursday", "थर्सडे"],\n    "friday": ["शुक्रवार", "शुक्रवारी", "shukravar", "shukrawar", "friday", "फ्राइडे"],\n    "saturday": ["शनिवार", "शनिवारी", "shanivar", "shaniwar", "saturday", "सैटरडे"],\n    "sunday": ["रविवार", "इतवार", "रविवारी", "ravivar", "raviwar", "itvar", "sunday", "संडे"],\n}\nDAY_OF = {key(w): d for d, words in DAYS.items() for w in words}\nDAY_AFTER_PHRASES = ["day after tomorrow", "day after"]\n\n\ndef time_of_day(text: str) -> str | None:\n    """The clock time said in the text as "HH:MM" (24 h), or None: "शाम साढ़े चार बजे" -> "16:30", "10:30" -> "10:30",\n    "सकाळी दहा वाजता" -> "10:00", "चार बजे" -> "16:00" (1 to 7 with no part of the day said: afternoon)."""\n    t = unicodedata.normalize("NFC", text or "")\n    words = tokens(t)\n    lowered = [w.lower() for w in words]\n    period = None\n    for w in lowered:\n        k = key(w)\n        for name, group in (("morning", MORNING), ("noon", NOON), ("evening", EVENING), ("night", NIGHT)):\n            if w in group or k in {key(g) for g in group}:\n                period = name\n    hour = minute = None\n    clock = re.search(r"\\b(\\d{1,2})[:.](\\d{2})\\b", t)\n    if clock and int(clock.group(1)) < 24 and int(clock.group(2)) < 60:\n        hour, minute = int(clock.group(1)), int(clock.group(2))\n    else:\n        clock_keys = {key(c) for c in CLOCK}\n        cue = next((i for i, w in enumerate(lowered) if w in CLOCK or key(w) in clock_keys), None)\n        if cue is None:  # "पाँच बदे": a misheard बजे right after a number\n            numbered = {s.end for s in find_numbers(t)}\n            cue = next((i for i, w in enumerate(lowered) if i in numbered and not is_latin(w) and len(key(w)) == 3\n                        and any(distance(key(w), c) <= 1 for c in ("बजे", "बजक"))), None)\n        if cue is not None:\n            spans = [s for s in find_numbers(t) if s.end <= cue]\n            if spans and spans[-1].end >= cue - 1:\n                value = float(spans[-1].value)\n                hour, minute = int(value), round((value - int(value)) * 60)\n                after = [s for s in find_numbers(t) if s.start > cue]  # "चार बजकर बीस मिनट"\n                if after and minute == 0 and after[0].start <= cue + 2 and 0 < after[0].value < 60:\n                    minute = int(after[0].value)\n        elif period:  # "शाम पाँच" with no बजे: the number right after the part of the day\n            for i, w in enumerate(lowered):\n                if any(w in g or key(w) in {key(x) for x in g} for g in (MORNING, NOON, EVENING, NIGHT)):\n                    nums = [s for s in find_numbers(t) if s.start == i + 1 or (s.start == i + 2)]\n                    if nums and 0 < nums[0].value <= 12:\n                        value = float(nums[0].value)\n                        hour, minute = int(value), round((value - int(value)) * 60)\n    if hour is None or not (0 <= hour <= 24) or not (0 <= (minute or 0) < 60):\n        return None\n    minute = minute or 0\n    if hour <= 12:\n        if (period in ("evening", "night") and hour < 12) or (period == "noon" and 1 <= hour <= 6):\n            hour += 12\n        elif period == "night" and hour == 12:\n            hour = 0\n        elif period is None and 1 <= hour <= 7:\n            hour += 12\n    hour %= 24\n    return f"{hour:02d}:{minute:02d}"\n\n\ndef distance(a: str, b: str) -> int:\n    """Edit distance (letters added, dropped or changed)."""\n    row = list(range(len(b) + 1))\n    for i, x in enumerate(a, 1):\n        prev, row[0] = row[0], i\n        for j, y in enumerate(b, 1):\n            prev, row[j] = row[j], min(row[j] + 1, row[j - 1] + 1, prev + (x != y))\n    return row[-1]\n\n\ndef near(word: str, choices: dict[str, str]) -> str | None:\n    """The value of the choice a misheard word is closest to ("संवार" -> सोमवार, "जविवार" -> रविवार): at most one\n    letter off for 4 letters, two for 5 or more, and only when one value is closest; else None."""\n    k = key(word)\n    if len(k) < 4:\n        return None\n    limit = 1 if len(k) < 5 else 2\n    scored = [(distance(k, c), v) for c, v in choices.items() if len(c) >= 4 and abs(len(c) - len(k)) <= limit]\n    scored = [(d, v) for d, v in scored if d <= limit]\n    if not scored:\n        return None\n    best = min(d for d, _ in scored)\n    found = {v for d, v in scored if d == best}\n    return found.pop() if len(found) == 1 else None\n\n\ndef day(text: str, fuzzy: bool = False) -> str | None:\n    """The day said in the text: "today", "tomorrow", "day_after" or a weekday ("monday" ...), or None. "कल" is\n    taken as tomorrow (a booking talks about the future). fuzzy (the day was just asked): a weekday misheard by\n    speech to text by a letter or two counts too."""\n    low = (text or "").lower()\n    if any(p in low for p in DAY_AFTER_PHRASES):\n        return "day_after"\n    found = None\n    weekdays = {k: d for k, d in DAY_OF.items() if d not in ("today", "tomorrow", "day_after")}\n    for w in tokens(text):\n        d = DAY_OF.get(key(w)) or DAY_OF.get(w.lower()) or (near(w, weekdays) if fuzzy and not is_latin(w) else None)\n        if d:\n            found = d if found in (None, "today") or d not in ("today",) else found\n    return found\n\n\n# --- phone numbers ----------------------------------------------------------------------------------------------\nREPEAT = {"डबल": 2, "double": 2, "दुबल": 2, "ट्रिपल": 3, "triple": 3, "ट्रिप्पल": 3}\nDIGIT_WORDS = {"oh": 0, "o": 0, "जीरो": 0, "झिरो": 0, "शुन्य": 0}\n# How speech to text writes digits it half-heard, in a phone number (eval 4): "साथ" for सात, "नो" for नौ, ...\nSOUNDS_LIKE = {"साथ": 7, "सात्": 7, "नो": 9, "नव": 9, "शून्ये": 0, "शुन्ये": 0, "शूनिये": 0, "शूने": 0, "शुन्या": 0,\n               "छे": 6, "छः": 6, "तिन": 3, "चर": 4, "पांच": 5, "पाच": 5, "एक्": 1, "दोन": 2, "आट": 8, "आठ्": 8}\n\n\nSOUND_KEYS = {key(w): n for w, n in SOUNDS_LIKE.items()}\n\n\ndef phone(text: str) -> str | None:\n    """A 10-digit Indian mobile number said in the text (digits, digit words, or pairs like "अट्ठानवे"), or None.\n    "+91" and a leading 0 are dropped; "डबल सात" is 77."""\n    digits = ""\n    repeat = 1\n    for w in tokens(text):\n        lw = w.lower()\n        if key(lw) in {key(r) for r in REPEAT} or lw in REPEAT:\n            repeat = REPEAT.get(lw) or next(v for r, v in REPEAT.items() if key(r) == key(lw))\n            continue\n        piece = None\n        if DIGITS.match(w) and "." not in w:\n            piece = w.replace(",", "")\n        elif lw in DIGIT_WORDS or key(lw) in {key(d) for d in DIGIT_WORDS}:\n            piece = "0"\n        elif key(w) in SOUND_KEYS:\n            piece = str(SOUND_KEYS[key(w)])\n        else:\n            parts = classify(w)\n            if len(parts) == 1 and parts[0][0] == "val" and float(parts[0][1]).is_integer() and parts[0][1] < 100:\n                piece = str(int(parts[0][1]))\n        if piece is None:\n            repeat = 1\n            continue\n        digits += piece * repeat if len(piece) == 1 else piece\n        repeat = 1\n    if digits.startswith("91") and len(digits) == 12:\n        digits = digits[2:]\n    if digits.startswith("0") and len(digits) == 11:\n        digits = digits[1:]\n    return digits if re.fullmatch(r"[6-9]\\d{9}", digits) else None\n\n\n# --- values back to words, for the voice -----------------------------------------------------------------------\nMR_HUNDRED_WORDS = {1: "एकशे", 2: "दोनशे", 3: "तीनशे", 4: "चारशे", 5: "पाचशे", 6: "सहाशे", 7: "सातशे", 8: "आठशे",\n                    9: "नऊशे"}\n\ndef _en_words(n: int) -> str:\n    if n < 20:\n        return EN_ONES[n]\n    if n < 100:\n        tens = next(w for w, v in EN_TENS.items() if v == n // 10 * 10 and w != "fourty")\n        return tens + (f" {EN_ONES[n % 10]}" if n % 10 else "")\n    for size, name in ((10_000_000, "crore"), (100_000, "lakh"), (1000, "thousand"), (100, "hundred")):\n        if n >= size:\n            rest = n % size\n            return f"{_en_words(n // size)} {name}" + (f" {_en_words(rest)}" if rest else "")\n    return str(n)\n\n\ndef to_words(n, lang: str = "hi") -> str:\n    """A whole number in words: to_words(220, "hi") -> "दो सौ बीस", (220, "mr") -> "दोनशे वीस", (220, "en") ->\n    "two hundred twenty". Halves and quarters of small numbers use साढ़े/सवा/पौने. Digits from 100 crore up."""\n    value = float(n)\n    if value < 0 or value >= 1_000_000_000:\n        return str(n)\n    if not value.is_integer():\n        base, frac = int(value), round(value - int(value), 2)\n        words = {0.5: ("साढ़े", "साडे", "and a half"), 0.25: ("सवा", "सव्वा", "and a quarter")}\n        if lang in ("hi", "mr") and frac in words and base >= 3:\n            return f"{words[frac][0 if lang == \'hi\' else 1]} {to_words(base, lang)}"\n        if lang in ("hi", "mr") and value in (1.5, 2.5):\n            return {(1.5, "hi"): "डेढ़", (2.5, "hi"): "ढाई", (1.5, "mr"): "दीड", (2.5, "mr"): "अडीच"}[(value, lang)]\n        return str(value)\n    n = int(value)\n    if lang == "en":\n        return _en_words(n)\n    ones = MR_ONES if lang == "mr" else HI_ONES\n    if n < 100:\n        return ones[n]\n    words = []\n    names = (("कोटी", "हजार", "लाख") if lang == "mr" else ("करोड़", "हज़ार", "लाख"))\n    for size, name in ((10_000_000, names[0]), (100_000, names[2]), (1000, names[1])):\n        if n >= size:\n            words.append(f"{to_words(n // size, lang)} {name}")\n            n %= size\n    if n >= 100:\n        words.append(MR_HUNDRED_WORDS[n // 100] if lang == "mr" else f"{ones[n // 100]} सौ")\n        n %= 100\n    if n:\n        words.append(ones[n])\n    return " ".join(words)\n\n\ndef money_words(price, lang: str = "hi") -> str:\n    """A price as the voice says it: 220 -> "दो सौ बीस रुपये" (mr "दोनशे वीस रुपये", en "two hundred twenty rupees");\n    paise too: 12.5 -> "बारह रुपये पचास पैसे"."""\n    value = float(price)\n    rupees, paise = int(value), round((value - int(value)) * 100)\n    unit = {"hi": ("रुपये", "पैसे"), "mr": ("रुपये", "पैसे"), "en": ("rupees", "paise")}.get(lang, ("रुपये", "पैसे"))\n    said = f"{to_words(rupees, lang)} {unit[0]}"\n    return said + (f" {to_words(paise, lang)} {unit[1]}" if paise else "")\n\n\ndef time_words(hhmm: str, lang: str = "hi") -> str:\n    """"16:30" as the voice says it: hi "शाम साढ़े चार बजे", mr "संध्याकाळी साडेचार वाजता", en "4:30 in the evening"."""\n    hour, minute = (int(x) for x in hhmm.split(":"))\n    h12 = hour % 12 or 12\n    part = ("morning" if 4 <= hour < 12 else "noon" if 12 <= hour < 16 else "evening" if 16 <= hour < 20 else "night")\n    if lang == "en":\n        when = {"morning": "in the morning", "noon": "in the afternoon", "evening": "in the evening",\n                "night": "at night"}[part]\n        said = (f"{_en_words(h12)} o\'clock" if not minute else\n                f"{_en_words(h12)} {\'oh \' if minute < 10 else \'\'}{_en_words(minute)}")\n        return f"{said} {when}"\n    mr = lang == "mr"\n    period = {"morning": ("सुबह", "सकाळी"), "noon": ("दोपहर", "दुपारी"), "evening": ("शाम", "संध्याकाळी"),\n              "night": ("रात", "रात्री")}[part][mr]\n    nxt = h12 % 12 + 1\n    o_clock = "वाजता" if mr else "बजे"\n    if minute == 0:\n        said = f"{to_words(h12, lang)} {o_clock}"\n    elif minute == 30:\n        said = (("दीड" if mr else "डेढ़") if h12 == 1 else ("अडीच" if mr else "ढाई") if h12 == 2\n                else f"{\'साडे\' if mr else \'साढ़े\'} {to_words(h12, lang)}") + f" {o_clock}"\n    elif minute == 15:\n        said = f"{\'सव्वा\' if mr else \'सवा\'} {to_words(h12, lang)} {o_clock}"\n    elif minute == 45:\n        said = f"{\'पावणे\' if mr else \'पौने\'} {to_words(nxt, lang)} {o_clock}"\n    else:\n        said = (f"{to_words(h12, lang)} वाजून {to_words(minute, lang)} मिनिटांनी" if mr\n                else f"{to_words(h12, lang)} बजकर {to_words(minute, lang)} मिनट")\n    return f"{period} {said}"\n\n\nDAY_WORDS = {\n    "today": ("आज", "आज", "today"), "tomorrow": ("कल", "उद्या", "tomorrow"),\n    "day_after": ("परसों", "परवा", "the day after tomorrow"), "monday": ("सोमवार", "सोमवारी", "Monday"),\n    "tuesday": ("मंगलवार", "मंगळवारी", "Tuesday"), "wednesday": ("बुधवार", "बुधवारी", "Wednesday"),\n    "thursday": ("गुरुवार", "गुरुवारी", "Thursday"), "friday": ("शुक्रवार", "शुक्रवारी", "Friday"),\n    "saturday": ("शनिवार", "शनिवारी", "Saturday"), "sunday": ("रविवार", "रविवारी", "Sunday"),\n}\n\n\ndef day_words(d: str, lang: str = "hi") -> str:\n    return DAY_WORDS.get(d, (d, d, d))[{"hi": 0, "mr": 1, "en": 2}.get(lang, 0)]\n\n\ndef digit_words(digits: str, lang: str = "hi") -> str:\n    """A phone number read digit by digit, in two groups: "9876543210" -> "नौ आठ सात छह पाँच, चार तीन दो एक शून्य"."""\n    ones = MR_ONES if lang == "mr" else EN_ONES if lang == "en" else HI_ONES\n    said = [ones[int(c)] for c in digits if c.isdigit()]\n    half = (len(said) + 1) // 2\n    return " ".join(said[:half]) + (", " + " ".join(said[half:]) if said[half:] else "")\n', 'lang.py': '"""Which language a line is in: Hindi, Marathi or English (Devanagari or Latin letters). Hinglish counts as Hindi.\n\n    detect("हो, आहे ना, दोनशे रुपये") -> ("mr", 3)        the language and how many marker words were found\n    detect("haan bhai hai, do sau ka") -> ("hi", 3)\n    script("नमस्ते") -> "deva"\n\nSpeech to text writes Marathi and Hindi in the same letters, so the words decide: आहे, नाही, होय, मी are Marathi;\nहै, नहीं, हाँ, मैं are Hindi. The letter ळ is almost only Marathi.\n"""\n\nimport re\nimport unicodedata\n\nMARKERS = {\n    "mr": """आहे आहेत आहोत आहात नाही नाहीये नाहीत होय काय मी तुम्ही आम्ही आपण पाहिजे नको कधी कुठे किती देतो देते देऊ\n             करतो करते मिळेल मिळणार मिळतील होते होता होती उद्या परवा सकाळी संध्याकाळी दुपारी वाजता बरं बरोबर झालं\n             संपलं संपला सांगा बोला येईल जाईल लागेल पडेल माझं माझा माझी तुमचं तुमचा तुमची आमचं आमचा आमची हवं हवा\n             हवी नाव फोन करा द्या घ्या ठेवतो ठेवा आता इथे तिथे म्हणजे म्हणून आणि पण दोन पाच सहा नऊ दहा शंभर दोनशे\n             पाचशे रुपयांना करू नंतर थांबा कोण बोलतंय बोलताय चुकीचा ऐकू येतंय कळलं समजलं""",\n    "hi": """है हैं हूँ हूं नहीं नही हाँ हां क्या कितना कितने कितनी आप मैं हम था थी थे होगा होगी जाएगा मिलेगा\n             मिलेंगे चाहिए भाई भैया की के में से को पर वाला वाली रहा रही रहे देंगे करेंगे कर देंगे परसों बजे\n             ठीक लेकिन और अभी यहाँ वहाँ कौन बोल रहे सुनिए बताइए बताओ दीजिए कीजिए हमारे आपका आपकी मेरा मेरी नाम\n             दो पाँच पांच छह नौ दस सौ दुकान बाद मत गलत समझ""",\n    "en": """is are the yes no we have it available price deliver delivery what how much not sorry okay ok please\n             thank thanks you your our this that there here sir madam stock rupees tomorrow today call later number\n             wrong who speaking hello hi send home yeah nope don\'t""",\n}\nLATIN_MARKERS = {\n    "mr": "aahe ahe ahet nahiye hoy kay mi tumhi amhi apan pahije nako kadhi kuthe kiti deto dete karto karte milel "\n          "milnar hota hoti udya parva sakali sandhyakali vajta bara barobar zala sampla sanga bola yeil lagel padel "\n          "majha maza tumcha amcha hava nantar thamba kon boltay",\n    "hi": "hai hain hoon nahi nahin haan han kya kitna kitne aap main hum tha thi hoga milega chahiye bhai bhaiya ka "\n          "ki ke mein se ko par wala wali raha rahi denge karenge kal parson baje theek thik lekin aur abhi kaun "\n          "boliye bataiye bata dijiye hamare aapka mera naam sau rupaye rupay dukaan baad mat galat samajh",\n}\nDEVANAGARI = re.compile(r"[ऀ-ॿ]")\nLATIN = re.compile(r"[A-Za-z]")\nWORD = re.compile(r"[ऀ-ॿ]+|[A-Za-z\']+")\n\n\ndef _keys(text: str) -> set[str]:\n    return set(text.split())\n\n\nDEVA_SETS = {lang: _keys(unicodedata.normalize("NFC", words)) for lang, words in MARKERS.items() if lang != "en"}\nEN_SET = _keys(MARKERS["en"])\nLATIN_SETS = {lang: _keys(words) for lang, words in LATIN_MARKERS.items()}\n\n\ndef script(text: str) -> str:\n    """"deva", "latin", "mixed" or "none" (no letters)."""\n    deva, latin = len(DEVANAGARI.findall(text or "")), len(LATIN.findall(text or ""))\n    if not deva and not latin:\n        return "none"\n    if deva and latin:\n        return "deva" if deva >= 3 * latin else "latin" if latin >= 3 * deva else "mixed"\n    return "deva" if deva else "latin"\n\n\ndef detect(text: str, default: str = "hi") -> tuple[str, int]:\n    """The language of the line and how many marker words decided it (0: none found, the default is returned).\n    Hinglish (Hindi words in Latin letters, or mixed with English) is "hi"."""\n    words = [w.lower() for w in WORD.findall(unicodedata.normalize("NFC", text or ""))]\n    if not words:\n        return default, 0\n    score = {"hi": 0, "mr": 0, "en": 0}\n    for w in words:\n        if DEVANAGARI.match(w):\n            for lang, markers in DEVA_SETS.items():\n                score[lang] += w in markers\n            if "ळ" in w:\n                score["mr"] += 1\n        else:\n            for lang, markers in LATIN_SETS.items():\n                score[lang] += w in markers\n            score["en"] += w in EN_SET\n    # Hindi in Latin letters often mixes in English words: it is Hinglish when any Hindi marker is there\n    if score["en"] and score["hi"] and score["hi"] >= score["mr"]:\n        score["hi"] += score["en"]\n        score["en"] = 0\n    best = max(score, key=lambda k: (score[k], k == default))\n    if score[best] == 0:\n        return default, 0\n    if score["hi"] == score["mr"] and best in ("hi", "mr"):\n        return (default if default in ("hi", "mr") else "hi"), score[best]\n    return best, score[best]\n', 'speech_text.py': '"""Text for the voice, and clean-up of what speech to text heard.\n\n    for_voice("16A MCB ₹220 में है", "hi") -> "सोलह ए एम सी बी दो सौ बीस रुपये में है"\n    clean_heard("Thank you for watching!") -> ""          a known speech-to-text hallucination on silence\n\nVoices read words, not digits or capital-letter short forms, and some read only one script; so digits, ₹, times\nand short forms like MCB are spelled out in the reply\'s language first.\n"""\n\nimport re\nimport unicodedata\n\nfrom voicebot.numerals import digit_words, money_words, time_words, to_words\n\nLETTERS_HI = {\n    "A": "ए", "B": "बी", "C": "सी", "D": "डी", "E": "ई", "F": "एफ", "G": "जी", "H": "एच", "I": "आई", "J": "जे",\n    "K": "के", "L": "एल", "M": "एम", "N": "एन", "O": "ओ", "P": "पी", "Q": "क्यू", "R": "आर", "S": "एस", "T": "टी",\n    "U": "यू", "V": "वी", "W": "डब्ल्यू", "X": "एक्स", "Y": "वाई", "Z": "ज़ेड",\n}\nPHONE_LIKE = re.compile(r"(?<!\\d)(?:\\+91[\\s-]?)?(\\d{5})[\\s-]?(\\d{5})(?!\\d)")\nCLOCK = re.compile(r"(?<!\\d)([01]?\\d|2[0-3]):([0-5]\\d)(?!\\d)")\nRUPEES = re.compile(r"₹\\s?(\\d[\\d,]*(?:\\.\\d{1,2})?)(?:/-)?|(?<![\\w.])(?:Rs\\.?|INR)\\s?(\\d[\\d,]*(?:\\.\\d{1,2})?)")\nCODE = re.compile(r"(?<![\\w])(\\d+)([A-Za-z]{1,3})(?![\\w])")  # 16A, 5kg, 32GB\nACRONYM = re.compile(r"(?<![A-Za-z])([A-Z]{2,6})(?![A-Za-z])")\nNUMBER = re.compile(r"(?<![\\w.])(\\d[\\d,]*(?:\\.\\d+)?)(?![\\w])")\nUNIT_WORDS = {"kg": ("किलो", "किलो", "kilo"), "g": ("ग्राम", "ग्राम", "gram"), "gb": ("जीबी", "जीबी", "GB"),\n              "ml": ("एमएल", "एमएल", "ml"), "mm": ("एमएम", "एमएम", "mm"), "w": ("वाट", "वॅट", "watt"),\n              "l": ("लीटर", "लिटर", "litre"), "m": ("मीटर", "मीटर", "metre")}\n\n\ndef spell(letters: str, lang: str) -> str:\n    """Capital letters one by one: "MCB" -> "एम सी बी" (English: "M C B")."""\n    if lang == "en":\n        return " ".join(letters.upper())\n    return " ".join(LETTERS_HI.get(c, c) for c in letters.upper())\n\n\ndef _number(text: str) -> float:\n    value = float(text.replace(",", ""))\n    return int(value) if value.is_integer() else value\n\n\ndef for_voice(text: str, lang: str = "hi") -> str:\n    """The text as the voice should say it, in lang ("hi", "mr" or "en"): phone numbers digit by digit, times, ₹\n    prices, sizes like 16A, short forms like MCB and every other number in words. Symbols the voice can\'t say go."""\n    if not text:\n        return ""\n    t = unicodedata.normalize("NFC", text)\n    li = {"hi": 0, "mr": 1, "en": 2}.get(lang, 0)\n    t = PHONE_LIKE.sub(lambda m: digit_words(m.group(1) + m.group(2), lang), t)\n    t = CLOCK.sub(lambda m: time_words(f"{int(m.group(1)):02d}:{m.group(2)}", lang), t)\n    t = RUPEES.sub(lambda m: money_words(_number(m.group(1) or m.group(2)), lang), t)\n\n    def code(m):\n        unit = UNIT_WORDS.get(m.group(2).lower())\n        tail = unit[li] if unit and m.group(2).islower() else spell(m.group(2), lang)\n        return f"{to_words(_number(m.group(1)), lang)} {tail}"\n\n    t = CODE.sub(code, t)\n    t = ACRONYM.sub(lambda m: spell(m.group(1), lang), t)\n    t = NUMBER.sub(lambda m: to_words(_number(m.group(1)), lang), t)\n    t = t.replace("%", " प्रतिशत" if lang != "en" else " percent").replace("&", " और " if lang == "hi" else " ")\n    t = re.sub(r"[*_#~`|<>\\[\\]{}^=]+", " ", t)\n    return re.sub(r"\\s+", " ", t).strip()\n\n\n# What Whisper writes on silence or noise: never something a person said on the call.\nHALLUCINATIONS = [\n    "thank you for watching", "thanks for watching", "please subscribe", "subscribe to", "like and subscribe",\n    "subtitles by", "amara.org", "transcribed by", "सब्सक्राइब", "उपशीर्षक", "चैनल को", "वीडियो देखने",\n    "ご視聴", "字幕", "www.", ".com", "♪",\n]\nOTHER_SCRIPTS = re.compile(r"[\u0600-ۿ\u0e00-\u0e7f\u3040-ヿ一-鿿가-\ud7afЀ-ӿ]")\n\n\ndef clean_heard(text: str) -> str:\n    """What speech to text heard, cleaned: "" when it is a known hallucination or mostly in another script (Arabic,\n    Chinese, ... from noise); a phrase repeated over and over (another Whisper failure) is kept once."""\n    t = " ".join((text or "").split())\n    if not t:\n        return ""\n    low = t.lower()\n    if any(h in low for h in HALLUCINATIONS):\n        return ""\n    letters = [c for c in t if c.isalpha()]\n    if letters and len(OTHER_SCRIPTS.findall(t)) > len(letters) / 3:\n        return ""\n    words = t.split()\n    for size in range(1, 6):  # "हाँ हाँ हाँ हाँ हाँ हाँ" -> "हाँ हाँ"; long loops of a phrase -> the phrase once\n        out, i = [], 0\n        while i < len(words):\n            chunk = words[i:i + size]\n            repeats = 1\n            while words[i + repeats * size:i + (repeats + 1) * size] == chunk:\n                repeats += 1\n            if repeats >= 3:\n                out += chunk * (2 if size == 1 else 1)\n                i += repeats * size\n            else:\n                out.append(words[i])\n                i += 1\n        words = out\n    return " ".join(words)\n', 'tasks.py': '"""Tasks: what the voice bot is for on one call, written as a small JSON file (voicebot/tasks/*.json), so the same bot\ncan ask a shop about stock, book a clinic visit or confirm an order, with no code change.\n\nA task says who the bot talks to and why, the greeting, the facts it may tell, and the "slots": the things it has to\nfind out, each with a type, the question that asks for it (in Hindi, Marathi and English) and when to ask it.\n\n    task = load("shop_stock")                     voicebot/tasks/shop_stock.json\n    task = from_json(text)                         a task someone wrote (checked the same way)\n    problems(spec) -> ["slot \'price\': ..."]        what is wrong with a task ([] when it is fine)\n\nSlot types: bool (yes/no), money (rupees), number, text, name, phone (10-digit mobile), day (today, tomorrow,\nday_after or a weekday), time ("HH:MM"), enum (one of "options").\n"""\n\nimport json\nimport re\nfrom pathlib import Path\n\nTASKS_DIR = Path(__file__).parent / "tasks"\nLANGS = ("hi", "mr", "en")\nTYPES = ("bool", "money", "number", "text", "name", "phone", "day", "time", "enum")\nDAY_VALUES = ("today", "tomorrow", "day_after", "monday", "tuesday", "wednesday", "thursday", "friday", "saturday",\n              "sunday")\nRESERVED = {"intent", "faq", "language", "slots"}  # field names the understanding step uses itself\nNAME = re.compile(r"^[a-z][a-z0-9_]{0,39}$")\n\n\nclass TaskError(ValueError):\n    pass\n\n\ndef text_of(value, lang: str, fallback: str = "hi") -> str:\n    """A text in lang from {"hi": ..., "mr": ..., "en": ...} (or a plain string): lang, else fallback, else any."""\n    if value is None:\n        return ""\n    if isinstance(value, str):\n        return value\n    return value.get(lang) or value.get(fallback) or next((v for v in value.values() if v), "")\n\n\ndef _texts_ok(value, where: str, langs, required: bool = True) -> list[str]:\n    if value is None:\n        return [f"{where}: missing"] if required else []\n    if isinstance(value, str):\n        return [] if value.strip() else [f"{where}: empty"]\n    if not isinstance(value, dict):\n        return [f"{where}: must be a text or {{\\"hi\\": ..., \\"mr\\": ..., \\"en\\": ...}}"]\n    missing = [lang for lang in langs if not str(value.get(lang) or "").strip()]\n    return [f"{where}: no {\', \'.join(missing)} text"] if missing else []\n\n\ndef problems(spec) -> list[str]:\n    """Everything wrong with a task spec, in plain words ([] when it can be used)."""\n    if not isinstance(spec, dict):\n        return ["the task must be a JSON object"]\n    out = []\n    if not NAME.match(str(spec.get("id") or "")):\n        out.append("id: lowercase letters, digits and _ (like shop_stock)")\n    langs = spec.get("languages") or ["hi"]\n    if not isinstance(langs, list) or not langs or any(lang not in LANGS for lang in langs):\n        out.append(f"languages: a list from {\', \'.join(LANGS)}")\n        langs = ["hi"]\n    if spec.get("default_language", langs[0]) not in langs:\n        out.append("default_language: must be one of languages")\n    if spec.get("bot_role", "caller") not in ("caller", "receiver"):\n        out.append(\'bot_role: "caller" (the bot calls) or "receiver" (the bot answers)\')\n    for field in ("purpose", "who"):\n        if not str(text_of(spec.get(field), "en")).strip():\n            out.append(f"{field}: missing")\n    out += _texts_ok(spec.get("greeting"), "greeting", langs)\n    closing = spec.get("closing")\n    if isinstance(closing, list):\n        for i, c in enumerate(closing):\n            out += _texts_ok(c.get("text") if isinstance(c, dict) else None, f"closing[{i}]", langs)\n        if not any(isinstance(c, dict) and not c.get("when") for c in closing):\n            out.append("closing: one entry needs no \\"when\\" (the default)")\n    else:\n        out += _texts_ok(closing, "closing", langs)\n    context = spec.get("context") or {}\n    if not isinstance(context, dict):\n        out.append("context: must be an object of names and values")\n    slots = spec.get("slots")\n    if not isinstance(slots, list) or not slots:\n        return out + ["slots: at least one"]\n    names = []\n    for i, slot in enumerate(slots):\n        where = f"slot {i + 1}"\n        if not isinstance(slot, dict):\n            out.append(f"{where}: must be an object")\n            continue\n        name = slot.get("name")\n        where = f"slot \'{name}\'"\n        if not NAME.match(str(name or "")) or name in RESERVED:\n            out.append(f"{where}: name must be lowercase letters, digits and _, and not {sorted(RESERVED)}")\n        if name in names:\n            out.append(f"{where}: used twice")\n        names.append(name)\n        if slot.get("type") not in TYPES:\n            out.append(f"{where}: type must be one of {\', \'.join(TYPES)}")\n        if not str(slot.get("describe") or "").strip():\n            out.append(f"{where}: describe: missing (what it means, in English, for the model)")\n        out += _texts_ok(slot.get("ask"), f"{where} ask", langs)\n        if slot.get("type") == "enum":\n            options = slot.get("options")\n            if not isinstance(options, dict) or len(options) < 2:\n                out.append(f"{where}: options: at least two")\n            else:\n                for value, o in options.items():\n                    out += _texts_ok(o if isinstance(o, dict) else None, f"{where} option \'{value}\'", langs)\n        for cond in (slot.get("when") or {}):\n            if cond not in names[:-1]:\n                out.append(f"{where}: when: \'{cond}\' must be a slot listed before it")\n    for key, fact in (spec.get("facts") or {}).items():\n        if not isinstance(fact, dict) or not str(fact.get("about") or "").strip():\n            out.append(f"fact \'{key}\': about: missing")\n        else:\n            out += _texts_ok(fact.get("answer"), f"fact \'{key}\' answer", langs)\n    try:\n        render_all(spec)\n    except KeyError as e:\n        out.append(f"a text uses {{{e.args[0]}}}, which is not in context or slots")\n    return out\n\n\ndef render(template: str, values: dict) -> str:\n    """The template with {name} filled from values; an unknown name raises KeyError."""\n    return re.sub(r"\\{([a-z_][a-z0-9_]*)\\}", lambda m: str(values[m.group(1)]), template)\n\n\ndef render_all(spec: dict) -> None:\n    """Fill every text once with the context and slot names, to find unknown {names}."""\n    values = {**(spec.get("context") or {}), **{s.get("name"): "x" for s in spec.get("slots") or []\n                                                 if isinstance(s, dict)}, "value": "x"}\n\n    def walk(v):\n        if isinstance(v, str):\n            render(v, values)\n        elif isinstance(v, dict):\n            for k, x in v.items():\n                if k not in ("describe", "about", "purpose", "who", "words"):\n                    walk(x)\n        elif isinstance(v, list):\n            for x in v:\n                walk(x)\n\n    walk({k: v for k, v in spec.items() if k not in ("id", "context", "languages")})\n\n\nclass Task:\n    """A checked task spec, with easy access to its parts."""\n\n    def __init__(self, spec: dict, context: dict | None = None):\n        spec = json.loads(json.dumps(spec))  # a copy\n        if context:\n            spec["context"] = {**(spec.get("context") or {}), **context}\n        found = problems(spec)\n        if found:\n            raise TaskError("; ".join(found))\n        self.spec = spec\n        self.id = spec["id"]\n        self.languages = list(spec.get("languages") or ["hi"])\n        self.default_language = spec.get("default_language") or self.languages[0]\n        self.bot_role = spec.get("bot_role", "caller")\n        self.context = dict(spec.get("context") or {})\n        self.slots = spec["slots"]\n        self.slot = {s["name"]: s for s in self.slots}\n        self.facts = spec.get("facts") or {}\n        self.confirm = bool(spec.get("confirm", False))\n\n    def name(self, lang: str = "en") -> str:\n        return text_of(self.spec.get("name"), lang, "en") or self.id\n\n    def purpose(self) -> str:\n        return render(text_of(self.spec["purpose"], "en"), self.context)\n\n    def who(self) -> str:\n        return render(text_of(self.spec["who"], "en"), self.context)\n\n    def say(self, value, lang: str, extra: dict | None = None) -> str:\n        """A text of the task in lang, filled with the context (and extra values)."""\n        return render(text_of(value, lang, self.default_language), {**self.context, **(extra or {})})\n\n    def applies(self, slot: dict, values: dict) -> bool:\n        """Whether a slot is asked: every "when" condition holds for the values found so far."""\n        return all(values.get(k) == v for k, v in (slot.get("when") or {}).items())\n\n    def closing(self, values: dict, lang: str) -> str:\n        closing = self.spec["closing"]\n        if isinstance(closing, list):\n            chosen = next((c for c in closing if c.get("when") and all(values.get(k) == v for k, v in\n                                                                         c["when"].items())), None)\n            chosen = chosen or next(c for c in closing if not c.get("when"))\n            return self.say(chosen["text"], lang)\n        return self.say(closing, lang)\n\n\ndef from_json(text: str, context: dict | None = None) -> Task:\n    try:\n        spec = json.loads(text)\n    except json.JSONDecodeError as e:\n        raise TaskError(f"not valid JSON: {e}") from e\n    return Task(spec, context)\n\n\ndef load(task_id: str, context: dict | None = None) -> Task:\n    path = TASKS_DIR / f"{task_id}.json"\n    if not NAME.match(task_id) or not path.exists():\n        raise TaskError(f"no task \'{task_id}\' (there are: {\', \'.join(available())})")\n    return from_json(path.read_text(encoding="utf-8"), context)\n\n\ndef available() -> list[str]:\n    return sorted(p.stem for p in TASKS_DIR.glob("*.json"))\n', 'understand.py': '"""Understanding one thing the other person said: what they want (the intent) and which slot values they gave.\n\nTwo ways, used together:\n- rules: fast and sure for short replies ("हाँ", "नाही", "ढाई सौ", "कल शाम पाँच बजे") and for "don\'t call me again";\n- a small language model (any function prompt, schema -> JSON text, e.g. Ollama) for everything longer.\n\n    u = understand("हाँ है, ढाई सौ का", task, View(pending="in_stock"), llm)\n    u.intent == "answer";  u.values == {"in_stock": True, "price_inr": 250}\n\nEvery value the model gives is checked against its type (a price must be a number, a phone 10 digits, a day one of\nthe known days) and prices are checked against what the number reader found in the words, so a model\'s slip on\n"ढाई सौ" (250) can\'t reach the result. With no model, or when it fails, the rules answer alone.\n"""\n\nimport json\nimport re\nfrom dataclasses import dataclass, field\n\nfrom voicebot import numerals\nfrom voicebot.tasks import DAY_VALUES, Task\n\nINTENTS = {\n    "answer": "gives information or answers the assistant\'s question (a bare yes or no too)",\n    "confirm_yes": "agrees that the summary the assistant read back is right",\n    "confirm_no": "says the summary the assistant read back is wrong",\n    "dont_know": "does not know the answer",\n    "repeat": "did not hear or understand, asks to say it again",\n    "who_are_you": "asks who is calling, from where, or why",\n    "hold": "asks to wait a moment",\n    "busy": "is busy now and asks to call later",\n    "stop_calling": "asks not to be called again, or is not interested",\n    "wrong_number": "says this is the wrong number, shop or person",\n    "human": "wants to talk to a real person",\n    "question": "asks a question of their own",\n    "bye": "ends the call",\n    "other": "anything else: a greeting, small talk, unclear",\n}\n\nYES = {"हाँ", "हां", "हा", "जी", "हांजी", "हाँजी", "बिल्कुल", "बिलकुल", "ज़रूर", "जरूर", "सही", "yes", "yeah", "yep", "yup",\n       "sure", "ok", "okay", "ओके", "ओक", "haan", "han", "haa", "ha", "ji", "bilkul", "zaroor", "jaroor", "sahi", "हो",\n       "होय", "बरोबर", "नक्की", "correct", "right", "करेक्ट", "राइट", "है", "हैं", "आहे", "आहेत", "hai", "hain", "aahe",\n       "ahe", "hoy", "ho", "available", "अवेलेबल", "मिलेगा", "मिळेल", "milega", "milel", "यस", "येस", "ठीक", "theek",\n       "thik", "बराबर", "हम्म", "hmm", "हाँहाँ", "करते", "करता", "देते", "देता", "देतो", "करतो", "देंगे", "denge",\n       "karte", "kar", "dete", "कर", "दे", "देऊ", "करू", "चलेगा", "chalega", "चालेल"}\nNO = {"नहीं", "नही", "ना", "न", "नाही", "नाहीये", "नाहीत", "नको", "no", "nope", "nahi", "nahin", "nai", "naa", "नई",\n      "नहीँ", "गलत", "ग़लत", "चुकीचं", "चुकीचे", "चूक", "wrong", "खत्म", "ख़त्म", "संपलं", "संपला", "संपले", "आउट",\n      "galat", "nako", "nahiye", "नॉट", "not", "khatam", "sampla"}\nSTRONG_YES = {"हाँ", "हां", "हो", "होय", "जी", "yes", "yeah", "haan", "han", "ji", "hoy", "बिल्कुल", "बिलकुल", "ओके", "ok",\n              "okay", "यस", "हाँजी", "हांजी"}\nFILLER = {"जी", "भाई", "भैया", "साहब", "सर", "मैडम", "तो", "अरे", "हाँ", "बस", "ji", "bhai", "bhaiya", "sir", "madam", "toh",\n          "to", "arre", "are", "please", "प्लीज़", "प्लीज", "बरं", "ना", "न", "है", "हैं", "आहे", "का", "की", "वो", "वाला",\n          "बोलिए", "बोलो", "बताइए", "बताइये", "बताओ", "हैलो", "हेलो", "hello", "बोला", "सांगा", "हां", "हाँजी", "bolo",\n          "boliye", "batao", "bataiye", "हम्म", "आ", "गया", "गई", "आलो", "आले"}\nDONT_KNOW = ["पता नहीं", "पता नही", "नहीं पता", "नही पता", "मालूम नहीं", "मालूम नही", "माहीत नाही", "माहिती नाही",\n             "माहित नाही", "don\'t know", "dont know", "do not know", "not sure", "pata nahi", "pata nahin", "malum nahi",\n             "maahit nahi", "mahit nahi", "kya pata", "क्या पता", "आइडिया नहीं", "idea nahi", "no idea"]\nINTENT_PHRASES = {  # checked in this order\n    "stop_calling": ["कॉल मत", "फोन मत", "फ़ोन मत", "मत करना कॉल", "दोबारा मत", "दुबारा मत", "परेशान मत", "परेशान कर",\n                     "don\'t call", "dont call", "do not call", "stop calling", "not interested", "इंटरेस्टेड नहीं",\n                     "इंटरेस्ट नहीं", "interested nahi", "फोन करू नका", "कॉल करू नका", "फोन नका करू", "call mat",\n                     "phone mat", "dobara call mat", "मत करो कॉल", "कॉल नहीं करना", "रुचि नहीं"],\n    "wrong_number": ["रॉन्ग नंबर", "रॉंग नंबर", "राँग नंबर", "गलत नंबर", "ग़लत नंबर", "wrong number", "चुकीचा नंबर",\n                     "galat number", "rong number", "गलत जगह", "ये दुकान नहीं", "यह दुकान नहीं"],\n    "human": ["इंसान से", "किसी आदमी से", "मैनेजर से", "असली आदमी", "real person", "human", "किसी व्यक्ति से", "माणसाशी",\n              "operator", "ऑपरेटर", "agent se", "एजेंट से", "मालिक से बात", "आदमी से बात", "person se baat"],\n    "repeat": ["फिर से बोल", "फिर से बता", "फिर से कह", "दोबारा बोल", "दोबारा बता", "दुबारा बोल", "क्या बोला", "क्या बोले",\n               "क्या कहा", "सुनाई नहीं", "समझ नहीं आया", "समझा नहीं", "समझी नहीं", "पुन्हा सांगा", "परत सांगा",\n               "ऐकू आलं नाही", "ऐकू नाही", "कळलं नाही", "समजलं नाही", "repeat", "say again", "come again", "pardon",\n               "kya bola", "phir se", "dobara bol", "samajh nahi", "sunai nahi", "क्या?"],\n    "who_are_you": ["कौन बोल", "कौन है", "कौन हो", "आप कौन", "कोण बोलत", "कोण आहे", "तुम्ही कोण", "who is this",\n                    "who are you", "who\'s calling", "who is calling", "kaun bol", "aap kaun", "कहाँ से बोल", "कहां से बोल",\n                    "कुठून बोलत", "calling from where", "कहाँ से", "कौन सी कंपनी"],\n    "hold": ["एक मिनट", "एक मिनिट", "रुको", "रुकिए", "रुकिये", "होल्ड", "थांबा", "थांब", "wait", "hold on", "one minute",\n             "ruko", "ek minute", "ek min", "एक सेकंड", "एक सेकेंड", "जरा रुक", "ज़रा रुक", "दो मिनट"],\n    "busy": ["बिज़ी", "बिजी", "व्यस्त", "बाद में कॉल", "बाद में फोन", "बाद में बात", "नंतर फोन", "नंतर कॉल", "busy",\n             "call later", "baad mein call", "baad me call", "अभी टाइम नहीं", "अभी समय नहीं", "आत्ता वेळ नाही",\n             "वेळ नाही", "time nahi", "बाद में करना", "later"],\n    "bye": ["बाय", "bye", "रखता हूँ", "रखती हूँ", "रखता हूं", "ठेवतो", "ठेवते", "चलो रखो", "फोन रखो", "good bye", "goodbye"],\n}\nSTRONG_INTENTS = ("stop_calling", "wrong_number", "human")  # a clear phrase wins over the model\nINTENT_MAX_WORDS = {"busy": 10, "who_are_you": 10, "repeat": 8, "hold": 8, "bye": 6}  # longer: likely more than that\nQUESTION_WORDS = {"क्या", "कब", "कहाँ", "कहां", "कितना", "कितनी", "कितने", "कैसे", "क्यों", "कौन", "काय", "कधी", "कुठे",\n                  "किती", "कसे", "कसं", "का?", "what", "when", "where", "how", "why", "who", "kab", "kahan", "kaise", "kyun",\n                  "kitna"}\nNAME_DROP = {"मेरा", "मेरी", "नाम", "है", "हैं", "जी", "माझं", "माझे", "माझा", "नाव", "आहे", "my", "name", "is", "i\'m",\n             "am", "i", "mera", "naam", "hai", "patient", "मरीज़", "मरीज", "का", "की", "रुग्णाचं", "रुग्णाचे", "पेशेंट",\n             "this", "it\'s", "its", "the", "हाँ", "haan", "ji", "मैं", "main", "बोल", "रहा", "रही", "हूँ", "हूं", "बोलतोय",\n             "बोलतेय", "मी", "मिस्टर", "mr", "mrs", "श्री", "श्रीमती"}\nWORD = re.compile(r"[ऀ-ॿ]+|[A-Za-z\']+|\\d+|\\?")\n\n\n@dataclass\nclass View:\n    """What the understanding needs to know about the call so far."""\n    pending: str | None = None  # the slot the assistant just asked for, "confirm" (after the read-back), or None\n    last_bot: str = ""  # what the assistant just said\n    values: dict = field(default_factory=dict)  # the slot values found so far\n\n\n@dataclass\nclass Understood:\n    intent: str = "other"\n    values: dict = field(default_factory=dict)  # only the slots this turn gave\n    fact: str | None = None  # the fact asked about (a key of the task\'s facts)\n    said_no: bool = False  # a no in the reply (for a text slot: "nothing else")\n    source: str = "rules"  # "rules", "rules (fast)", "model", "model+rules"\n    fast: bool = False  # sure enough from the rules alone: the model is not asked\n    raw: str | None = None  # the model\'s reply\n    error: str | None = None  # why the model\'s reply was not used\n\n    def as_dict(self) -> dict:\n        return {"intent": self.intent, "values": self.values, "fact": self.fact, "source": self.source,\n                "error": self.error}\n\n\ndef words(text: str) -> list[str]:\n    return [w.lower() for w in WORD.findall(text or "")]\n\n\ndef has(text: str, phrases) -> bool:\n    low = " ".join((text or "").lower().split())\n    return any(p in low for p in phrases)\n\n\ndef yes_no(text: str) -> bool | None:\n    """True for a yes, False for a no (a no anywhere wins: "हाँ जी, नहीं है" is a no), None when neither."""\n    ws = words(text)\n    if has(text, DONT_KNOW):\n        return None\n    tag = {"ना", "न", "naa", "na"}  # "है ना" is a yes: these are a no only first or alone\n    if any(w in NO and (w not in tag or i == 0) for i, w in enumerate(ws)) or has(\n            text, ["not in stock", "out of stock", "नहीं है", "नाही आहे"]):\n        return False\n    if any(w in YES for w in ws):\n        return True\n    return None\n\n\ndef content_words(text: str) -> list[str]:\n    return [w for w in words(text) if w not in FILLER and w not in YES and w not in NO and w != "?"]\n\n\n# --- checking values ---------------------------------------------------------------------------------------------\ndef check(slot: dict, value):\n    """The value in its proper form for the slot\'s type, or None when it is not a valid value of that type."""\n    kind = slot["type"]\n    if value is None or value == "" or (isinstance(value, str) and value.strip().lower() in ("null", "none", "n/a")):\n        return None\n    if kind == "bool":\n        if isinstance(value, bool):\n            return value\n        if isinstance(value, str) and value.strip().lower() in ("true", "yes", "false", "no"):\n            return value.strip().lower() in ("true", "yes")\n        return None\n    if kind in ("money", "number"):\n        if isinstance(value, bool):\n            return None\n        if isinstance(value, str):\n            found = re.sub(r"[₹,\\s]|rs\\.?|/-", "", value.lower())\n            try:\n                value = float(found)\n            except ValueError:\n                return None\n        if not isinstance(value, (int, float)) or value != value:  # noqa: PLR0124 - NaN\n            return None\n        if kind == "money" and not 0 < value < 10_000_000:\n            return None\n        return int(value) if float(value).is_integer() else round(float(value), 2)\n    if kind == "phone":\n        digits = re.sub(r"\\D", "", str(value))\n        if digits.startswith("91") and len(digits) == 12:\n            digits = digits[2:]\n        if digits.startswith("0") and len(digits) == 11:\n            digits = digits[1:]\n        return digits if re.fullmatch(r"[6-9]\\d{9}", digits) else None\n    if kind == "day":\n        v = str(value).strip().lower().replace(" ", "_").replace("-", "_")\n        v = {"day_after_tomorrow": "day_after", "dayafter": "day_after"}.get(v, v)\n        return v if v in DAY_VALUES else numerals.day(str(value))\n    if kind == "time":\n        if isinstance(value, (int, float)) and not isinstance(value, bool) and 0 <= value < 24:\n            return f"{int(value):02d}:{round((value - int(value)) * 60):02d}"\n        m = re.fullmatch(r"\\s*(\\d{1,2})(?:[:.](\\d{2}))?\\s*", str(value))\n        if m and int(m.group(1)) < 24 and int(m.group(2) or 0) < 60:\n            return f"{int(m.group(1)):02d}:{int(m.group(2) or 0):02d}"\n        return numerals.time_of_day(str(value))\n    if kind == "enum":\n        v = str(value).strip().lower()\n        options = slot["options"]\n        if v in options:\n            return v\n        for key, option in options.items():\n            names = [str(option.get(lang, "")).lower() for lang in ("hi", "mr", "en")] + [\n                str(w).lower() for w in option.get("words", [])]\n            if v in names:\n                return key\n        return None\n    # text and name\n    v = " ".join(str(value).split()).strip(" .,!?।\\"\'")\n    limit = 40 if kind == "name" else 80\n    return v[:limit] if v else None\n\n\ndef enum_from_words(slot: dict, text: str):\n    ws = set(words(text))\n    low = (text or "").lower()\n    found = [key for key, option in slot["options"].items()\n             if any(str(w).lower() in ws or (" " in str(w) and str(w).lower() in low)\n                    for w in [option.get("hi"), option.get("mr"), option.get("en"), *option.get("words", [])] if w)]\n    return found[0] if len(found) == 1 else None\n\n\ndef trimmed(text: str) -> str:\n    """A text value without the filler around it: "हाँ, हैवेल्स का बत्तीस ए है" -> "हैवेल्स का बत्तीस ए"."""\n    ws = (text or "").strip(" .,!?।").split()\n    edge = FILLER | YES | {"है।", "आहे."}\n    while ws and ws[-1].strip(" .,!?।").lower() in edge:\n        ws.pop()\n    while ws and ws[0].strip(" .,!?।").lower() in edge:\n        ws.pop(0)\n    return " ".join(ws).strip(" .,!?।")\n\n\nNAME_AFTER = re.compile(r"(?:नाम|नाव|naam|name is|name\'s)\\s+(?:है\\s+|आहे\\s+|hai\\s+|is\\s+)?"\n                        r"([\\u0900-\\u097FA-Za-z]+(?:\\s+[\\u0900-\\u097FA-Za-z]+)?)", re.IGNORECASE)\n\n\ndef name_from(text: str) -> str | None:\n    if "?" in (text or "") or any(w in QUESTION_WORDS for w in words(text)):\n        return None\n    found = NAME_AFTER.search(text or "")\n    if found:\n        named = [w for w in found.group(1).split() if w.lower() not in NAME_DROP]\n        if named:\n            return " ".join(w.capitalize() if w.isascii() else w for w in named)\n    kept = [w for w in re.findall(r"[ऀ-ॿ]+|[A-Za-z]+", text or "") if w.lower() not in NAME_DROP]\n    if not kept or len(kept) > 4:\n        return None\n    return " ".join(w.capitalize() if w.isascii() else w for w in kept)\n\n\n# --- the rules -------------------------------------------------------------------------------------------------\ndef by_rules(text: str, task: Task, view: View) -> Understood:\n    u = Understood()\n    ws = words(text)\n    for intent, phrases in INTENT_PHRASES.items():\n        if has(text, phrases) and len(ws) <= INTENT_MAX_WORDS.get(intent, 99):\n            u.intent = intent\n            break\n    if u.intent == "other" and has(text, DONT_KNOW):\n        u.intent = "dont_know"\n    answer = yes_no(text)\n    asking = "?" in (text or "") or any(w in QUESTION_WORDS for w in ws)\n    if asking and answer is True and not any(w in STRONG_YES for w in ws):\n        answer = None  # "यूपीआई से पेमेंट कर सकते हैं क्या?" is a question, not a yes\n    u.said_no = answer is False\n    pending = task.slot.get(view.pending) if view.pending not in (None, "confirm") else None\n    if view.pending == "confirm" and u.intent == "other" and answer is not None:\n        u.intent = "confirm_yes" if answer else "confirm_no"\n    for slot in task.slots:\n        name, kind = slot["name"], slot["type"]\n        is_pending = pending is not None and name == pending["name"]\n        value = None\n        if kind == "bool" and is_pending and u.intent in ("other", "answer"):\n            value = answer\n        elif kind == "money" and (is_pending or sum(s["type"] == "money" for s in task.slots) == 1):\n            value = numerals.money(text, expected=is_pending)\n        elif kind == "number" and is_pending:\n            spans = numerals.find_numbers(text)\n            value = spans[0].value if spans else None\n        elif kind == "day":\n            value = numerals.day(text, fuzzy=is_pending)\n        elif kind == "time":\n            value = numerals.time_of_day(text)\n        elif kind == "phone":\n            value = numerals.phone(text)\n        elif kind == "enum":\n            value = enum_from_words(slot, text)\n        elif kind == "name" and (is_pending and u.intent == "other" or NAME_AFTER.search(text or "")):\n            value = name_from(text)\n        elif kind == "text" and is_pending and u.intent == "other" and not u.said_no:\n            value = trimmed(text) if content_words(text) else None\n        value = check(slot, value)\n        if value is not None:\n            u.values[name] = value\n    if u.intent == "other" and (u.values or (pending is not None and answer is not None)):\n        u.intent = "answer"\n    if u.intent == "other" and ("?" in (text or "") or any(w in QUESTION_WORDS for w in ws)) and not u.values:\n        u.intent = "question"\n    # sure enough without the model: a short reply that answers what was asked, or a clear request\n    short = len([w for w in ws if w != "?"]) <= 4\n    if short and pending is not None and pending["name"] in u.values and pending["type"] not in ("name", "text"):\n        u.fast = len(content_words(text)) <= 3 or pending["type"] in ("money", "time", "day", "phone")\n    else:\n        u.fast = (\n            (short and view.pending == "confirm" and u.intent in ("confirm_yes", "confirm_no")\n             and not content_words(text))\n            or u.intent in STRONG_INTENTS\n            or (short and u.intent in ("repeat", "hold", "who_are_you", "busy", "bye"))\n            or (short and pending is not None and pending["type"] == "text" and u.said_no\n                and not content_words(text)))\n    u.source = "rules (fast)" if u.fast else "rules"\n    return u\n\n\n# --- the model -------------------------------------------------------------------------------------------------\ndef describe(task: Task, slot: dict) -> str:\n    return task.say(slot["describe"], "en")\n\n\ndef field_line(task: Task, slot: dict) -> str:\n    kind = slot["type"]\n    shape = {\n        "bool": "true or false", "money": "a number of rupees", "number": "a number", "phone": "10 digits",\n        "day": "one of " + ", ".join(DAY_VALUES), "time": \'"HH:MM", 24-hour\', "text": "short text, as said",\n        "name": "the name, as said",\n    }.get(kind)\n    if kind == "enum":\n        shape = "one of " + ", ".join(f\'"{k}"\' for k in slot["options"])\n    return f"- {slot[\'name\']}: {shape}. {describe(task, slot)}"\n\n\ndef schema(task: Task) -> dict:\n    """The JSON schema of the model\'s reply (Ollama\'s "format"): the intent, the fact, and every slot (null when\n    this turn does not say it)."""\n    props = {"intent": {"type": "string", "enum": list(INTENTS)}}\n    for slot in task.slots:\n        kind = slot["type"]\n        if kind == "bool":\n            props[slot["name"]] = {"type": ["boolean", "null"]}\n        elif kind in ("money", "number"):\n            props[slot["name"]] = {"type": ["number", "null"]}\n        elif kind == "enum":\n            props[slot["name"]] = {"type": ["string", "null"], "enum": [*slot["options"], None]}\n        elif kind == "day":\n            props[slot["name"]] = {"type": ["string", "null"], "enum": [*DAY_VALUES, None]}\n        else:\n            props[slot["name"]] = {"type": ["string", "null"]}\n    props["fact"] = ({"type": ["string", "null"], "enum": [*task.facts, None]} if task.facts else {"type": "null"})\n    return {"type": "object", "properties": props, "required": list(props)}\n\n\ndef prompt(task: Task, view: View, text: str) -> str:\n    """The instructions for the model: the call, what was just asked, the fields, and the line to read."""\n    if view.pending == "confirm":\n        waiting = ("The assistant read back a summary and asked if it is right: intent confirm_yes if they agree, "\n                   "confirm_no if not. If they correct a value, give the corrected value too.")\n    elif view.pending in task.slot:\n        waiting = f"The assistant was asking for: {view.pending} ({describe(task, task.slot[view.pending])})."\n    else:\n        waiting = "The assistant was not asking for anything in particular."\n    known = {k: v for k, v in view.values.items() if v is not None}\n    facts = "".join(f"\\n   {k}: {task.say(f[\'about\'], \'en\')}" for k, f in task.facts.items()) or " (none)"\n    intents = "".join(f"\\n   {k}: {v}" for k, v in INTENTS.items())\n    fields = "\\n".join(field_line(task, s) for s in task.slots)\n    return f"""You read one turn of a phone call and fill in a form. Reply with JSON only.\n\nThe call: {task.purpose()}\nThe other person is {task.who()}.\nThe assistant just said: "{view.last_bot}"\n{waiting}\nKnown so far: {json.dumps(known, ensure_ascii=False) if known else "nothing"}\n\nThe other person now said: "{numerals.annotate(text)}"\n\nFields (null when this turn does not say it):\n{fields}\n- intent: one of:{intents}\n- fact: when they ask a question, which of these it is about, else null:{facts}\n\nRules:\n- A bare yes or no (हाँ, जी, हो, होय, है, नहीं, नाही, yes, no) answers the assistant\'s last question.\n- Numbers said in words have their digits added, like "ढाई सौ (=250)": use those digits.\n- Fill in only what this turn says. Never guess. Keep text values short, in the words they used."""\n\n\ndef first_json(raw: str) -> dict:\n    raw = (raw or "").strip()\n    raw = re.sub(r"^```(?:json)?\\s*|\\s*```$", "", raw)\n    start, end = raw.find("{"), raw.rfind("}")\n    if start < 0 or end <= start:\n        raise ValueError("no JSON object in the reply")\n    found = json.loads(raw[start:end + 1])\n    if not isinstance(found, dict):\n        raise TypeError("the reply is not a JSON object")\n    return found\n\n\ndef from_model(raw: str, task: Task) -> Understood:\n    data = first_json(raw)\n    u = Understood(source="model", raw=raw)\n    intent = str(data.get("intent") or "other").strip().lower()\n    u.intent = intent if intent in INTENTS else "other"\n    fact = data.get("fact")\n    u.fact = fact if isinstance(fact, str) and fact in task.facts else None\n    for slot in task.slots:\n        value = check(slot, data.get(slot["name"]))\n        if value is not None:\n            u.values[slot["name"]] = value\n    return u\n\n\nTYPED = ("money", "number", "day", "time", "phone")  # the number reader decides these when it finds one\nSOFT_INTENTS = ("busy", "hold", "repeat", "who_are_you", "bye", "dont_know")  # a clear phrase beats the model\'s guess\nQUIET_INTENTS = SOFT_INTENTS + STRONG_INTENTS + ("question",)  # no new values without the rules\' support\nNOTHING = {"कुछ", "कोई", "काही", "काहीच", "kuch", "koi", "nothing", "नथिंग", "भी", "bhi", "पण"}\n\n\ndef junk_text(value: str, text: str) -> bool:\n    """A text value that says nothing: "नहीं, कुछ नहीं है", "खत्म हो गया", or a "पता नहीं" line."""\n    rest = [w for w in content_words(value) if w not in NOTHING]\n    return not rest or has(value, DONT_KNOW) or yes_no(value) is False and len(rest) <= 3\n\n\ndef reconcile(model: Understood, rules: Understood, text: str, task: Task, view: View) -> Understood:\n    """The model\'s understanding, checked against the words and the rules. Small models repeat every value they\n    were shown (and garble a phone number while at it), flip a yes they were not asked about, and put the whole\n    line into a text field; so a value is kept only with evidence:\n    - prices, numbers, days, times and phones: the number reader\'s value when it found one; a number nobody said\n      or a phone it can\'t find is dropped;\n    - a value already known changes only when it is the question asked, during the read-back, or clearly said;\n    - text (e.g. the other brand) only for the question asked, and not when it is only a no or "पता नहीं";\n    - a question, "बाद में कॉल करो", "पता नहीं"...: only what the rules also found."""\n    u = model\n    u.said_no = rules.said_no\n    if rules.intent in STRONG_INTENTS or (rules.intent in SOFT_INTENTS and view.pending not in u.values):\n        u.intent = rules.intent\n    said = {s.value for s in numerals.find_numbers(text, colloquial=True) + numerals.find_numbers(text)}\n    correcting = view.pending in ("confirm", "fix")\n    for slot in task.slots:\n        name, kind = slot["name"], slot["type"]\n        mine, theirs, known = u.values.get(name), rules.values.get(name), view.values.get(name)\n        asked = name == view.pending\n        if mine is not None:\n            if kind in TYPED and theirs is not None:\n                mine = theirs\n            elif kind in ("money", "number") and mine not in said:\n                mine = None  # a number the person never said\n            elif kind == "phone" or (kind in ("day", "time") and not asked and not correcting):\n                mine = None\n            if kind == "text" and (not asked or junk_text(mine, text)):\n                mine = None\n            if kind == "name" and not asked and theirs is None:\n                mine = None\n            if known is not None and mine != known and not asked and not correcting and theirs is None:\n                mine = None  # a known value changes only when asked, at the read-back, or clearly said\n            if u.intent in QUIET_INTENTS and theirs is None:\n                mine = None\n            if (kind == "bool" and mine is True and theirs is False and asked and rules.said_no\n                    and len(words(text)) <= 3):  # a short "नहीं" can\'t be a yes\n                mine = False\n        elif theirs is not None and not (kind == "bool" and not asked):\n            if kind not in ("text", "name") or (asked and u.intent == "answer"):\n                mine = theirs\n        if mine != u.values.get(name):\n            u.source = "model+rules"\n        if mine is None:\n            u.values.pop(name, None)\n        else:\n            u.values[name] = mine\n    if view.pending in u.values and u.intent not in STRONG_INTENTS and not u.fact:\n        u.intent = "answer"  # it answers what was asked, whatever the model called it\n    if u.intent == "other" and u.values:\n        u.intent = "answer"\n    if view.pending == "confirm" and u.intent == "answer" and not u.values and rules.intent in ("confirm_yes",\n                                                                                                "confirm_no"):\n        u.intent = rules.intent\n    return u\n\n\ndef understand(text: str, task: Task, view: View, llm=None) -> Understood:\n    """What the line means for this call. llm(prompt, schema) -> the model\'s JSON text; None: rules only."""\n    rules = by_rules(text, task, view)\n    if llm is None or rules.fast:\n        return rules\n    try:\n        raw = llm(prompt(task, view, text), schema(task))\n        model = from_model(raw, task)\n    except Exception as e:  # noqa: BLE001 - a model failure must not stop the call: the rules answer\n        rules.error = f"model: {type(e).__name__}: {e}"[:300]\n        return rules\n    return reconcile(model, rules, text, task, view)\n\n', 'engine.py': '"""The conversation: one call, turn by turn, for any task (voicebot/tasks/*.json).\n\n    call = Call(load("shop_stock"), llm=my_model)      llm: None = rules only\n    reply = call.start()                                the greeting and the first question\n    reply = call.hear("हाँ है, ढाई सौ का")               what the other person said (from speech to text)\n    reply.text, reply.speech, reply.done                what to show, what the voice says, whether the call is over\n    call.result()                                       the values found, the outcome, every turn\n\nThe model only reads (what was said, which values); what the bot says next is decided here, by plain rules, from\nthe task: ask the next missing slot, say it again when not understood (twice at most, then move on), read\neverything back at the end and fix what the person corrects. So the bot never makes up a price, a promise or a\nfact: it only says the task\'s own texts and the values the person gave.\n\nOutcomes: done (every needed slot found), partial (the call ended with some missing), callback (busy, call later),\ndo_not_call, wrong_number, handoff (wants a person), no_answer (silence), too_long.\n"""\n\nimport time\nfrom dataclasses import dataclass, field\n\nfrom voicebot import lang as langs\nfrom voicebot.numerals import day_words, digit_words, money_words, time_words\nfrom voicebot.speech_text import clean_heard, for_voice\nfrom voicebot.tasks import Task, render, text_of\nfrom voicebot.understand import Understood, View, content_words, understand, words\n\nMAX_ASKS = 2  # a slot is asked at most this many times, then left as unknown\nMAX_READBACKS = 3\nMAX_SILENCES = 3\n\nPHRASES = {\n    "ok": {"hi": "ठीक है।", "mr": "ठीक आहे.", "en": "Okay."},\n    "repeat": {"hi": "जी, फिर से कहती हूँ।", "mr": "हो, पुन्हा सांगते.", "en": "Sure, let me say that again."},\n    "hold": {"hi": "जी, ज़रूर, मैं रुकती हूँ।", "mr": "हो, नक्की, मी थांबते.", "en": "Sure, I\'ll wait."},\n    "not_understood": {"hi": "माफ़ कीजिए, ठीक से समझ नहीं आया।", "mr": "माफ करा, नीट समजलं नाही.",\n                       "en": "Sorry, I didn\'t quite get that."},\n    "silence": {"hi": "हैलो? क्या आप मुझे सुन पा रहे हैं?", "mr": "हॅलो? तुम्हाला ऐकू येतंय का?",\n                "en": "Hello? Can you hear me?"},\n    "back": {"hi": "जी।", "mr": "हो.", "en": "Sure."},\n    "dont_know": {"hi": "कोई बात नहीं।", "mr": "काही हरकत नाही.", "en": "No problem."},\n    "busy": {"hi": "कोई बात नहीं, हम बाद में कॉल कर लेंगे। धन्यवाद!",\n             "mr": "काही हरकत नाही, आम्ही नंतर फोन करू. धन्यवाद!", "en": "No problem, we\'ll call back later. Thank you!"},\n    "stop": {"hi": "माफ़ कीजिए, अब हम आपको कॉल नहीं करेंगे। धन्यवाद!",\n             "mr": "माफ करा, आम्ही आता तुम्हाला फोन करणार नाही. धन्यवाद!",\n             "en": "Sorry about that, we won\'t call you again. Thank you!"},\n    "wrong_number": {"hi": "माफ़ कीजिए, गलत नंबर लग गया। धन्यवाद!", "mr": "माफ करा, चुकीचा नंबर लागला. धन्यवाद!",\n                     "en": "Sorry, wrong number. Thank you!"},\n    "human": {"hi": "ठीक है, हमारी टीम का कोई व्यक्ति आपको जल्द ही कॉल करेगा। धन्यवाद!",\n              "mr": "ठीक आहे, आमच्या टीममधील कोणीतरी तुम्हाला लवकरच फोन करेल. धन्यवाद!",\n              "en": "Okay, a person from our team will call you shortly. Thank you!"},\n    "unknown_fact": {"hi": "ये मुझे अभी पता नहीं, मैं टीम से पूछकर बता दूँगी।",\n                     "mr": "हे मला आत्ता माहीत नाही, मी टीमला विचारून सांगते.",\n                     "en": "I don\'t know that right now, I\'ll check with the team."},\n    "readback": {"hi": "तो मैंने नोट किया: {items}। सही है?", "mr": "तर मी नोंद केलं: {items}. बरोबर आहे का?",\n                 "en": "So I have: {items}. Is that right?"},\n    "fix": {"hi": "माफ़ कीजिए, क्या गलत है? सही बता दीजिए।", "mr": "माफ करा, काय चुकलं? बरोबर सांगा.",\n            "en": "Sorry, what is wrong? Please tell me the right one."},\n    "no_answer": {"hi": "लगता है आवाज़ नहीं आ रही। हम बाद में कॉल करेंगे, धन्यवाद!",\n                  "mr": "आवाज येत नाहीये असं वाटतं. आम्ही नंतर फोन करू, धन्यवाद!",\n                  "en": "I can\'t hear you, it seems. We\'ll call back later, thank you!"},\n    "bye": {"hi": "ठीक है, धन्यवाद! नमस्ते।", "mr": "ठीक आहे, धन्यवाद! नमस्कार.", "en": "Okay, thank you! Goodbye."},\n    "too_long": {"hi": "आपके समय के लिए धन्यवाद, हम बाद में फिर संपर्क करेंगे।",\n                 "mr": "तुमच्या वेळेबद्दल धन्यवाद, आम्ही नंतर पुन्हा संपर्क करू.",\n                 "en": "Thanks for your time, we\'ll get back to you later."},\n    "yes": {"hi": "हाँ", "mr": "हो", "en": "yes"},\n    "no": {"hi": "नहीं", "mr": "नाही", "en": "no"},\n    "and": {"hi": ", ", "mr": ", ", "en": ", "},\n}\n\n\ndef phrase(name: str, lang: str) -> str:\n    return text_of(PHRASES[name], lang)\n\n\n@dataclass\nclass Reply:\n    text: str  # what is shown\n    speech: str  # what the voice says (numbers and short forms in words)\n    lang: str\n    done: bool = False\n    outcome: str | None = None\n\n\n@dataclass\nclass Turn:\n    who: str  # "bot" or "person"\n    text: str\n    lang: str\n    seconds: float = 0.0\n    understood: dict | None = None\n    note: str = ""\n\n\n@dataclass\nclass Call:\n    task: Task\n    lang: str | None = None\n    llm: object = None  # llm(prompt, schema) -> JSON text; None: rules only\n    aside: object = None  # aside(question, lang) -> a short answer to an off-script question; None: "I\'ll check"\n    max_turns: int = 16\n    values: dict = field(default_factory=dict)  # slot -> value\n    status: dict = field(default_factory=dict)  # slot -> "filled", "unknown" (asked, no answer) or "none" (no)\n    asks: dict = field(default_factory=dict)  # slot -> times asked\n    pending: str | None = None  # the slot just asked, "confirm" (after the read-back), "fix", or None\n    turns: list = field(default_factory=list)\n    done: bool = False\n    outcome: str | None = None\n    readbacks: int = 0\n    silences: int = 0\n    heard: int = 0\n    last_question: tuple = ("", "")  # (text, speech) of the last question, to say again\n    holding: bool = False  # asked to wait: the next "हाँ, बोलिए" is them coming back, not an answer\n    _maybe_lang: str | None = None\n\n    def __post_init__(self):\n        if self.lang not in self.task.languages:\n            self.lang = self.task.default_language\n\n    # --- what the bot says ---------------------------------------------------------------------------------------\n    def _say(self, template, extra: dict | None = None) -> tuple[str, str]:\n        """A task text in the call\'s language: (shown, spoken)."""\n        text = self.task.say(template, self.lang, extra)\n        return text, for_voice(text, self.lang)\n\n    def _phrase(self, name: str, items: tuple[str, str] | None = None) -> tuple[str, str]:\n        p = phrase(name, self.lang)\n        if items is None:\n            return p, for_voice(p, self.lang)\n        return render(p, {"items": items[0]}), for_voice(render(p, {"items": items[1]}), self.lang)\n\n    def show(self, slot: dict, value) -> tuple[str, str]:\n        """A value as shown and as spoken: ₹220 / "दो सौ बीस रुपये", 16:30 / "शाम साढ़े चार बजे", ..."""\n        kind, lang = slot["type"], self.lang\n        if kind == "money":\n            return f"₹{value}", money_words(value, lang)\n        if kind == "time":\n            words = time_words(value, lang)\n            return (words if lang != "en" else f"{value}"), words\n        if kind == "day":\n            return day_words(value, lang), day_words(value, lang)\n        if kind == "phone":\n            return f"{value[:5]} {value[5:]}", digit_words(value, lang)\n        if kind == "enum":\n            label = text_of(slot["options"][value], lang)\n            return label, label\n        if kind == "bool":\n            return phrase("yes" if value else "no", lang), phrase("yes" if value else "no", lang)\n        return str(value), str(value)\n\n    def _ask(self, name: str) -> tuple[str, str]:\n        self.asks[name] = self.asks.get(name, 0) + 1\n        self.pending = name\n        self.last_question = self._say(self.task.slot[name]["ask"])\n        return self.last_question\n\n    def _aside(self, question: str) -> tuple[str, str]:\n        """An off-script question: a short answer from the model (general knowledge only; the business\'s own\n        details are never made up), else "I\'ll check with the team"."""\n        if self.aside is not None:\n            try:\n                answer = self.aside(question, self.lang)\n            except Exception:  # noqa: BLE001 - the call goes on without it\n                answer = None\n            if answer:\n                return answer, for_voice(answer, self.lang)\n        return self._phrase("unknown_fact")\n\n    def _readback(self) -> tuple[str, str]:\n        shown, spoken = [], []\n        for slot in self.task.slots:\n            name = slot["name"]\n            if not self.task.applies(slot, self.values) or self.status.get(name) != "filled":\n                continue\n            value = self.values[name]\n            template = slot.get("readback")\n            if slot["type"] == "bool" and isinstance(template, dict) and "true" in template:\n                template = template["true" if value else "false"]\n            elif template is None:\n                template = {"hi": f"{name}: {{value}}", "mr": f"{name}: {{value}}", "en": f"{name}: {{value}}"}\n            a, b = self.show(slot, value)\n            shown.append(self.task.say(template, self.lang, {"value": a}))\n            spoken.append(self.task.say(template, self.lang, {"value": b}))\n        if not shown:\n            return "", ""\n        self.readbacks += 1\n        self.pending = "confirm"\n        joiner = phrase("and", self.lang)\n        self.last_question = self._phrase("readback", (joiner.join(shown), joiner.join(spoken)))\n        return self.last_question\n\n    # --- the flow --------------------------------------------------------------------------------------------------\n    def next_slot(self) -> str | None:\n        """The next slot to ask: needed now (its "when" holds), not answered, asked fewer than MAX_ASKS times."""\n        for slot in self.task.slots:\n            name = slot["name"]\n            if not self.task.applies(slot, self.values) or name in self.status:\n                continue\n            if self.asks.get(name, 0) >= MAX_ASKS:\n                self.status[name] = "unknown"\n                continue\n            return name\n        return None\n\n    def missing(self) -> list[str]:\n        """The needed slots (required, and their "when" holds) with no value."""\n        return [s["name"] for s in self.task.slots if s.get("required", True) and self.task.applies(s, self.values)\n                and self.status.get(s["name"]) != "filled"]\n\n    def _reply(self, pieces: list[tuple[str, str]], done: bool = False, outcome: str | None = None,\n               started: float | None = None) -> Reply:\n        pieces = [p for p in pieces if p and p[0]]\n        text = " ".join(p[0] for p in pieces)\n        speech = " ".join(p[1] for p in pieces)\n        if done:\n            self.done, self.outcome, self.pending = True, outcome, None\n        self.turns.append(Turn("bot", text, self.lang, round(time.perf_counter() - started, 3) if started else 0.0,\n                               note=outcome or ""))\n        return Reply(text, speech, self.lang, done, outcome)\n\n    def _end(self, pieces, outcome, started=None) -> Reply:\n        return self._reply(pieces, done=True, outcome=outcome, started=started)\n\n    def _finish(self, pieces, started=None) -> Reply:\n        """Every slot is done: the closing (and the outcome: done, or partial when a needed one is missing)."""\n        closing = self.task.closing(self.values, self.lang)\n        return self._end(pieces + [(closing, for_voice(closing, self.lang))], "partial" if self.missing() else "done",\n                         started)\n\n    def _continue(self, pieces, started=None) -> Reply:\n        """Ask the next slot; with none left, read back (once the task wants it) or finish."""\n        name = self.next_slot()\n        if name:\n            return self._reply(pieces + [self._ask(name)], started=started)\n        if self.task.confirm and self.readbacks < MAX_READBACKS and any(\n                v == "filled" for v in self.status.values()):\n            readback = self._readback()\n            if readback[0]:\n                return self._reply(pieces + [readback], started=started)\n        return self._finish(pieces, started)\n\n    def start(self) -> Reply:\n        started = time.perf_counter()\n        greeting = self._say(self.task.spec["greeting"])\n        return self._continue([greeting], started)\n\n    def _switch_language(self, text: str) -> None:\n        """Answer in the language the person speaks: switch after two lines in it, or one clear line."""\n        found, score = langs.detect(text, default=self.lang)\n        if score == 0 or found not in self.task.languages:\n            return\n        if found != self.lang and (score >= 2 or self._maybe_lang == found):\n            self.lang = found\n        self._maybe_lang = found\n\n    def _apply(self, u: Understood) -> tuple[list[str], tuple[str, str] | None]:\n        """Put the understood values in; returns the slots that changed, and the task\'s "invalid" text (spoken)\n        when a value breaks the task\'s rules (a Sunday when the clinic is closed)."""\n        changed = []\n        for name, value in u.values.items():\n            slot = self.task.slot[name]\n            bad = self._invalid(slot, value)\n            if bad:\n                self.status.pop(name, None)\n                self.values.pop(name, None)\n                self.pending = name\n                return changed, self._say(slot.get("invalid") or PHRASES["not_understood"])\n            if self.values.get(name) != value or self.status.get(name) != "filled":\n                changed.append(name)\n            self.values[name] = value\n            self.status[name] = "filled"\n        return changed, None\n\n    @staticmethod\n    def _invalid(slot: dict, value) -> bool:\n        rules = slot.get("valid") or {}\n        if value in (rules.get("not") or []):\n            return True\n        low, high = rules.get("min"), rules.get("max")\n        try:\n            return (low is not None and value < low) or (high is not None and value > high)\n        except TypeError:\n            return False\n\n    def hear(self, text: str, heard_seconds: float = 0.0) -> Reply:\n        """The next thing the other person said; returns what the bot says back."""\n        if self.done:\n            return Reply("", "", self.lang, True, self.outcome)\n        started = time.perf_counter()\n        text = clean_heard(text)\n        if not text:\n            self.silences += 1\n            self.turns.append(Turn("person", "", self.lang, heard_seconds, note="silence"))\n            if self.silences >= MAX_SILENCES:\n                return self._end([self._phrase("no_answer")], "no_answer", started)\n            return self._reply([self._phrase("silence")] + ([self.last_question] if self.silences > 1 else []),\n                               started=started)\n        self.silences = 0\n        self.heard += 1\n        self._switch_language(text)\n        view = View(pending=self.pending, last_bot=next((t.text for t in reversed(self.turns) if t.who == "bot"), ""),\n                    values=dict(self.values))\n        u = understand(text, self.task, view, self.llm)\n        self.turns.append(Turn("person", text, self.lang, heard_seconds, understood=u.as_dict()))\n        if self.heard > self.max_turns:\n            return self._end([self._phrase("too_long")], "too_long", started)\n        if u.intent == "stop_calling":\n            return self._end([self._phrase("stop")], "do_not_call", started)\n        if u.intent == "wrong_number":\n            return self._end([self._phrase("wrong_number")], "wrong_number", started)\n        if u.intent == "human":\n            return self._end([self._phrase("human")], "handoff", started)\n        if u.intent == "busy" and not u.values:\n            return self._end([self._phrase("busy")], "callback", started)\n        if u.intent == "bye" and not u.values:\n            if not self.missing():\n                return self._finish([], started)\n            return self._end([self._phrase("bye")], "partial", started)\n        if u.intent == "repeat" and not u.values:\n            return self._reply([self._phrase("repeat"), self.last_question], started=started)\n        if u.intent == "hold" and not u.values:\n            self.holding = True\n            return self._reply([self._phrase("hold")], started=started)\n        if self.holding:\n            self.holding = False\n            if len(words(text)) <= 3 and set(u.values) <= {self.pending} and not content_words(text):\n                return self._reply([self._phrase("back"), self.last_question], started=started)\n        if u.intent == "who_are_you" and not u.values:\n            return self._reply([self._say(self.task.spec["greeting"]), self.last_question], started=started)\n\n        pending = self.pending\n        changed, invalid = self._apply(u)\n        if invalid:\n            return self._reply([invalid, self._ask(self.pending)], started=started)\n        pieces = []\n        fact = self.task.facts.get(u.fact) if u.fact else None\n        if fact:\n            pieces.append(self._say(fact["answer"]))\n        elif u.intent == "question" and not changed:\n            pieces.append(self._aside(text))\n        if pending in ("confirm", "fix"):\n            if changed:\n                self.readbacks = min(self.readbacks, MAX_READBACKS - 1)\n                return self._continue(pieces + [self._phrase("ok")], started)\n            if u.intent == "confirm_yes" or (u.intent == "answer" and not u.said_no and pending == "fix"):\n                return self._finish(pieces, started)\n            if u.intent == "confirm_no" or u.said_no:\n                if self.readbacks >= MAX_READBACKS:\n                    return self._finish(pieces, started)\n                self.pending = "fix"\n                self.last_question = self._phrase("fix")\n                return self._reply(pieces + [self.last_question], started=started)\n            return self._reply(pieces + ([] if pieces else [self._phrase("not_understood")]) + [self.last_question],\n                               started=started)\n        slot = self.task.slot.get(pending) if pending else None\n        if slot is not None and pending not in changed and self.status.get(pending) is None:\n            if u.intent == "dont_know":\n                self.status[pending] = "unknown"\n                pieces.append(self._phrase("dont_know"))\n            elif slot["type"] in ("text", "name") and u.said_no and not u.values:\n                self.status[pending] = "none"  # "no other brand"\n            elif not pieces and not changed:\n                pieces.append(self._phrase("not_understood"))\n        elif changed and not pieces:\n            pieces.append(self._phrase("ok"))\n        return self._continue(pieces, started)\n\n    # --- the result --------------------------------------------------------------------------------------------------\n    def result(self) -> dict:\n        """What the call found: the values of the slots that apply (None: not found), each slot\'s status, the\n        outcome, the language and every turn."""\n        values = {s["name"]: (self.values.get(s["name"]) if self.task.applies(s, self.values) else None)\n                  for s in self.task.slots}\n        return {"task": self.task.id, "outcome": self.outcome, "values": values,\n                "status": {k: v for k, v in self.status.items()}, "missing": self.missing(), "lang": self.lang,\n                "turns": [t.__dict__ for t in self.turns]}\n', 'evaluate.py': '"""Play the test calls (voicebot/scenarios/*.jsonl) against the bot and score them.\n\nA test call is a pretend person: for each thing the bot asks, the line they say back ("replies", by the slot the\nbot asked for, "confirm" after the read-back, "fix" when it asks what is wrong, "*" for anything else), and lines\nsaid first ("first", like "कौन बोल रहा है?"). A list gives the line for the 1st, 2nd, ... time it is asked. The call\npasses when the outcome and every expected value match ("contains": a text value holding any of the words).\n\n    python -m voicebot.evaluate                    every test call, rules only (no model): a baseline\n    python -m voicebot.evaluate --show shop-hi-who    one call, turn by turn\n\nThe same code runs on Kaggle with a model (llm) and, for the spoken test, with say_and_hear: the person\'s line\ngoes through the voice and speech to text before the bot hears it.\n"""\n\nimport argparse\nimport json\nimport sys\nimport time\nfrom pathlib import Path\n\nHERE = Path(__file__).parent\nif sys.path and Path(sys.path[0]).resolve() == HERE.resolve():  # run as a script: don\'t shadow standard modules\n    sys.path.pop(0)\nsys.path.insert(0, str(HERE.parent))\n\nfrom voicebot.engine import Call\nfrom voicebot.tasks import load\n\nSCENARIOS = HERE / "scenarios"\n\n\ndef scenarios(task: str | None = None) -> list[dict]:\n    """Every test call (of one task, when given), with its task id filled in."""\n    out = []\n    for path in sorted(SCENARIOS.glob("*.jsonl")):\n        if task and path.stem != task:\n            continue\n        for line in path.read_text(encoding="utf-8").splitlines():\n            if line.strip():\n                row = json.loads(line)\n                row.setdefault("task", path.stem)\n                out.append(row)\n    return out\n\n\nclass Person:\n    """The pretend person of a test call."""\n\n    def __init__(self, scenario: dict):\n        self.first = list(scenario.get("first") or [])\n        self.replies = scenario.get("replies") or {}\n        self.times: dict[str, int] = {}\n\n    def line(self, pending: str | None) -> str:\n        if self.first:\n            return self.first.pop(0)\n        k = pending if pending in self.replies else "*"\n        said = self.replies.get(k, "")\n        n = self.times.get(k, 0)\n        self.times[k] = n + 1\n        if isinstance(said, list):\n            return said[min(n, len(said) - 1)] if said else ""\n        return said\n\n\ndef matches(scenario: dict, result: dict) -> list[str]:\n    """What the call got wrong ([]: it passed)."""\n    expect = scenario.get("expect") or {}\n    wrong = []\n    if expect.get("outcome") and result["outcome"] != expect["outcome"]:\n        wrong.append(f"outcome {result[\'outcome\']} (want {expect[\'outcome\']})")\n    if expect.get("lang") and result["lang"] != expect["lang"]:\n        wrong.append(f"language {result[\'lang\']} (want {expect[\'lang\']})")\n    for name, want in (expect.get("values") or {}).items():\n        got = result["values"].get(name)\n        if got != want and not (isinstance(got, (int, float)) and isinstance(want, (int, float))\n                                and not isinstance(got, bool) and abs(got - want) < 0.01):\n            wrong.append(f"{name} {got!r} (want {want!r})")\n    for name, words in (expect.get("contains") or {}).items():\n        got = str(result["values"].get(name) or "").lower()\n        if not any(w.lower() in got for w in words):\n            wrong.append(f"{name} {result[\'values\'].get(name)!r} (want one of {words})")\n    return wrong\n\n\ndef play(scenario: dict, llm=None, say_and_hear=None, on_bot=None, max_lines: int = 20) -> dict:\n    """One test call. say_and_hear(line, call) -> (what speech to text heard, seconds): the spoken test (call.lang,\n    call.turns: the bot\'s language and what it said); None: the bot reads the person\'s line as typed.\n    on_bot(reply): called with each thing the bot says (to make its audio)."""\n    task = load(scenario["task"], scenario.get("context"))\n    call = Call(task, lang=scenario.get("lang"), llm=llm)\n    person = Person(scenario)\n    started = time.perf_counter()\n    reply = call.start()\n    lines = []\n    while True:\n        if on_bot is not None:\n            on_bot(reply)\n        if reply.done or len(lines) >= max_lines:\n            break\n        said = person.line(call.pending)\n        heard, seconds = (said, 0.0) if say_and_hear is None or not said else say_and_hear(said, call)\n        lines.append({"said": said, "heard": heard})\n        reply = call.hear(heard, seconds)\n    result = call.result()\n    result["wrong"] = matches(scenario, result)\n    result["id"] = scenario["id"]\n    result["tags"] = scenario.get("tags", [])\n    result["lines"] = lines\n    result["seconds"] = round(time.perf_counter() - started, 2)\n    understood = [t["understood"] for t in result["turns"] if t.get("understood")]\n    result["model_calls"] = sum(u["source"].startswith("model") or bool(u.get("error")) for u in understood)\n    result["model_errors"] = sum(bool(u.get("error")) for u in understood)\n    return result\n\n\ndef summary(results: list[dict]) -> dict:\n    """Pass rates overall, per task and per tag."""\n    def rate(rows):\n        return {"passed": sum(not r["wrong"] for r in rows), "total": len(rows)}\n\n    tasks = sorted({r["task"] for r in results})\n    tags = sorted({t for r in results for t in r["tags"]})\n    return {"all": rate(results), "tasks": {t: rate([r for r in results if r["task"] == t]) for t in tasks},\n            "tags": {t: rate([r for r in results if t in r["tags"]]) for t in tags},\n            "model_calls": sum(r["model_calls"] for r in results),\n            "model_errors": sum(r["model_errors"] for r in results),\n            "person_lines": sum(len(r["lines"]) for r in results)}\n\n\ndef summary_lines(results: list[dict], label: str = "") -> list[str]:\n    s = summary(results)\n    a = s["all"]\n    lines = [f"{label}{a[\'passed\']} of {a[\'total\']} test calls passed"\n             + "".join(f"; {t} {v[\'passed\']}/{v[\'total\']}" for t, v in s["tasks"].items())]\n    lines.append("   by kind: " + ", ".join(f"{t} {v[\'passed\']}/{v[\'total\']}" for t, v in s["tags"].items()))\n    if s["model_calls"]:\n        lines.append(f"   the model was asked for {s[\'model_calls\']} of {s[\'person_lines\']} lines "\n                     f"({s[\'model_errors\']} failed: the rules answered)")\n    for r in results:\n        if r["wrong"]:\n            lines.append(f"   FAILED {r[\'id\']}: {\'; \'.join(r[\'wrong\'])}")\n    return lines\n\n\ndef transcript(result: dict) -> list[str]:\n    out = []\n    for t in result["turns"]:\n        if t["who"] == "bot":\n            out.append(f"   BOT: {t[\'text\']}")\n        else:\n            u = t.get("understood") or {}\n            out.append(f"   YOU: {t[\'text\'] or \'(silence)\'}   → {u.get(\'intent\')} {u.get(\'values\')} [{u.get(\'source\')}]")\n    return out\n\n\ndef main() -> None:\n    args = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    args.add_argument("--task", help="only this task\'s test calls")\n    args.add_argument("--show", metavar="ID", help="print one test call turn by turn")\n    opts = args.parse_args()\n    rows = scenarios(opts.task)\n    if opts.show:\n        rows = [r for r in rows if r["id"] == opts.show]\n        if not rows:\n            args.error(f"no test call {opts.show}")\n    results = [play(r) for r in rows]\n    if opts.show:\n        print("\\n".join(transcript(results[0])))\n        print("   result:", results[0]["outcome"], results[0]["values"], results[0]["wrong"] or "PASSED")\n        return\n    print("\\n".join(summary_lines(results, "Rules only (no model): ")))\n\n\nif __name__ == "__main__":\n    main()\n', 'audio.py': '"""The ears (speech to text) and the mouth (text to speech) of the voice bot, and audio as a phone line makes it.\n\nHeavy libraries (torch, faster-whisper, kokoro, parler-tts) are imported only when a model is loaded, so this file\ncan be read and its plain parts tested without them.\n\n    ears = Ears.load("faster-whisper")              or "transformers", "indic-conformer" (needs an HF token)\n    text, lang, seconds = ears.hear(samples, hint="hi", allowed=("hi", "mr", "en"))\n    mouth = Mouth.load("kokoro")                     or "parler" (needs an HF token), "mms" (non-commercial: test only)\n    samples, rate, seconds = mouth.say("नमस्ते", "hi", voice="bot")\n    phone_line(samples)                              what a phone call does to a voice: 8 kHz, 300–3400 Hz, μ-law, noise\n"""\n\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport time\n\nRATE = 16000  # what speech to text takes\nCACHE_LINES = 256  # lines of speech kept, so a task\'s greeting and questions are said at once\n\n# Every model, its licence, and whether a business may use it. The results always say which one ran.\nASR_INFO = {\n    "faster-whisper": {"model": "mobiuslabsgmbh/faster-whisper-large-v3-turbo", "licence": "MIT", "business": True},\n    "transformers": {"model": "openai/whisper-large-v3-turbo", "licence": "MIT", "business": True},\n    "indic-conformer": {"model": "ai4bharat/indic-conformer-600m-multilingual", "licence": "MIT", "business": True,\n                        "token": True},\n}\nTTS_INFO = {\n    "kokoro": {"model": "hexgrad/Kokoro-82M (kokoro-onnx)", "licence": "Apache-2.0", "business": True},\n    "parler": {"model": "ai4bharat/indic-parler-tts", "licence": "Apache-2.0", "business": True, "token": True},\n    "mms": {"model": "facebook/mms-tts-hin, facebook/mms-tts-mar", "licence": "CC-BY-NC-4.0", "business": False},\n}\nKOKORO_FILES = {  # Kokoro-82M as ONNX, from the kokoro-onnx project\'s release\n    "kokoro-v1.0.onnx": "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.onnx",\n    "voices-v1.0.bin": "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin",\n}\n# Voices: the bot is one calm woman\'s voice; test callers use other voices.\nKOKORO_VOICES = {"bot": "hf_alpha", "person": "hm_omega", "person2": "hf_beta", "person3": "hm_psi"}\nKOKORO_EN = {"bot": "af_heart", "person": "am_adam", "person2": "bf_emma", "person3": "bm_george"}\nPARLER_SPEAKERS = {  # Indic Parler-TTS speakers by language\n    "hi": {"bot": "Divya", "person": "Rohit", "person2": "Aman", "person3": "Rani"},\n    "mr": {"bot": "Sunita", "person": "Sanjay", "person2": "Nikhil", "person3": "Radha"},\n    "en": {"bot": "Mary", "person": "Thoma", "person2": "Swapna", "person3": "Dinesh"},\n}\nPARLER_STYLE = {\n    "bot": "speaks in a clear, calm and polite voice at a moderate pace. The recording is very clear, with no "\n           "background noise.",\n    "person": "speaks quickly in a casual tone. The recording is close, with a little background noise.",\n}\nLANG_NAMES = {"hi": "Hindi", "mr": "Marathi", "en": "English"}\n\n\ndef download(url: str, path: str) -> None:\n    import urllib.request\n\n    part = path + ".part"\n    with urllib.request.urlopen(url, timeout=600) as resp, open(part, "wb") as fh:\n        shutil.copyfileobj(resp, fh)\n    os.replace(part, path)\n\n\ndef ffmpeg_read(path: str):\n    """Any audio file (wav, mp3, m4a, ogg, opus, webm) as 16 kHz mono float32 samples."""\n    import numpy as np\n    import soundfile as sf\n\n    if shutil.which("ffmpeg"):\n        with tempfile.TemporaryDirectory() as tmp:\n            wav = os.path.join(tmp, "a.wav")\n            done = subprocess.run(["ffmpeg", "-nostdin", "-loglevel", "error", "-y", "-i", path, "-ac", "1", "-ar",\n                                   str(RATE), wav], capture_output=True, text=True, timeout=120, check=False)\n            if done.returncode != 0:\n                raise RuntimeError(f"ffmpeg could not read {os.path.basename(path)}: {done.stderr[-300:]}")\n            audio, _ = sf.read(wav, dtype="float32")\n        return audio\n    audio, rate = sf.read(path, dtype="float32", always_2d=True)\n    return resample(audio.mean(axis=1), rate, RATE).astype(np.float32)\n\n\ndef resample(audio, rate: int, to: int):\n    import numpy as np\n\n    if rate == to:\n        return np.asarray(audio, dtype=np.float32)\n    from scipy.signal import resample_poly\n\n    g = np.gcd(int(rate), int(to))\n    return resample_poly(np.asarray(audio, dtype=np.float32), to // g, rate // g).astype(np.float32)\n\n\ndef as_samples(audio, rate: int | None = None):\n    """Samples at 16 kHz mono from a file path, (rate, samples) as Gradio gives them, or samples at rate."""\n    import numpy as np\n\n    if isinstance(audio, str):\n        return ffmpeg_read(audio)\n    if isinstance(audio, tuple):\n        rate, audio = audio\n    audio = np.asarray(audio)\n    if audio.ndim > 1:\n        audio = audio.mean(axis=1) if audio.shape[1] <= 2 else audio.mean(axis=0)\n    if audio.dtype.kind == "i":\n        audio = audio.astype(np.float32) / np.iinfo(audio.dtype).max\n    return resample(audio.astype(np.float32), rate or RATE, RATE)\n\n\ndef phone_line(audio, rate: int = RATE, snr_db: float = 20.0, seed: int = 0):\n    """The voice as a phone call carries it: 8 kHz, only 300–3400 Hz, 8-bit μ-law, with some line noise (snr_db),\n    back at the input rate. Speech to text must work on this, not only on clean studio audio."""\n    import numpy as np\n    from scipy.signal import butter, sosfilt\n\n    x = resample(audio, rate, 8000)\n    x = sosfilt(butter(4, [300, 3400], btype="bandpass", fs=8000, output="sos"), x)\n    peak = float(np.max(np.abs(x))) or 1.0\n    x = x / peak * 0.9\n    mu = 255.0  # μ-law: what the phone network does to each sample\n    y = np.sign(x) * np.log1p(mu * np.abs(x)) / np.log1p(mu)\n    y = np.round(y * 127) / 127\n    x = np.sign(y) * ((1 + mu) ** np.abs(y) - 1) / mu\n    rng = np.random.default_rng(seed)\n    power = float(np.mean(x ** 2)) or 1e-9\n    x = x + rng.normal(0, np.sqrt(power / 10 ** (snr_db / 10)), size=x.shape)\n    return np.clip(resample(x.astype(np.float32), 8000, rate), -1, 1)\n\n\ndef to_int16(audio):\n    import numpy as np\n\n    return (np.clip(np.asarray(audio, dtype=np.float32), -1, 1) * 32767).astype(np.int16)\n\n\nNO_REPEAT = 6  # Whisper can loop on a short clip ("कर दो कर दो कर दो ..."): no 6-token phrase twice\n\n\ndef token_limit(audio) -> int:\n    """At most this many tokens for a clip: about 30 per second of speech (Devanagari takes many; 15 cut real\n    sentences short in eval 4), so a hallucination loop can\'t run on for long."""\n    return int(min(440, 30 + 30 * len(audio) / RATE))\n\n\ndef preload_cuda_libraries() -> None:\n    """faster-whisper (CTranslate2) looks for cuDNN and cuBLAS on the system path; on Kaggle they come with torch\'s\n    pip packages instead. Loading them first lets it find them."""\n    import ctypes\n    import glob\n    import site\n\n    for base in site.getsitepackages():\n        for folder in ("cublas", "cudnn"):\n            for lib in sorted(glob.glob(os.path.join(base, "nvidia", folder, "lib", "lib*.so*"))):\n                try:\n                    ctypes.CDLL(lib, mode=ctypes.RTLD_GLOBAL)\n                except OSError:\n                    pass\n\n\nclass Ears:\n    """Speech to text. hear() picks the language among the allowed ones (Whisper alone may say Urdu for Hindi)."""\n\n    def __init__(self, backend: str, run, detect=None, device: str = "?"):\n        self.backend, self.run, self.detect, self.device = backend, run, detect, device\n        self.info = ASR_INFO[backend]\n\n    @classmethod\n    def load(cls, backend: str, token: str | None = None) -> "Ears":\n        import torch\n\n        gpu = torch.cuda.is_available()\n        if backend == "faster-whisper":\n            if gpu:\n                preload_cuda_libraries()\n            from faster_whisper import WhisperModel\n\n            model = WhisperModel(ASR_INFO[backend]["model"], device="cuda" if gpu else "cpu",\n                                 compute_type="float16" if gpu else "int8")\n\n            def detect(audio):\n                _, _, probs = model.detect_language(audio)\n                return dict(probs)\n\n            def run(audio, lang, prompt):\n                segments, _ = model.transcribe(audio, language=lang, beam_size=1, vad_filter=True,\n                                               initial_prompt=prompt or None, condition_on_previous_text=False,\n                                               without_timestamps=True, no_repeat_ngram_size=NO_REPEAT,\n                                               max_new_tokens=token_limit(audio))\n                return " ".join(s.text.strip() for s in segments)\n\n            import numpy as np\n\n            run(np.zeros(RATE, dtype=np.float32), "hi", None)  # some CUDA problems only show on the first use\n            return cls(backend, run, detect, "GPU" if gpu else "CPU")\n        if backend == "transformers":\n            from transformers import pipeline\n\n            asr = pipeline("automatic-speech-recognition", model=ASR_INFO[backend]["model"],\n                           device="cuda:0" if gpu else "cpu", torch_dtype=torch.float16 if gpu else torch.float32,\n                           chunk_length_s=30)  # clips over 30 s too\n\n            def run(audio, lang, prompt):\n                options = {"task": "transcribe", "max_new_tokens": token_limit(audio), "no_repeat_ngram_size": NO_REPEAT,\n                           **({"language": lang} if lang else {})}\n                if prompt and len(audio) <= 30 * RATE:  # a hint (the bot\'s question) helps with short answers\n                    try:\n                        ids = asr.tokenizer.get_prompt_ids(prompt, return_tensors="pt").to(asr.device)\n                        return asr({"raw": audio, "sampling_rate": RATE},\n                                   generate_kwargs={**options, "prompt_ids": ids})["text"]\n                    except Exception as e:  # noqa: BLE001 - not supported here: without the hint\n                        print("speech-to-text hint not used:", repr(e)[:200])\n                return asr({"raw": audio, "sampling_rate": RATE}, generate_kwargs=options)["text"]\n\n            return cls(backend, run, None, "GPU" if gpu else "CPU")\n        if backend == "indic-conformer":\n            from transformers import AutoModel\n\n            model = AutoModel.from_pretrained(ASR_INFO[backend]["model"], trust_remote_code=True, token=token)\n\n            def run(audio, lang, prompt):\n                wav = torch.tensor(audio, dtype=torch.float32).unsqueeze(0)\n                return model(wav, lang or "hi", "rnnt")\n\n            return cls(backend, run, None, "GPU" if gpu else "CPU")\n        raise ValueError(f"unknown speech-to-text backend {backend}")\n\n    def pick_language(self, audio, hint: str | None, allowed) -> str:\n        """The language to decode in: the most likely of the allowed ones (when the model can tell), else hint."""\n        if self.detect is None:\n            return hint or allowed[0]\n        try:\n            probs = self.detect(audio)\n        except Exception:  # noqa: BLE001 - a guess only\n            return hint or allowed[0]\n        best = max(allowed, key=lambda lang: (probs.get(lang, 0.0) + (0.15 if lang == hint else 0.0)))\n        return best\n\n    def hear(self, audio, hint: str | None = "hi", allowed=("hi", "mr", "en"), prompt: str | None = None,\n             detect: bool = True):\n        """(text, language decoded in, seconds) for 16 kHz samples."""\n        started = time.perf_counter()\n        lang = self.pick_language(audio, hint, list(allowed)) if detect else (hint or allowed[0])\n        text = self.run(audio, lang, prompt)\n        return " ".join(str(text or "").split()), lang, round(time.perf_counter() - started, 3)\n\n\nclass Mouth:\n    """Text to speech. say() returns float32 samples, the sample rate and the seconds it took."""\n\n    def __init__(self, backend: str, run, rate: int, device: str = "?"):\n        self.backend, self.run, self.rate, self.device = backend, run, rate, device\n        self.info = TTS_INFO[backend]\n        self.cache: dict = {}\n\n    @classmethod\n    def load(cls, backend: str, token: str | None = None) -> "Mouth":\n        import numpy as np\n        import torch\n\n        gpu = torch.cuda.is_available()\n        device = "cuda:0" if gpu else "cpu"\n        if backend == "kokoro":\n            # kokoro-onnx: the same Kokoro-82M voices, run by onnxruntime (the "kokoro" package needs Python < 3.13,\n            # Kaggle has 3.13). Pronunciation by espeak-ng: Hindi rules for Hindi and Marathi (same letters).\n            from kokoro_onnx import Kokoro\n\n            files = {}\n            for name, url in KOKORO_FILES.items():\n                files[name] = os.path.join(tempfile.gettempdir(), name)\n                if not os.path.exists(files[name]):\n                    download(url, files[name])\n            if gpu:\n                preload_cuda_libraries()\n            import onnxruntime as ort\n\n            if gpu and hasattr(ort, "preload_dlls"):  # onnxruntime 1.21+: load CUDA and cuDNN from the pip packages\n                try:\n                    ort.preload_dlls()\n                except Exception as e:  # noqa: BLE001 - it then runs on the CPU, and says so\n                    print("onnxruntime preload_dlls:", repr(e)[:200])\n\n            on = f"CPU (onnxruntime; providers: {\', \'.join(ort.get_available_providers())})"\n            if "CUDAExecutionProvider" in ort.get_available_providers():\n                session = ort.InferenceSession(files["kokoro-v1.0.onnx"],\n                                               providers=["CUDAExecutionProvider", "CPUExecutionProvider"])\n                net = Kokoro.from_session(session, files["voices-v1.0.bin"])\n                on = "GPU (onnxruntime)" if "CUDAExecutionProvider" in session.get_providers() else on\n            else:\n                net = Kokoro(files["kokoro-v1.0.onnx"], files["voices-v1.0.bin"])\n\n            def run(text, lang, voice):\n                voices, code = (KOKORO_EN, "en-us") if lang == "en" else (KOKORO_VOICES, "hi")\n                audio, _ = net.create(text, voice=voices.get(voice, voices["bot"]), speed=1.0, lang=code)\n                return np.asarray(audio, dtype=np.float32)\n\n            run("नमस्ते", "hi", "bot")  # some problems only show on the first use\n            return cls(backend, run, 24000, on)\n        if backend == "parler":\n            from parler_tts import ParlerTTSForConditionalGeneration\n            from transformers import AutoTokenizer\n\n            name = TTS_INFO[backend]["model"]\n            net = ParlerTTSForConditionalGeneration.from_pretrained(name, token=token).to(device)\n            tok = AutoTokenizer.from_pretrained(name, token=token)\n            describe = AutoTokenizer.from_pretrained(net.config.text_encoder._name_or_path)\n\n            def run(text, lang, voice):\n                speaker = PARLER_SPEAKERS.get(lang, PARLER_SPEAKERS["hi"]).get(voice, "Divya")\n                style = PARLER_STYLE["bot" if voice == "bot" else "person"]\n                d = describe(f"{speaker} {style}", return_tensors="pt").to(device)\n                p = tok(text, return_tensors="pt").to(device)\n                with torch.no_grad():\n                    out = net.generate(input_ids=d.input_ids, attention_mask=d.attention_mask,\n                                       prompt_input_ids=p.input_ids, prompt_attention_mask=p.attention_mask)\n                return out.cpu().float().numpy().squeeze()\n\n            return cls(backend, run, net.config.sampling_rate, device)\n        if backend == "mms":\n            from transformers import AutoTokenizer, VitsModel\n\n            models = {}\n\n            def model(lang):\n                code = {"mr": "mar", "en": "eng"}.get(lang, "hin")\n                if code not in models:\n                    name = f"facebook/mms-tts-{code}"\n                    models[code] = (VitsModel.from_pretrained(name).to(device), AutoTokenizer.from_pretrained(name))\n                return models[code]\n\n            romanize = []\n\n            def run(text, lang, voice):\n                net, tok = model(lang)\n                if getattr(tok, "is_uroman", False):  # this voice reads Latin letters only\n                    if not romanize:\n                        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uroman"], check=True,\n                                       capture_output=True, timeout=600)\n                        import uroman\n\n                        romanize.append(uroman.Uroman().romanize_string)\n                    text = romanize[0](text)\n                inputs = tok(text, return_tensors="pt").to(device)\n                torch.manual_seed({"bot": 0, "person": 1, "person2": 2, "person3": 3}.get(voice, 0))\n                with torch.no_grad():\n                    return net(**inputs).waveform.cpu().float().numpy().squeeze()\n\n            return cls(backend, run, 16000, device)\n        raise ValueError(f"unknown text-to-speech backend {backend}")\n\n    def say(self, text: str, lang: str = "hi", voice: str = "bot"):\n        """(samples, rate, seconds). The same line said again (a task\'s greeting and questions) comes from a\n        cache, at once."""\n        started = time.perf_counter()\n        key = (text, lang, voice)\n        audio = self.cache.get(key)\n        if audio is None:\n            audio = self.run(text, lang, voice)\n            if audio is None or len(audio) == 0:\n                raise RuntimeError(f"{self.backend} made no sound for {text[:40]!r}")\n            if len(self.cache) >= CACHE_LINES:\n                self.cache.pop(next(iter(self.cache)))\n            self.cache[key] = audio\n        return audio, self.rate, round(time.perf_counter() - started, 3)\n\n\ndef load_first(kind: str, backends, token: str | None = None, log=print):\n    """The first backend that loads ("ears" or "mouth"), and what happened to each tried: [{backend, ok, why}]."""\n    tried = []\n    cls, info = (Ears, ASR_INFO) if kind == "ears" else (Mouth, TTS_INFO)\n    for backend in backends:\n        if info.get(backend, {}).get("token") and not token:\n            tried.append({"backend": backend, "ok": False, "why": "skipped: needs HF_TOKEN (Kaggle secret)"})\n            continue\n        started = time.time()\n        try:\n            loaded = cls.load(backend, token)\n        except Exception as e:  # noqa: BLE001 - try the next one\n            tried.append({"backend": backend, "ok": False, "why": f"{type(e).__name__}: {e}"[:400]})\n            log(f"{kind}: {backend} did not load: {e!r}"[:500])\n            continue\n        tried.append({"backend": backend, "ok": True, "why": f"loaded in {time.time() - started:.0f} s",\n                      "device": loaded.device})\n        log(f"{kind}: {backend} ({info[backend][\'model\']}, {info[backend][\'licence\']}) on {loaded.device}")\n        return loaded, tried\n    return None, tried\n', 'assistant.py': '"""The general assistant: talk about anything, like a voice assistant, in the language the person speaks.\n\n    chat = ollama_chat("gemma3:4b")                       chat(messages, stream) -> text, or an iterator of pieces\n    helper = Assistant(chat)\n    for sentence in helper.reply("मुंबई में कितने लोग रहते हैं?", "hi"):   one sentence at a time, as the model writes\n        speak(sentence)\n\nIn a task call (shop, clinic...), a question the task\'s facts don\'t cover gets a short answer from aside(): general\nknowledge is answered; the business\'s own details (a price, stock, an address) are never made up.\n"""\n\nimport json\nimport re\nimport urllib.request\n\nfrom voicebot.tasks import Task\n\nLANG_NAMES = {"hi": "Hindi, in Devanagari letters", "mr": "Marathi, in Devanagari letters", "en": "English"}\nSYSTEM = """You are Wimb, a friendly voice assistant from India, talking with someone by voice.\nReply in {language}, the language they speak (if they mix Hindi and English, you may too).\nKeep it short and spoken: one to three sentences, no lists, no markdown, no emojis, no headings.\nSay numbers the way people say them. If you are not sure, or it needs live information (today\'s news, weather,\nprices, scores), say so briefly instead of guessing. Be warm and polite, and use "आप" in Hindi."""\nASIDE = """You are a voice assistant on a phone call. The call: {purpose}\nThe other person ({who}) just asked something the call\'s script does not cover: "{question}"\nAnswer in {language}, in one short spoken sentence, no markdown.\nIf it is about this business\'s own details (its prices, stock, fees, address, timings, offers) say politely that\nyou will check with the team and tell them. Otherwise answer it briefly from general knowledge."""\nSENTENCE_END = re.compile(r"([।.!?]+[\\"\')\\]]*)(\\s+)")\nEMOJI = re.compile("[\\U0001F300-\\U0001FAFF☀-➿️]")\nMAX_HISTORY = 12  # messages kept (6 back-and-forths): enough context, still fast\n\n\ndef clean(text: str) -> str:\n    """Model text as it should be spoken: no markdown, emojis or list marks."""\n    text = EMOJI.sub("", text or "")\n    text = re.sub(r"[*_#`>|]+", "", text)\n    text = re.sub(r"^\\s*(?:[-•]|\\d+[.)])\\s+", "", text, flags=re.MULTILINE)\n    return " ".join(text.split())\n\n\ndef sentences(pieces, min_chars: int = 12):\n    """Whole sentences from a stream of text pieces (the model\'s tokens), each as soon as it is complete, so the\n    voice can start on the first while the model writes the rest."""\n    buffer = ""\n    for piece in pieces:\n        buffer += piece\n        while True:\n            found = None\n            for m in SENTENCE_END.finditer(buffer):\n                if m.end(1) >= min_chars:\n                    found = m\n                    break\n            if not found:\n                break\n            out = clean(buffer[:found.end(1)])\n            buffer = buffer[found.end():]\n            if out:\n                yield out\n    rest = clean(buffer)\n    if rest:\n        yield rest\n\n\nclass Assistant:\n    """A conversation with the model, remembered for the last MAX_HISTORY messages."""\n\n    def __init__(self, chat):\n        self.chat = chat\n        self.history: list[dict] = []\n\n    def reply(self, text: str, lang: str = "hi"):\n        """The answer to text, sentence by sentence (a generator); remembered once complete."""\n        system = SYSTEM.format(language=LANG_NAMES.get(lang, LANG_NAMES["hi"]))\n        messages = [{"role": "system", "content": system}, *self.history[-MAX_HISTORY:],\n                    {"role": "user", "content": text}]\n        said = []\n        for sentence in sentences(self.chat(messages, stream=True)):\n            said.append(sentence)\n            yield sentence\n        self.history += [{"role": "user", "content": text}, {"role": "assistant", "content": " ".join(said)}]\n\n\ndef aside(chat, task: Task, question: str, lang: str) -> str | None:\n    """A short answer to an off-script question in a task call (None when the model gives nothing)."""\n    prompt = ASIDE.format(purpose=task.purpose(), who=task.who(), question=question,\n                          language=LANG_NAMES.get(lang, LANG_NAMES["hi"]))\n    text = clean(chat([{"role": "user", "content": prompt}], stream=False))\n    return text[:300] or None\n\n\ndef ollama_chat(model: str, url: str = "http://127.0.0.1:11434", max_tokens: int = 200):\n    """chat(messages, stream) through Ollama\'s /api/chat: the whole reply, or (stream=True) its pieces."""\n\n    def chat(messages, stream=False):\n        body = {"model": model, "messages": messages, "stream": stream, "keep_alive": "60m",\n                "options": {"temperature": 0.4, "num_predict": max_tokens, "num_ctx": 4096}}\n        request = urllib.request.Request(url + "/api/chat", json.dumps(body).encode(),\n                                         {"Content-Type": "application/json"})\n        if not stream:\n            with urllib.request.urlopen(request, timeout=120) as resp:\n                return json.load(resp).get("message", {}).get("content", "")\n\n        def pieces():\n            with urllib.request.urlopen(request, timeout=120) as resp:\n                for line in resp:\n                    if line.strip():\n                        data = json.loads(line)\n                        yield data.get("message", {}).get("content", "")\n                        if data.get("done"):\n                            return\n\n        return pieces()\n\n    return chat\n', 'server.py': '"""The voice page: talk to Wimb hands-free, like a voice assistant. Open it, tap the circle once, then just talk:\nthe page hears when you start and stop (a voice detector in the browser), Wimb answers by voice, and you can\ninterrupt it. Modes at the top: Assistant (ask anything) and the tasks (shop, clinic, order...).\n\n    app = make_app(ears, mouth, llm=understanding_llm, chat=ollama_chat("gemma3:4b"), password="...")\n    serve(app, 7860); url = open_tunnel(7860)            on Kaggle: a public https link (the page asks the password)\n\nEach turn streams back as lines of JSON, so the voice starts on the first sentence while the rest is still being\nwritten: {"type": "heard"}, then {"type": "say", "text", "audio"} per sentence, then {"type": "done"}.\nCalls are saved (audio and text) only when the person leaves "save my voice" on.\n"""\n\nimport base64\nimport hashlib\nimport hmac\nimport io\nimport json\nimport os\nimport re\nimport secrets\nimport shutil\nimport subprocess\nimport tempfile\nimport threading\nimport time\nimport uuid\nfrom pathlib import Path\n\nfrom voicebot import tasks\nfrom voicebot.assistant import SENTENCE_END, Assistant, aside\nfrom voicebot.engine import Call\nfrom voicebot.lang import detect\nfrom voicebot.speech_text import clean_heard, for_voice\n\nWEB = Path(__file__).parent / "web" / "index.html"\nVAD_CDN = "https://cdn.jsdelivr.net/npm/@ricky0123/vad-web@0.0.22/dist/"\nORT_CDN = "https://cdn.jsdelivr.net/npm/onnxruntime-web@1.14.0/dist/"\nASSISTANT = "assistant"\nGREETING = {"hi": "नमस्ते! मैं विम्ब हूँ। बताइए, मैं आपकी क्या मदद कर सकती हूँ?",\n            "mr": "नमस्कार! मी विम्ब. सांगा, मी तुमची काय मदत करू?",\n            "en": "Hi, I\'m Wimb. How can I help you?"}\nNO_BRAIN = {"hi": "माफ़ कीजिए, अभी मेरा मॉडल चालू नहीं है, थोड़ी देर बाद कोशिश कीजिए।",\n            "mr": "माफ करा, आत्ता माझं मॉडेल चालू नाही, थोड्या वेळाने प्रयत्न करा.",\n            "en": "Sorry, my model is not running right now. Please try again in a bit."}\nNOT_HEARD = {"hi": "माफ़ कीजिए, सुनाई नहीं दिया। फिर से बोलिए?", "mr": "माफ करा, ऐकू आलं नाही. पुन्हा बोला?",\n             "en": "Sorry, I didn\'t catch that. Could you say it again?"}\nCLOUDFLARED = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"\nMODEL_LOCK = threading.Lock()  # one GPU: one model call at a time\n\n\ndef modes() -> list[dict]:\n    order = sorted(tasks.available(), key=lambda t: (t != "shop_stock", t))\n    return [{"id": ASSISTANT, "name": "Assistant"}] + [{"id": t, "name": tasks.load(t).name("en").split(":")[0]}\n                                                      for t in order]\n\n\ndef pairs(text: str, speech: str) -> list[tuple[str, str]]:\n    """A reply cut into sentences, shown and spoken side by side (so the voice can start on the first)."""\n    def cut(t):\n        parts, start = [], 0\n        for m in SENTENCE_END.finditer(t + " "):\n            parts.append(t[start:m.end(1)].strip())\n            start = m.end()\n        tail = t[start:].strip()\n        return [p for p in parts + ([tail] if tail else []) if p]\n\n    shown, said = cut(text), cut(speech)\n    if len(shown) != len(said):\n        return [(text, speech)]\n    return list(zip(shown, said))\n\n\ndef echo_of(heard: str, bot: str) -> bool:\n    """Whether what was heard while Wimb spoke is just Wimb\'s own voice coming back through the microphone."""\n    words = re.findall(r"\\w+", heard.lower())\n    if not words:\n        return True\n    own = set(re.findall(r"\\w+", bot.lower()))\n    return sum(w in own for w in words) / len(words) >= 0.6\n\n\nclass Conversation:\n    """One conversation on the page: the assistant, or a task call; saved only with consent."""\n\n    def __init__(self, mode: str, chat=None, llm=None, save_dir: str | None = None, save: bool = False,\n                 lang: str = "hi"):\n        self.id = uuid.uuid4().hex[:12]\n        self.mode, self.lang, self.chat = mode, lang, chat\n        self.helper = Assistant(chat) if mode == ASSISTANT and chat else None\n        self.call = None\n        if mode != ASSISTANT:\n            task = tasks.load(mode)\n            answer = (lambda question, lang: aside(chat, task, question, lang)) if chat else None\n            self.call = Call(task, llm=llm, aside=answer)\n            self.lang = self.call.lang\n        self.folder = os.path.join(save_dir, time.strftime("%Y%m%d-%H%M%S-") + self.id) if save_dir and save else None\n        if self.folder:\n            os.makedirs(self.folder, exist_ok=True)\n        self.log: list[dict] = []\n        self.last_bot = ""\n        self.n = 0\n        self.ended = False\n\n    def opening(self):\n        """What Wimb says first: [(shown, spoken)], and whether the call is over already."""\n        if self.call is not None:\n            r = self.call.start()\n            return pairs(r.text, r.speech), r.done\n        greeting = GREETING.get(self.lang, GREETING["hi"])\n        return [(greeting, for_voice(greeting, self.lang))], False\n\n    def answer(self, heard: str, heard_lang: str | None = None):\n        """Wimb\'s answer to what was heard, sentence by sentence: (shown, spoken). Sets self.ended."""\n        if self.call is not None:\n            r = self.call.hear(heard)\n            self.lang, self.ended = r.lang, r.done\n            yield from pairs(r.text, r.speech)\n            return\n        found, score = detect(heard, default=self.lang)\n        self.lang = found if score else (heard_lang if heard_lang in GREETING else self.lang)\n        if not heard:\n            line = NOT_HEARD[self.lang]\n            yield line, for_voice(line, self.lang)\n            return\n        if self.helper is None:\n            line = NO_BRAIN[self.lang]\n            yield line, for_voice(line, self.lang)\n            return\n        for sentence in self.helper.reply(heard, self.lang):\n            yield sentence, for_voice(sentence, self.lang)\n\n    def found(self) -> str:\n        if self.call is None:\n            return ""\n        r = self.call.result()\n        values = ", ".join(f"{k}: {v}" for k, v in r["values"].items() if v is not None)\n        return (values or "nothing yet") + (f"  ({r[\'outcome\']})" if r["outcome"] else "")\n\n    def save(self) -> None:\n        if not self.folder:\n            return\n        data = {"id": self.id, "mode": self.mode, "log": self.log,\n                "result": self.call.result() if self.call is not None else None}\n        with open(os.path.join(self.folder, "call.json"), "w", encoding="utf-8") as fh:\n            json.dump(data, fh, ensure_ascii=False, indent=1)\n\n\ndef wav_b64(audio, rate: int) -> str:\n    import soundfile as sf\n\n    from voicebot.audio import to_int16\n\n    buffer = io.BytesIO()\n    sf.write(buffer, to_int16(audio), rate, format="WAV", subtype="PCM_16")\n    return base64.b64encode(buffer.getvalue()).decode()\n\n\ndef samples_from(body: bytes, content_type: str):\n    """16 kHz samples from what the page sent (WAV from the voice detector, or webm/ogg/mp4 from tap-to-talk)."""\n    from voicebot.audio import as_samples\n\n    if "wav" in (content_type or "") or body[:4] == b"RIFF":\n        import soundfile as sf\n\n        audio, rate = sf.read(io.BytesIO(body), dtype="float32", always_2d=True)\n        return as_samples((rate, audio.mean(axis=1)))\n    suffix = ".ogg" if "ogg" in content_type else ".mp4" if "mp4" in content_type else ".webm"\n    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as fh:\n        fh.write(body)\n    try:\n        return as_samples(fh.name)\n    finally:\n        os.remove(fh.name)\n\n\ndef line(event: dict) -> str:\n    return json.dumps(event, ensure_ascii=False) + "\\n"\n\n\ndef make_app(ears=None, mouth=None, llm=None, chat=None, password: str = "", save_dir: str | None = None,\n             vad_base: str = VAD_CDN, ort_base: str = ORT_CDN, vendor_dir: str | None = None):\n    """The FastAPI app. ears/mouth: voicebot.audio Ears and Mouth (None: text only); llm: the understanding model\n    for tasks; chat: the assistant model (voicebot.assistant.ollama_chat); password: asked once (cookie)."""\n    from fastapi import FastAPI, Form, Request\n    from fastapi.responses import HTMLResponse, JSONResponse, RedirectResponse, StreamingResponse\n\n    app = FastAPI(title="Wimb voice")\n    secret = secrets.token_bytes(32)\n    token = hmac.new(secret, (password or "").encode(), hashlib.sha256).hexdigest()\n    conversations: dict[str, Conversation] = {}\n    page = WEB.read_text(encoding="utf-8").replace("{{VAD_BASE}}", vad_base).replace("{{ORT_BASE}}", ort_base)\n    app.state.conversations = conversations\n\n    if vendor_dir:  # local tests: the voice detector\'s files served from here instead of the CDN\n        from fastapi.staticfiles import StaticFiles\n\n        app.mount("/vendor", StaticFiles(directory=vendor_dir), name="vendor")\n\n    def allowed(request: Request) -> bool:\n        return not password or hmac.compare_digest(request.cookies.get("wimb", ""), token)\n\n    @app.middleware("http")\n    async def guard(request: Request, call_next):\n        if request.url.path in ("/login",) or request.url.path.startswith("/vendor") or allowed(request):\n            return await call_next(request)\n        if request.url.path.startswith("/api/"):\n            return JSONResponse({"error": "log in first"}, status_code=401)\n        return HTMLResponse(LOGIN.replace("{{ERROR}}", ""))\n\n    @app.post("/login")\n    async def login(password_typed: str = Form("", alias="password")):\n        if password and not hmac.compare_digest(password_typed, password):\n            return HTMLResponse(LOGIN.replace("{{ERROR}}", "Wrong password"), status_code=401)\n        response = RedirectResponse("./", status_code=303)\n        response.set_cookie("wimb", token, httponly=True, samesite="lax", secure=False, max_age=12 * 3600)\n        return response\n\n    @app.get("/")\n    async def index():\n        return HTMLResponse(page)\n\n    @app.get("/api/modes")\n    async def list_modes():\n        return modes()\n\n    def speak(text: str, lang: str):\n        if mouth is None or not text:\n            return None, 0.0\n        with MODEL_LOCK:\n            audio, rate, seconds = mouth.say(text, lang, "bot")\n        return wav_b64(audio, rate), seconds\n\n    def reply_lines(conv: Conversation, pieces, timing: dict, started: float):\n        """The \'say\' lines, one per sentence with its audio, then \'done\'."""\n        said = []\n        first = True\n        try:\n            for shown, spoken in pieces:\n                audio, seconds = speak(spoken, conv.lang)\n                if first:\n                    timing["first_sentence"] = round(time.perf_counter() - started - seconds, 2)\n                    timing["first_audio"] = round(time.perf_counter() - started, 2)\n                    first = False\n                timing["speak"] = round(timing.get("speak", 0) + seconds, 2)\n                said.append(shown)\n                yield line({"type": "say", "text": shown, "audio": audio, "lang": conv.lang})\n        except Exception as e:  # noqa: BLE001 - say what broke, keep the page alive\n            yield line({"type": "error", "text": f"Sorry, something broke: {type(e).__name__}"})\n            timing["error"] = repr(e)[:300]\n        conv.last_bot = " ".join(said)\n        conv.log.append({"who": "bot", "text": conv.last_bot, "lang": conv.lang, **timing})\n        conv.save()\n        yield line({"type": "done", "found": conv.found(), "end": conv.ended, "timing": timing})\n\n    @app.post("/api/start")\n    async def start(request: Request):\n        body = await request.json()\n        mode = body.get("mode") or ASSISTANT\n        if mode != ASSISTANT and mode not in tasks.available():\n            return JSONResponse({"error": "no such mode"}, status_code=400)\n        conv = Conversation(mode, chat=chat, llm=llm, save_dir=save_dir, save=bool(body.get("save")),\n                            lang=body.get("lang") if body.get("lang") in GREETING else "hi")\n        conversations[conv.id] = conv\n        started = time.perf_counter()\n\n        def run():\n            yield line({"type": "session", "sid": conv.id})\n            pieces, conv.ended = conv.opening()\n            yield from reply_lines(conv, pieces, {}, started)\n\n        return StreamingResponse(run(), media_type="application/x-ndjson")\n\n    def conversation(sid: str | None) -> Conversation | None:\n        return conversations.get(sid or "")\n\n    @app.post("/api/turn")\n    async def turn(request: Request, sid: str = "", during: int = 0):\n        conv = conversation(sid)\n        if conv is None:\n            return JSONResponse({"error": "no such conversation: start again"}, status_code=404)\n        body = await request.body()\n        content_type = request.headers.get("content-type", "")\n        started = time.perf_counter()\n\n        def run():\n            timing = {}\n            conv.n += 1\n            heard, heard_lang = "", None\n            try:\n                samples = samples_from(body, content_type)\n                if conv.folder:\n                    import soundfile as sf\n\n                    sf.write(os.path.join(conv.folder, f"{conv.n:02d}_you.wav"), samples, 16000)\n                if ears is not None and len(samples) > 3200:\n                    with MODEL_LOCK:\n                        heard, heard_lang, timing["hear"] = ears.hear(samples, hint=conv.lang,\n                                                                       allowed=("hi", "mr", "en"))\n                    heard = clean_heard(heard)\n            except Exception as e:  # noqa: BLE001\n                timing["error"] = f"hear: {e!r}"[:300]\n            if during and echo_of(heard, conv.last_bot):\n                yield line({"type": "heard", "text": heard, "ignored": True})\n                return\n            conv.log.append({"who": "person", "text": heard, "during_bot": bool(during), **timing})\n            yield line({"type": "heard", "text": heard, "ignored": False})\n            if conv.ended:\n                return\n            think = time.perf_counter()\n            pieces = conv.answer(heard, heard_lang)\n            timing["think_start"] = round(time.perf_counter() - think, 2)\n            yield from reply_lines(conv, pieces, timing, started)\n\n        return StreamingResponse(run(), media_type="application/x-ndjson")\n\n    @app.post("/api/text")\n    async def text_turn(request: Request):\n        body = await request.json()\n        conv = conversation(body.get("sid"))\n        if conv is None:\n            return JSONResponse({"error": "no such conversation: start again"}, status_code=404)\n        typed = str(body.get("text") or "")[:500]\n        started = time.perf_counter()\n\n        def run():\n            conv.log.append({"who": "person", "text": typed, "typed": True})\n            yield line({"type": "heard", "text": typed, "ignored": False})\n            if not conv.ended:\n                yield from reply_lines(conv, conv.answer(typed), {}, started)\n\n        return StreamingResponse(run(), media_type="application/x-ndjson")\n\n    @app.post("/api/end")\n    async def end(request: Request):\n        body = await request.json()\n        conv = conversations.pop(body.get("sid") or "", None)\n        if conv is not None:\n            conv.save()\n        return {"ok": True}\n\n    return app\n\n\nLOGIN = """<!doctype html><html><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,\ninitial-scale=1"><title>Wimb Voice</title><style>body{margin:0;min-height:100dvh;display:flex;align-items:center;\njustify-content:center;background:#0b1020;color:#eef1ff;font-family:system-ui,sans-serif;padding:16px}\nform{display:flex;flex-direction:column;gap:12px;width:100%;max-width:320px}input,button{padding:14px;border-radius:12px;\nfont-size:16px;border:1px solid #2a3360;background:#121834;color:#eef1ff}button{background:#4f8cff;border:none}\np{color:#ff8a95;min-height:1em;margin:0}</style></head><body><form method="post" action="login">\n<h2>Wimb Voice</h2><input type="password" name="password" placeholder="Password" autofocus>\n<button>Open</button><p>{{ERROR}}</p></form></body></html>"""\n\n\ndef serve(app, port: int = 7860) -> threading.Thread:\n    """Run the app on 127.0.0.1:port in a background thread."""\n    import uvicorn\n\n    server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning"))\n    thread = threading.Thread(target=server.run, daemon=True)\n    thread.start()\n    for _ in range(100):\n        if server.started:\n            break\n        time.sleep(0.1)\n    return thread\n\n\ndef open_tunnel(port: int = 7860, log=print) -> str | None:\n    """A public https link to 127.0.0.1:port: Gradio\'s share tunnel, else a Cloudflare quick tunnel."""\n    try:\n        from gradio.networking import setup_tunnel\n\n        return setup_tunnel("127.0.0.1", port, secrets.token_urlsafe(32), None, None)\n    except Exception as e:  # noqa: BLE001\n        log("gradio tunnel failed:", repr(e)[:300])\n    binary = shutil.which("cloudflared") or "/tmp/cloudflared"\n    try:\n        if not os.path.exists(binary):\n            subprocess.run(["curl", "-fsSL", "-o", binary, CLOUDFLARED], check=True, timeout=300)\n            os.chmod(binary, 0o755)\n        proc = subprocess.Popen([binary, "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"],\n                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)\n        deadline = time.time() + 90\n        while time.time() < deadline:\n            out = proc.stdout.readline()\n            found = re.search(r"https://[a-z0-9-]+\\.trycloudflare\\.com", out or "")\n            if found:\n                return found.group(0)\n    except Exception as e:  # noqa: BLE001\n        log("cloudflared failed:", repr(e)[:300])\n    return None\n', 'tasks/clinic_booking.json': '{\n  "id": "clinic_booking",\n  "name": {"en": "Clinic: book a visit", "hi": "क्लिनिक: अपॉइंटमेंट बुक करें", "mr": "दवाखाना: भेट बुक करा"},\n  "bot_role": "receiver",\n  "who": "a patient or their family member calling the clinic",\n  "purpose": "The assistant answers the phone at {clinic} and books a visit with {doctor}: the patient\'s name, the day, the time, a mobile number, and what the visit is for. The clinic is open Monday to Saturday, 10:00 to 19:00, closed on Sunday.",\n  "languages": ["hi", "mr", "en"],\n  "default_language": "hi",\n  "context": {"clinic": "सिटी केयर क्लिनिक", "doctor": "डॉक्टर मेहता"},\n  "greeting": {\n    "hi": "नमस्ते, {clinic} में आपका स्वागत है। मैं AI सहायक हूँ, आपका अपॉइंटमेंट बुक कर देती हूँ।",\n    "mr": "नमस्कार, {clinic} मध्ये आपलं स्वागत आहे. मी AI सहायक आहे, तुमची अपॉइंटमेंट बुक करते.",\n    "en": "Hello, welcome to {clinic}. I\'m the AI assistant, I\'ll book your appointment."\n  },\n  "slots": [\n    {"name": "patient_name", "type": "name", "describe": "the patient\'s name (as said, without words like \'my name is\')",\n     "ask": {"hi": "मरीज़ का नाम क्या है?", "mr": "रुग्णाचं नाव काय आहे?", "en": "May I have the patient\'s name?"},\n     "readback": {"hi": "नाम {value}", "mr": "नाव {value}", "en": "name {value}"}},\n    {"name": "day", "type": "day", "describe": "the day of the visit",\n     "ask": {"hi": "आप किस दिन आना चाहेंगे?", "mr": "तुम्हाला कोणत्या दिवशी यायचं आहे?", "en": "Which day would you like to come?"},\n     "valid": {"not": ["sunday"]},\n     "invalid": {"hi": "माफ़ कीजिए, रविवार को क्लिनिक बंद रहता है। कोई और दिन बताइए।", "mr": "माफ करा, रविवारी दवाखाना बंद असतो. दुसरा दिवस सांगा.", "en": "Sorry, the clinic is closed on Sunday. Please choose another day."},\n     "readback": {"hi": "{value}", "mr": "{value}", "en": "{value}"}},\n    {"name": "time", "type": "time", "describe": "the time of the visit (24 h HH:MM)",\n     "ask": {"hi": "किस समय आना ठीक रहेगा? क्लिनिक सुबह दस से शाम सात बजे तक खुला है।", "mr": "कोणत्या वेळी यायला जमेल? दवाखाना सकाळी दहा ते संध्याकाळी सात उघडा असतो.", "en": "What time suits you? We\'re open from 10 in the morning to 7 in the evening."},\n     "valid": {"min": "10:00", "max": "19:00"},\n     "invalid": {"hi": "माफ़ कीजिए, उस समय क्लिनिक बंद रहता है। सुबह दस से शाम सात के बीच का समय बताइए।", "mr": "माफ करा, त्या वेळी दवाखाना बंद असतो. सकाळी दहा ते संध्याकाळी सात मधली वेळ सांगा.", "en": "Sorry, we\'re closed then. Please choose a time between 10 am and 7 pm."},\n     "readback": {"hi": "{value}", "mr": "{value}", "en": "at {value}"}},\n    {"name": "phone", "type": "phone", "describe": "the caller\'s 10-digit mobile number",\n     "ask": {"hi": "आपका मोबाइल नंबर बता दीजिए।", "mr": "तुमचा मोबाईल नंबर सांगा.", "en": "Your mobile number, please?"},\n     "readback": {"hi": "नंबर {value}", "mr": "नंबर {value}", "en": "number {value}"}},\n    {"name": "reason", "type": "text", "required": false, "describe": "what the visit is for: the health problem, in a few words",\n     "ask": {"hi": "किस तकलीफ़ के लिए दिखाना है?", "mr": "कोणत्या त्रासासाठी दाखवायचं आहे?", "en": "What is the visit for?"},\n     "readback": {"hi": "{value} के लिए", "mr": "{value} साठी", "en": "for {value}"}}\n  ],\n  "confirm": true,\n  "facts": {\n    "fee": {"about": "the doctor\'s fee or consultation charges",\n      "answer": {"hi": "डॉक्टर की फ़ीस ₹500 है।", "mr": "डॉक्टरांची फी ₹500 आहे.", "en": "The doctor\'s fee is ₹500."}},\n    "address": {"about": "where the clinic is, the address or how to reach",\n      "answer": {"hi": "क्लिनिक एमजी रोड पर, सिटी मॉल के सामने है।", "mr": "दवाखाना एमजी रोडवर, सिटी मॉलच्या समोर आहे.", "en": "The clinic is on MG Road, opposite City Mall."}},\n    "timings": {"about": "when the clinic is open, opening hours or days",\n      "answer": {"hi": "क्लिनिक सोमवार से शनिवार, सुबह दस से शाम सात बजे तक खुला है। रविवार बंद।", "mr": "दवाखाना सोमवार ते शनिवार, सकाळी दहा ते संध्याकाळी सात उघडा असतो. रविवारी बंद.", "en": "We\'re open Monday to Saturday, 10 am to 7 pm. Closed on Sunday."}},\n    "doctor": {"about": "which doctor, or what kind of doctor",\n      "answer": {"hi": "{doctor} जनरल फिज़िशियन हैं।", "mr": "{doctor} जनरल फिजिशियन आहेत.", "en": "{doctor} is a general physician."}}\n  },\n  "closing": {"hi": "आपका अपॉइंटमेंट बुक हो गया है। समय से दस मिनट पहले आ जाइए। धन्यवाद!", "mr": "तुमची अपॉइंटमेंट बुक झाली आहे. वेळेच्या दहा मिनिटं आधी या. धन्यवाद!", "en": "Your appointment is booked. Please come ten minutes early. Thank you!"}\n}\n', 'tasks/order_confirm.json': '{\n  "id": "order_confirm",\n  "name": {"en": "Shop order: confirm cash on delivery", "hi": "ऑर्डर: कैश ऑन डिलीवरी कन्फर्म", "mr": "ऑर्डर: कॅश ऑन डिलिव्हरी पक्की करा"},\n  "bot_role": "caller",\n  "who": "a customer who placed an online order",\n  "purpose": "The assistant calls a customer of {store} to confirm their cash-on-delivery order of {item} for {amount} rupees: whether they confirm it, and the delivery time they prefer; if they cancel, the reason.",\n  "languages": ["hi", "mr", "en"],\n  "default_language": "hi",\n  "context": {"store": "बाज़ार मार्ट", "item": "मिक्सर ग्राइंडर", "amount": 2499},\n  "greeting": {\n    "hi": "नमस्ते! मैं {store} की AI सहायक बोल रही हूँ, आपके ऑर्डर के बारे में कॉल किया है।",\n    "mr": "नमस्कार! मी {store} ची AI सहायक बोलतेय, तुमच्या ऑर्डरबद्दल फोन केला आहे.",\n    "en": "Hello! I\'m an AI assistant from {store}, calling about your order."\n  },\n  "slots": [\n    {"name": "confirmed", "type": "bool", "describe": "whether the customer confirms (keeps) the order; false when they cancel it or say they did not order",\n     "ask": {"hi": "आपने {item} ऑर्डर किया है, ₹{amount} कैश ऑन डिलीवरी। क्या आप ये ऑर्डर कन्फर्म करते हैं?", "mr": "तुम्ही {item} ऑर्डर केला आहे, ₹{amount} कॅश ऑन डिलिव्हरी. ही ऑर्डर पक्की करायची का?", "en": "You ordered a {item}, ₹{amount} cash on delivery. Do you confirm this order?"},\n     "readback": {"true": {"hi": "ऑर्डर कन्फर्म", "mr": "ऑर्डर पक्की", "en": "the order is confirmed"},\n                  "false": {"hi": "ऑर्डर कैंसल", "mr": "ऑर्डर रद्द", "en": "the order is cancelled"}}},\n    {"name": "delivery_time", "type": "enum", "describe": "when the customer wants the delivery: morning, afternoon or evening", "when": {"confirmed": true},\n     "ask": {"hi": "डिलीवरी किस समय ठीक रहेगी, सुबह, दोपहर या शाम?", "mr": "डिलिव्हरी कोणत्या वेळी सोयीची, सकाळी, दुपारी की संध्याकाळी?", "en": "When should we deliver: morning, afternoon or evening?"},\n     "options": {\n       "morning": {"hi": "सुबह", "mr": "सकाळी", "en": "in the morning", "words": ["सुबह", "सवेरे", "सकाळी", "सकाळ", "subah", "morning", "मॉर्निंग"]},\n       "afternoon": {"hi": "दोपहर", "mr": "दुपारी", "en": "in the afternoon", "words": ["दोपहर", "दुपहर", "दुपारी", "दुपार", "dopahar", "afternoon", "आफ्टरनून"]},\n       "evening": {"hi": "शाम", "mr": "संध्याकाळी", "en": "in the evening", "words": ["शाम", "संध्याकाळी", "संध्याकाळ", "shaam", "sham", "evening", "ईवनिंग", "इवनिंग"]}\n     },\n     "readback": {"hi": "डिलीवरी {value}", "mr": "डिलिव्हरी {value}", "en": "delivery {value}"}},\n    {"name": "cancel_reason", "type": "text", "required": false, "when": {"confirmed": false},\n     "describe": "why the customer cancels, in a few words",\n     "ask": {"hi": "कोई बात नहीं। क्या आप बता सकते हैं कि ऑर्डर क्यों कैंसल करना है?", "mr": "काही हरकत नाही. ऑर्डर का रद्द करायची ते सांगाल का?", "en": "No problem. May I ask why you want to cancel?"},\n     "readback": {"hi": "वजह: {value}", "mr": "कारण: {value}", "en": "reason: {value}"}}\n  ],\n  "confirm": true,\n  "facts": {\n    "when_arrive": {"about": "when the order will arrive, delivery date",\n      "answer": {"hi": "ऑर्डर दो से तीन दिन में पहुँच जाएगा।", "mr": "ऑर्डर दोन ते तीन दिवसांत पोहोचेल.", "en": "The order will arrive in two to three days."}},\n    "pay_online": {"about": "paying by UPI, card or online instead of cash",\n      "answer": {"hi": "हाँ, डिलीवरी के समय आप यूपीआई से भी पेमेंट कर सकते हैं।", "mr": "हो, डिलिव्हरीच्या वेळी तुम्ही यूपीआयनेही पैसे देऊ शकता.", "en": "Yes, you can also pay by UPI at delivery."}},\n    "return": {"about": "return, replacement or warranty",\n      "answer": {"hi": "सात दिन में रिटर्न हो सकता है, और एक साल की वारंटी है।", "mr": "सात दिवसांत परत करता येईल, आणि एक वर्षाची वॉरंटी आहे.", "en": "You can return it within seven days, and there\'s a one-year warranty."}}\n  },\n  "closing": [\n    {"when": {"confirmed": false}, "text": {"hi": "ठीक है, आपका ऑर्डर कैंसल कर दिया जाएगा। धन्यवाद!", "mr": "ठीक आहे, तुमची ऑर्डर रद्द केली जाईल. धन्यवाद!", "en": "Okay, your order will be cancelled. Thank you!"}},\n    {"text": {"hi": "धन्यवाद! आपका ऑर्डर जल्द पहुँचेगा।", "mr": "धन्यवाद! तुमची ऑर्डर लवकरच पोहोचेल.", "en": "Thank you! Your order will reach you soon."}}\n  ]\n}\n', 'tasks/shop_stock.json': '{\n  "id": "shop_stock",\n  "name": {"en": "Shop: is it in stock?", "hi": "दुकान: सामान है या नहीं", "mr": "दुकान: माल आहे का"},\n  "bot_role": "caller",\n  "who": "a shopkeeper",\n  "purpose": "The assistant calls a shop for a customer to find out whether {product} is in stock now, its price in rupees, whether the shop delivers to the home, and, if it is not in stock, what other brand or size the shop offers instead.",\n  "languages": ["hi", "mr", "en"],\n  "default_language": "hi",\n  "context": {"product": "16A MCB"},\n  "greeting": {\n    "hi": "नमस्ते! मैं WIMB की AI सहायक बोल रही हूँ, एक ग्राहक के लिए पूछना था।",\n    "mr": "नमस्कार! मी WIMB ची AI सहायक बोलतेय, एका ग्राहकासाठी विचारायचं होतं.",\n    "en": "Hello! I\'m an AI assistant calling from WIMB, asking for a customer."\n  },\n  "slots": [\n    {"name": "in_stock", "type": "bool", "describe": "whether {product} is available in the shop right now",\n     "ask": {"hi": "क्या आपके पास {product} है?", "mr": "तुमच्याकडे {product} आहे का?", "en": "Do you have {product} in stock?"},\n     "readback": {"true": {"hi": "{product} है", "mr": "{product} आहे", "en": "{product} is in stock"},\n                  "false": {"hi": "{product} अभी नहीं है", "mr": "{product} सध्या नाही", "en": "{product} is not in stock"}}},\n    {"name": "price_inr", "type": "money", "describe": "the price of one {product} in rupees", "when": {"in_stock": true},\n     "ask": {"hi": "{product} का दाम कितना है?", "mr": "{product} ची किंमत किती आहे?", "en": "What is the price of {product}?"},\n     "readback": {"hi": "दाम {value}", "mr": "किंमत {value}", "en": "price {value}"}},\n    {"name": "delivery", "type": "bool", "describe": "whether the shop delivers to the customer\'s home", "when": {"in_stock": true},\n     "ask": {"hi": "क्या आप घर पर डिलीवरी करते हैं?", "mr": "तुम्ही घरपोच डिलिव्हरी देता का?", "en": "Do you deliver to the home?"},\n     "readback": {"true": {"hi": "डिलीवरी हो जाएगी", "mr": "डिलिव्हरी मिळेल", "en": "you deliver"},\n                  "false": {"hi": "डिलीवरी नहीं होगी", "mr": "डिलिव्हरी नाही", "en": "no delivery"}}},\n    {"name": "alternative", "type": "text", "required": false, "when": {"in_stock": false},\n     "describe": "another product, brand or size the shopkeeper offers instead of {product} (short, as said, e.g. \'Havells 32A\')",\n     "ask": {"hi": "कोई दूसरा ब्रांड या साइज़ है जो आप दे सकें?", "mr": "दुसरा कोणता ब्रँड किंवा साइज आहे का?", "en": "Do you have another brand or size instead?"},\n     "readback": {"hi": "दूसरा विकल्प {value}", "mr": "दुसरा पर्याय {value}", "en": "instead: {value}"}}\n  ],\n  "confirm": true,\n  "facts": {\n    "quantity": {"about": "how many pieces or what quantity the customer needs",\n      "answer": {"hi": "अभी एक पीस चाहिए।", "mr": "सध्या एक नग हवा आहे.", "en": "Just one piece for now."}},\n    "who_customer": {"about": "who the customer is, where they live, or their name",\n      "answer": {"hi": "ग्राहक आपके इलाके के ही हैं। सामान मिल गया तो वे खुद आएँगे या डिलीवरी मँगवाएँगे।",\n                 "mr": "ग्राहक तुमच्याच भागातले आहेत. माल असेल तर ते स्वतः येतील किंवा डिलिव्हरी मागवतील.",\n                 "en": "The customer lives in your area. If you have it, they will come by or ask for delivery."}},\n    "what_wimb": {"about": "what WIMB is, or why we are calling",\n      "answer": {"hi": "WIMB लोगों को पास की दुकानों में सामान ढूँढने में मदद करता है, ताकि ग्राहक सीधे आपके पास आएँ।",\n                 "mr": "WIMB लोकांना जवळच्या दुकानात माल शोधायला मदत करतं, म्हणजे ग्राहक थेट तुमच्याकडे येतील.",\n                 "en": "WIMB helps people find things in nearby shops, so customers come straight to you."}}\n  },\n  "closing": [\n    {"when": {"in_stock": false}, "text": {"hi": "कोई बात नहीं, धन्यवाद! आपका दिन शुभ हो।", "mr": "काही हरकत नाही, धन्यवाद! तुमचा दिवस छान जावो.", "en": "No problem, thank you! Have a good day."}},\n    {"text": {"hi": "बहुत धन्यवाद! मैं ग्राहक को बता दूँगी। आपका दिन शुभ हो।", "mr": "खूप धन्यवाद! मी ग्राहकाला कळवते. तुमचा दिवस छान जावो.", "en": "Thank you so much! I\'ll let the customer know. Have a good day."}}\n  ]\n}\n', 'scenarios/clinic_booking.jsonl': '{"id": "clinic-hi-basic", "tags": ["hi"], "replies": {"patient_name": "राहुल शर्मा", "day": "कल", "time": "शाम पाँच बजे", "phone": "नौ आठ सात छह पाँच चार तीन दो एक शून्य", "reason": "बुखार और खाँसी", "confirm": "हाँ सही है"}, "expect": {"outcome": "done", "values": {"day": "tomorrow", "time": "17:00", "phone": "9876543210"}, "contains": {"patient_name": ["राहुल"], "reason": ["बुखार"]}}}\n{"id": "clinic-hi-sunday", "tags": ["hi", "rules"], "replies": {"patient_name": "मेरा नाम पूजा है", "day": ["रविवार को", "सोमवार को"], "time": "सुबह ग्यारह बजे", "phone": "9123456780", "reason": "पेट दर्द", "confirm": "जी हाँ"}, "expect": {"outcome": "done", "values": {"day": "monday", "time": "11:00", "phone": "9123456780"}, "contains": {"patient_name": ["पूजा"]}}}\n{"id": "clinic-hi-closed-hours", "tags": ["hi", "rules"], "replies": {"patient_name": "अमित", "day": "परसों", "time": ["रात नौ बजे", "शाम छह बजे"], "phone": "नौ नौ आठ आठ सात सात छह छह पाँच पाँच", "reason": "चेकअप", "confirm": "हाँ"}, "expect": {"outcome": "done", "values": {"day": "day_after", "time": "18:00", "phone": "9988776655"}}}\n{"id": "clinic-hi-fee-question", "tags": ["hi", "question"], "replies": {"patient_name": ["पहले बताइए डॉक्टर की फीस कितनी है?", "सीमा वर्मा"], "day": "शुक्रवार", "time": "साढ़े चार बजे", "phone": "8877665544", "reason": "सिर दर्द", "confirm": "हाँ"}, "expect": {"outcome": "done", "values": {"day": "friday", "time": "16:30", "phone": "8877665544"}, "contains": {"patient_name": ["सीमा"]}}}\n{"id": "clinic-mr-basic", "tags": ["mr"], "replies": {"patient_name": "माझं नाव सुनीता पाटील", "day": "उद्या", "time": "सकाळी अकरा वाजता", "phone": "नऊ आठ सात सहा पाच चार तीन दोन एक शून्य", "reason": "डोकेदुखी", "confirm": "हो बरोबर"}, "expect": {"outcome": "done", "lang": "mr", "values": {"day": "tomorrow", "time": "11:00", "phone": "9876543210"}, "contains": {"patient_name": ["सुनीता"]}}}\n{"id": "clinic-en-basic", "tags": ["en"], "replies": {"patient_name": "My name is Rohan Mehra", "day": "Wednesday", "time": "Around 3:30 in the afternoon", "phone": "9812345678", "reason": "back pain", "confirm": "Yes, correct"}, "expect": {"outcome": "done", "lang": "en", "values": {"day": "wednesday", "time": "15:30", "phone": "9812345678"}, "contains": {"patient_name": ["rohan"]}}}\n{"id": "clinic-hi-phone-pairs", "tags": ["hi", "phone"], "replies": {"patient_name": "गीता", "day": "मंगलवार", "time": "दोपहर दो बजे", "phone": "अट्ठानवे सत्तासी छिहत्तर पचपन चौंतीस", "reason": "दाँत में दर्द", "confirm": "हाँ"}, "expect": {"outcome": "done", "values": {"day": "tuesday", "time": "14:00", "phone": "9887765534"}}}\n{"id": "clinic-hi-all-in-one", "tags": ["hi", "many-in-one"], "replies": {"patient_name": "नाम अमित कुमार है, कल शाम पाँच बजे आना है", "phone": "9000011111", "reason": "बुखार", "confirm": "हाँ"}, "expect": {"outcome": "done", "values": {"day": "tomorrow", "time": "17:00", "phone": "9000011111"}, "contains": {"patient_name": ["अमित"]}}}\n{"id": "clinic-hi-human", "tags": ["hi", "human"], "replies": {"patient_name": "मुझे रिसेप्शन पर किसी इंसान से बात करनी है"}, "expect": {"outcome": "handoff"}}\n{"id": "clinic-hi-day-fix", "tags": ["hi", "correction"], "replies": {"patient_name": "कविता", "day": "गुरुवार", "time": "बारह बजे", "phone": "7766554433", "reason": "नहीं, बस चेकअप", "confirm": ["नहीं, गुरुवार नहीं, शुक्रवार", "हाँ अब ठीक है"]}, "expect": {"outcome": "done", "values": {"day": "friday", "time": "12:00", "phone": "7766554433"}}}\n{"id": "clinic-hinglish", "tags": ["hinglish", "latin"], "replies": {"patient_name": "mera naam Vikas hai", "day": "kal", "time": "subah das baje", "phone": "9876501234", "reason": "fever", "confirm": "haan theek hai"}, "expect": {"outcome": "done", "values": {"day": "tomorrow", "time": "10:00", "phone": "9876501234"}, "contains": {"patient_name": ["vikas"]}}}\n', 'scenarios/order_confirm.jsonl': '{"id": "order-hi-yes-evening", "tags": ["hi"], "replies": {"confirmed": "हाँ जी, कन्फर्म है", "delivery_time": "शाम को भेजिए", "confirm": "हाँ"}, "expect": {"outcome": "done", "values": {"confirmed": true, "delivery_time": "evening", "cancel_reason": null}}}\n{"id": "order-hi-cancel", "tags": ["hi"], "replies": {"confirmed": "नहीं, कैंसल कर दीजिए", "cancel_reason": "दूसरी जगह सस्ता मिल गया", "confirm": "हाँ"}, "expect": {"outcome": "done", "values": {"confirmed": false, "delivery_time": null}, "contains": {"cancel_reason": ["सस्ता"]}}}\n{"id": "order-hi-not-mine", "tags": ["hi"], "replies": {"confirmed": "मैंने तो कोई ऑर्डर किया ही नहीं", "cancel_reason": "मैंने ऑर्डर नहीं किया", "confirm": "हाँ"}, "expect": {"outcome": "done", "values": {"confirmed": false}}}\n{"id": "order-mr-yes", "tags": ["mr"], "replies": {"confirmed": "हो, पक्की आहे", "delivery_time": "संध्याकाळी द्या", "confirm": "हो"}, "expect": {"outcome": "done", "lang": "mr", "values": {"confirmed": true, "delivery_time": "evening"}}}\n{"id": "order-hi-upi", "tags": ["hi", "question"], "replies": {"confirmed": ["यूपीआई से पेमेंट कर सकते हैं क्या?", "ठीक है, कन्फर्म"], "delivery_time": "सुबह", "confirm": "हाँ"}, "expect": {"outcome": "done", "values": {"confirmed": true, "delivery_time": "morning"}}}\n{"id": "order-hinglish", "tags": ["hinglish", "latin"], "replies": {"confirmed": "haan confirm hai", "delivery_time": "afternoon mein", "confirm": "haan"}, "expect": {"outcome": "done", "values": {"confirmed": true, "delivery_time": "afternoon"}}}\n{"id": "order-hi-busy", "tags": ["hi", "busy"], "first": ["अभी मीटिंग में हूँ, बाद में कॉल करो"], "expect": {"outcome": "callback"}}\n{"id": "order-en", "tags": ["en"], "replies": {"confirmed": "Yes please, I want it", "delivery_time": "Morning is better", "confirm": "Yes"}, "expect": {"outcome": "done", "lang": "en", "values": {"confirmed": true, "delivery_time": "morning"}}}\n{"id": "order-hi-stop", "tags": ["hi", "stop"], "first": ["मुझे परेशान मत करो, दोबारा कॉल मत करना"], "expect": {"outcome": "do_not_call"}}\n', 'scenarios/shop_stock.jsonl': '{"id": "shop-hi-basic", "tags": ["hi"], "replies": {"in_stock": "हाँ जी, है", "price_inr": "दो सौ बीस रुपये का है", "delivery": "हाँ, घर पर भिजवा देंगे", "confirm": "हाँ, सही है"}, "expect": {"outcome": "done", "values": {"in_stock": true, "price_inr": 220, "delivery": true, "alternative": null}}}\n{"id": "shop-hi-out-alt", "tags": ["hi"], "replies": {"in_stock": "नहीं भाई, वो वाला खत्म हो गया", "alternative": "लेग्रैंड का सोलह ए वाला है", "confirm": "हाँ"}, "expect": {"outcome": "done", "values": {"in_stock": false, "price_inr": null, "delivery": null}, "contains": {"alternative": ["लेग्रैंड", "legrand"]}}}\n{"id": "shop-hi-out-none", "tags": ["hi"], "replies": {"in_stock": "नहीं है", "alternative": "नहीं, कुछ नहीं है", "confirm": "हाँ जी"}, "expect": {"outcome": "done", "values": {"in_stock": false, "alternative": null}}}\n{"id": "shop-hi-all-at-once", "tags": ["hi", "many-in-one"], "replies": {"in_stock": "हाँ है, दो सौ पचास का पड़ेगा, और घर पे भी पहुँचा देंगे", "confirm": "हाँ बिल्कुल"}, "expect": {"outcome": "done", "values": {"in_stock": true, "price_inr": 250, "delivery": true}}}\n{"id": "shop-hi-two-thirty", "tags": ["hi", "colloquial-price"], "replies": {"in_stock": "यस, स्टॉक में है", "price_inr": "रेट टू थर्टी है", "delivery": "डिलीवरी नहीं करते", "confirm": "हाँ"}, "expect": {"outcome": "done", "values": {"in_stock": true, "price_inr": 230, "delivery": false}}}\n{"id": "shop-hi-dedh-sau", "tags": ["hi", "fraction-price"], "replies": {"in_stock": "है", "price_inr": "डेढ़ सौ रुपये", "delivery": "नहीं", "confirm": "जी हाँ"}, "expect": {"outcome": "done", "values": {"in_stock": true, "price_inr": 150, "delivery": false}}}\n{"id": "shop-hi-saadhe-teen-sau", "tags": ["hi", "fraction-price"], "replies": {"in_stock": "हाँ रखा है", "price_inr": "साढ़े तीन सौ का आएगा", "delivery": "हाँ कर देंगे", "confirm": "सही है"}, "expect": {"outcome": "done", "values": {"in_stock": true, "price_inr": 350, "delivery": true}}}\n{"id": "shop-hi-paune", "tags": ["hi", "fraction-price"], "replies": {"in_stock": "हाँ है जी", "price_inr": "पौने तीन सौ", "delivery": "हाँ", "confirm": "हाँ"}, "expect": {"outcome": "done", "values": {"in_stock": true, "price_inr": 275, "delivery": true}}}\n{"id": "shop-hi-size-distractor", "tags": ["hi", "numbers"], "replies": {"in_stock": "बत्तीस ए वाला नहीं है, सोलह ए वाला है", "price_inr": "एक सौ अस्सी रुपये", "delivery": "नहीं जी", "confirm": "हाँ"}, "expect": {"outcome": "done", "values": {"in_stock": true, "price_inr": 180, "delivery": false}}}\n{"id": "shop-hi-who", "tags": ["hi", "who"], "first": ["कौन बोल रहा है?"], "replies": {"in_stock": "अच्छा, हाँ है", "price_inr": "दो सौ", "delivery": "नहीं", "confirm": "हाँ"}, "expect": {"outcome": "done", "values": {"in_stock": true, "price_inr": 200, "delivery": false}}}\n{"id": "shop-hi-hold", "tags": ["hi", "hold"], "first": ["एक मिनट रुकिए, देखता हूँ", "हाँ बोलिए"], "replies": {"in_stock": "हाँ, मिल गया, है", "price_inr": "दो सौ दस", "delivery": "हाँ", "confirm": "हाँ"}, "expect": {"outcome": "done", "values": {"in_stock": true, "price_inr": 210, "delivery": true}}}\n{"id": "shop-hi-busy", "tags": ["hi", "busy"], "first": ["अभी बिज़ी हूँ, बाद में कॉल करना"], "expect": {"outcome": "callback", "values": {"in_stock": null}}}\n{"id": "shop-hi-stop", "tags": ["hi", "stop"], "first": ["बार बार फोन मत करो, इंटरेस्टेड नहीं हूँ"], "expect": {"outcome": "do_not_call"}}\n{"id": "shop-hi-wrong", "tags": ["hi", "wrong-number"], "first": ["रॉन्ग नंबर है, ये दुकान नहीं है"], "expect": {"outcome": "wrong_number"}}\n{"id": "shop-hi-correction", "tags": ["hi", "correction"], "replies": {"in_stock": "है", "price_inr": "दो सौ बीस", "delivery": "हाँ", "confirm": ["नहीं, दाम दो सौ चालीस है", "हाँ अब सही है"]}, "expect": {"outcome": "done", "values": {"in_stock": true, "price_inr": 240, "delivery": true}}}\n{"id": "shop-hi-dont-know-price", "tags": ["hi", "dont-know"], "replies": {"in_stock": "हाँ है", "price_inr": "पता नहीं, मालिक से पूछना पड़ेगा", "delivery": "हाँ", "confirm": "हाँ"}, "expect": {"outcome": "partial", "values": {"in_stock": true, "price_inr": null, "delivery": true}}}\n{"id": "shop-hi-repeat", "tags": ["hi", "repeat"], "first": ["क्या? फिर से बोलिए"], "replies": {"in_stock": "हाँ है", "price_inr": "तीन सौ", "delivery": "नहीं", "confirm": "हाँ"}, "expect": {"outcome": "done", "values": {"in_stock": true, "price_inr": 300, "delivery": false}}}\n{"id": "shop-hi-question", "tags": ["hi", "question"], "replies": {"in_stock": ["कितने पीस चाहिए?", "हाँ, दस बारह पीस हैं"], "price_inr": "एक का एक सौ नब्बे", "delivery": "नहीं, आकर ले जाओ", "confirm": "हाँ"}, "expect": {"outcome": "done", "values": {"in_stock": true, "price_inr": 190, "delivery": false}}}\n{"id": "shop-hinglish-basic", "tags": ["hinglish", "latin"], "replies": {"in_stock": "haan bhai hai", "price_inr": "do sau pachas", "delivery": "nahi, delivery nahi hoti", "confirm": "haan sahi"}, "expect": {"outcome": "done", "values": {"in_stock": true, "price_inr": 250, "delivery": false}}}\n{"id": "shop-hinglish-out", "tags": ["hinglish", "latin"], "replies": {"in_stock": "nahi yaar, khatam ho gaya", "alternative": "Havells ka 32A hai", "confirm": "haan"}, "expect": {"outcome": "done", "values": {"in_stock": false}, "contains": {"alternative": ["havells", "हैवेल्स"]}}}\n{"id": "shop-mr-basic", "tags": ["mr"], "replies": {"in_stock": "हो, आहे ना", "price_inr": "दोनशे वीस रुपये", "delivery": "नाही, डिलिव्हरी देत नाही", "confirm": "हो बरोबर"}, "expect": {"outcome": "done", "lang": "mr", "values": {"in_stock": true, "price_inr": 220, "delivery": false}}}\n{"id": "shop-mr-adich", "tags": ["mr", "fraction-price"], "replies": {"in_stock": "हो आहे", "price_inr": "अडीचशे रुपयांना मिळेल", "delivery": "हो, घरपोच देतो", "confirm": "हो"}, "expect": {"outcome": "done", "values": {"in_stock": true, "price_inr": 250, "delivery": true}}}\n{"id": "shop-mr-out-alt", "tags": ["mr"], "replies": {"in_stock": "नाही, संपलं आहे", "alternative": "लेग्रँडचा आहे, तो घ्या", "confirm": "हो"}, "expect": {"outcome": "done", "values": {"in_stock": false}, "contains": {"alternative": ["लेग्रँड", "legrand"]}}}\n{"id": "shop-mr-saade", "tags": ["mr", "fraction-price"], "replies": {"in_stock": "आहे", "price_inr": "साडेतीनशे", "delivery": "नाही", "confirm": "बरोबर आहे"}, "expect": {"outcome": "done", "values": {"in_stock": true, "price_inr": 350, "delivery": false}}}\n{"id": "shop-en-basic", "tags": ["en"], "replies": {"in_stock": "Yes, we have it", "price_inr": "It\'s two hundred and twenty rupees", "delivery": "Yes, we deliver", "confirm": "Yes, that\'s right"}, "expect": {"outcome": "done", "lang": "en", "values": {"in_stock": true, "price_inr": 220, "delivery": true}}}\n{"id": "shop-hi-silence", "tags": ["silence"], "replies": {"*": ""}, "expect": {"outcome": "no_answer"}}\n{"id": "shop-hi-digits", "tags": ["hi", "digits"], "replies": {"in_stock": "हाँ है", "price_inr": "₹ 245", "delivery": "हाँ", "confirm": "हाँ"}, "expect": {"outcome": "done", "values": {"in_stock": true, "price_inr": 245, "delivery": true}}}\n{"id": "shop-hi-human", "tags": ["hi", "human"], "first": ["मुझे किसी आदमी से बात करनी है, मशीन से नहीं"], "expect": {"outcome": "handoff"}}\n', 'web/index.html': '<!doctype html>\n<html lang="hi">\n<head>\n<meta charset="utf-8">\n<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">\n<title>Wimb Voice</title>\n<style>\n  :root { --bg1:#0b1020; --bg2:#151b33; --ink:#eef1ff; --mute:#9aa3c7; --blue:#4f8cff; --violet:#9b6bff;\n          --green:#2fd38a; --red:#ff4d5e; }\n  * { box-sizing:border-box; -webkit-tap-highlight-color:transparent; }\n  html, body { margin:0; height:100%; background:radial-gradient(1200px 800px at 50% 30%, var(--bg2), var(--bg1));\n               color:var(--ink); font-family:system-ui, -apple-system, "Segoe UI", "Noto Sans Devanagari", sans-serif; }\n  body { display:flex; flex-direction:column; min-height:100dvh; padding:env(safe-area-inset-top) 16px\n         env(safe-area-inset-bottom); overflow:hidden; }\n  header { display:flex; gap:8px; justify-content:center; flex-wrap:wrap; padding:14px 0 6px; }\n  .chip { border:1px solid #2a3360; background:#121834; color:var(--mute); padding:8px 14px; border-radius:999px;\n          font-size:14px; cursor:pointer; }\n  .chip.on { background:linear-gradient(135deg, var(--blue), var(--violet)); color:white; border-color:transparent; }\n  main { flex:1; display:flex; flex-direction:column; align-items:center; justify-content:center; gap:22px; }\n  #orb { width:min(56vw, 240px); aspect-ratio:1; border-radius:50%; cursor:pointer; position:relative;\n         background:radial-gradient(circle at 35% 30%, #8fb4ff, var(--blue) 45%, var(--violet) 100%);\n         box-shadow:0 0 60px rgba(79,140,255,.35); transition:transform .25s; }\n  #orb::after { content:""; position:absolute; inset:-14px; border-radius:50%; border:2px solid rgba(143,180,255,.25); }\n  body.idle #orb { filter:saturate(.6) brightness(.8); }\n  body.listening #orb { animation:breathe 1.6s ease-in-out infinite; }\n  body.hearing #orb { transform:scale(1.12); box-shadow:0 0 90px rgba(47,211,138,.55);\n          background:radial-gradient(circle at 35% 30%, #a7ffd9, var(--green) 50%, #1a8f73 100%); }\n  body.thinking #orb { animation:spin 1.1s linear infinite;\n          background:conic-gradient(var(--blue), var(--violet), #ff7ad9, var(--blue)); }\n  body.speaking #orb { animation:talk .5s ease-in-out infinite alternate; }\n  @keyframes breathe { 50% { transform:scale(1.06); box-shadow:0 0 80px rgba(79,140,255,.5); } }\n  @keyframes spin { to { transform:rotate(360deg) scale(.92); } }\n  @keyframes talk { from { transform:scale(1); } to { transform:scale(1.09); } }\n  #status { color:var(--mute); font-size:15px; min-height:20px; text-align:center; }\n  #captions { width:100%; max-width:640px; text-align:center; display:flex; flex-direction:column; gap:10px;\n              min-height:120px; }\n  .you { color:var(--mute); font-size:15px; }\n  .bot { font-size:20px; line-height:1.45; }\n  footer { display:flex; justify-content:center; align-items:center; gap:18px; padding:12px 0 20px; }\n  .round { width:56px; height:56px; border-radius:50%; border:none; font-size:22px; cursor:pointer; color:white;\n           background:#1d2447; display:flex; align-items:center; justify-content:center; }\n  .round.end { background:var(--red); }\n  #typebar { display:none; width:100%; max-width:640px; gap:8px; margin:0 auto 10px; }\n  #typebar.on { display:flex; }\n  #typebar input { flex:1; padding:12px 14px; border-radius:12px; border:1px solid #2a3360; background:#121834;\n                   color:var(--ink); font-size:16px; }\n  #typebar button { padding:0 16px; border-radius:12px; border:none; background:var(--blue); color:white; font-size:16px; }\n  #found { font-size:13px; color:var(--mute); max-width:640px; text-align:center; white-space:pre-wrap; }\n  label.save { font-size:12px; color:var(--mute); display:flex; gap:6px; align-items:center; justify-content:center; }\n</style>\n</head>\n<body class="idle">\n<header id="modes"></header>\n<main>\n  <div id="orb" role="button" aria-label="talk"></div>\n  <div id="status">Tap the circle and start talking</div>\n  <div id="captions"><div class="you" id="you"></div><div class="bot" id="bot"></div></div>\n  <div id="found"></div>\n</main>\n<div id="typebar"><input id="typed" placeholder="Type instead…" autocomplete="off"><button id="send">Send</button></div>\n<footer>\n  <button class="round" id="kbd" title="Type">⌨</button>\n  <button class="round end" id="end" title="End">✕</button>\n  <label class="save"><input type="checkbox" id="save" checked> save my voice to improve Wimb</label>\n</footer>\n<script src="{{ORT_BASE}}ort.js"></script>\n<script src="{{VAD_BASE}}bundle.min.js"></script>\n<script>\nconst VAD_BASE = "{{VAD_BASE}}", ORT_BASE = "{{ORT_BASE}}";\nconst $ = id => document.getElementById(id);\nconst player = new Audio();\nlet mode = "assistant", sid = null, vadObj = null, recorder = null, started = false, busy = false;\nlet queue = [], speaking = false, ending = false, turnSeq = 0;\n\nfunction state(s, text) {\n  document.body.className = s;\n  if (text !== undefined) $("status").textContent = text;\n}\nconst SILENT = "data:audio/wav;base64,UklGRiQAAABXQVZFZm10IBAAAAABAAEAQB8AAIA+AAACABAAZGF0YQAAAAA=";\n\nasync function loadModes() {\n  const r = await fetch("api/modes");\n  const modes = await r.json();\n  $("modes").innerHTML = "";\n  for (const m of modes) {\n    const b = document.createElement("button");\n    b.className = "chip" + (m.id === mode ? " on" : "");\n    b.textContent = m.name;\n    b.onclick = () => switchMode(m.id);\n    $("modes").appendChild(b);\n  }\n}\n\nfunction wav(samples, rate = 16000) {\n  const b = new ArrayBuffer(44 + samples.length * 2), v = new DataView(b);\n  const w = (o, s) => { for (let i = 0; i < s.length; i++) v.setUint8(o + i, s.charCodeAt(i)); };\n  w(0, "RIFF"); v.setUint32(4, 36 + samples.length * 2, true); w(8, "WAVE"); w(12, "fmt ");\n  v.setUint32(16, 16, true); v.setUint16(20, 1, true); v.setUint16(22, 1, true); v.setUint32(24, rate, true);\n  v.setUint32(28, rate * 2, true); v.setUint16(32, 2, true); v.setUint16(34, 16, true); w(36, "data");\n  v.setUint32(40, samples.length * 2, true);\n  for (let i = 0; i < samples.length; i++) {\n    const s = Math.max(-1, Math.min(1, samples[i]));\n    v.setInt16(44 + i * 2, s < 0 ? s * 0x8000 : s * 0x7fff, true);\n  }\n  return new Blob([b], { type: "audio/wav" });\n}\n\nasync function* lines(resp) {\n  const reader = resp.body.getReader(), dec = new TextDecoder();\n  let buf = "";\n  while (true) {\n    const { value, done } = await reader.read();\n    if (done) break;\n    buf += dec.decode(value, { stream: true });\n    let i;\n    while ((i = buf.indexOf("\\n")) >= 0) {\n      const l = buf.slice(0, i).trim(); buf = buf.slice(i + 1);\n      if (l) yield JSON.parse(l);\n    }\n  }\n  if (buf.trim()) yield JSON.parse(buf);\n}\n\nfunction b64blob(b64, type = "audio/wav") {\n  const bin = atob(b64), u = new Uint8Array(bin.length);\n  for (let i = 0; i < bin.length; i++) u[i] = bin.charCodeAt(i);\n  return new Blob([u], { type });\n}\n\nfunction playNext() {\n  if (!queue.length) {\n    speaking = false;\n    if (!busy) afterSpeaking();\n    return;\n  }\n  speaking = true;\n  state("speaking", "Wimb is speaking… (tap to interrupt)");\n  player.src = URL.createObjectURL(queue.shift());\n  player.volume = 1;\n  player.play().catch(() => playNext());\n}\nplayer.onended = playNext;\n\nfunction stopSpeaking() {\n  queue = []; speaking = false;\n  try { player.pause(); } catch (e) {}\n}\n\nfunction afterSpeaking() {\n  if (ending) { state("idle", "Call ended. Tap the circle to start again."); stopListening(); sid = null;\n                started = false; ending = false; return; }\n  listen();\n}\n\nfunction listen() {\n  if (vadObj) { vadObj.start(); state("listening", "Listening… speak any time"); }\n  else if (recorder) state("listening", "Tap the circle, speak, tap again to send");\n}\n\nfunction stopListening() { if (vadObj) vadObj.pause(); }\n\nasync function stream(resp, seq, during = false) {\n  for await (const ev of lines(resp)) {\n    if (during && ev.type === "heard") {  // said while Wimb was speaking\n      if (ev.ignored) { if (speaking) player.volume = 1; return; }  // only Wimb\'s own voice, picked up again\n      seq = ++turnSeq; stopSpeaking(); busy = true; during = false;  // a real interruption: this turn now leads\n    }\n    if (seq !== turnSeq) return;  // an older turn, interrupted\n    if (ev.type === "session") sid = ev.sid;\n    else if (ev.type === "heard") { $("you").textContent = ev.text ? "“" + ev.text + "”" : ""; $("bot").textContent = ""; }\n    else if (ev.type === "say") {\n      $("bot").textContent = ($("bot").textContent + " " + ev.text).trim();\n      if (ev.audio) { queue.push(b64blob(ev.audio)); if (!speaking) playNext(); }\n    } else if (ev.type === "done") {\n      if (ev.found) $("found").textContent = ev.found;\n      if (ev.end) ending = true;\n    } else if (ev.type === "error") { $("bot").textContent = ev.text; }\n  }\n  if (seq !== turnSeq) return;\n  busy = false;\n  if (!speaking) afterSpeaking();\n}\n\nasync function startSession() {\n  const seq = ++turnSeq;\n  busy = true; state("thinking", "Connecting…");\n  const r = await fetch("api/start", { method: "POST", headers: { "Content-Type": "application/json" },\n                                      body: JSON.stringify({ mode, save: $("save").checked }) });\n  await stream(r, seq);\n}\n\nasync function sendAudio(blob, during) {\n  if (!sid) return;\n  if (during) {\n    const r = await fetch(`api/turn?sid=${sid}&during=1`, { method: "POST", body: blob,\n                                                           headers: { "Content-Type": blob.type } });\n    await stream(r, turnSeq, true);\n    return;\n  }\n  const seq = ++turnSeq;\n  busy = true; state("thinking", "Thinking…");\n  const r = await fetch(`api/turn?sid=${sid}&during=0`, { method: "POST", body: blob,\n                                                         headers: { "Content-Type": blob.type } });\n  await stream(r, seq);\n}\n\nasync function sendText(text) {\n  if (!text.trim()) return;\n  if (!sid) { $("status").textContent = "Tap the circle first"; return; }\n  stopSpeaking(); stopListening();\n  const seq = ++turnSeq;\n  busy = true; state("thinking", "Thinking…");\n  const r = await fetch("api/text", { method: "POST", headers: { "Content-Type": "application/json" },\n                                     body: JSON.stringify({ sid, text }) });\n  await stream(r, seq);\n}\n\nasync function setupMic() {\n  try {\n    vadObj = await vad.MicVAD.new({\n      baseAssetPath: VAD_BASE, onnxWASMBasePath: ORT_BASE,\n      positiveSpeechThreshold: 0.6, negativeSpeechThreshold: 0.4, redemptionFrames: 9, minSpeechFrames: 4,\n      onSpeechStart: () => { if (speaking) player.volume = 0.3; else if (!busy) state("hearing", "Hearing you…"); },\n      onVADMisfire: () => { if (speaking) player.volume = 1; else if (!busy) listen(); },\n      onSpeechEnd: (audio) => {\n        const during = speaking;\n        if (!during) stopListening();\n        sendAudio(wav(audio), during);\n      },\n    });\n    return true;\n  } catch (e) {\n    console.warn("VAD unavailable, tap to talk instead", e);\n    vadObj = null;\n  }\n  try {  // tap to talk: tap to start, tap again to send\n    const s = await navigator.mediaDevices.getUserMedia({ audio: { echoCancellation: true, noiseSuppression: true } });\n    recorder = { stream: s, rec: null, chunks: [] };\n    return true;\n  } catch (e) {\n    state("idle", "No microphone: allow it in the browser, or type with ⌨");\n    $("typebar").classList.add("on");\n    return false;\n  }\n}\n\nfunction tapToTalk() {\n  if (!recorder) return;\n  if (recorder.rec && recorder.rec.state === "recording") { recorder.rec.stop(); return; }\n  recorder.chunks = [];\n  recorder.rec = new MediaRecorder(recorder.stream);\n  recorder.rec.ondataavailable = e => recorder.chunks.push(e.data);\n  recorder.rec.onstop = () => sendAudio(new Blob(recorder.chunks, { type: recorder.rec.mimeType || "audio/webm" }), false);\n  recorder.rec.start();\n  state("hearing", "Recording… tap again to send");\n}\n\n$("orb").onclick = async () => {\n  if (!started) {\n    started = true;\n    player.src = SILENT; player.play().catch(() => {});  // lets the phone play Wimb\'s voice later\n    state("thinking", "Starting the microphone…");\n    await setupMic();\n    await startSession();\n    return;\n  }\n  if (speaking) { stopSpeaking(); if (!busy) listen(); return; }  // interrupt\n  if (!vadObj) tapToTalk();\n};\n\nasync function switchMode(id) {\n  mode = id; stopSpeaking(); stopListening();\n  if (sid) fetch("api/end", { method: "POST", headers: { "Content-Type": "application/json" },\n                              body: JSON.stringify({ sid }) });\n  sid = null; $("you").textContent = ""; $("bot").textContent = ""; $("found").textContent = "";\n  await loadModes();\n  if (started) await startSession();\n  else state("idle", "Tap the circle and start talking");\n}\n\n$("end").onclick = () => {\n  stopSpeaking(); stopListening();\n  if (sid) fetch("api/end", { method: "POST", headers: { "Content-Type": "application/json" },\n                              body: JSON.stringify({ sid }) });\n  sid = null; started = false; state("idle", "Call ended. Tap the circle to start again.");\n};\n$("kbd").onclick = () => { $("typebar").classList.toggle("on"); $("typed").focus(); };\n$("send").onclick = () => { sendText($("typed").value); $("typed").value = ""; };\n$("typed").onkeydown = e => { if (e.key === "Enter") $("send").onclick(); };\nloadModes();\n</script>\n</body>\n</html>\n'}

SRC = "/tmp/voicebot_src"
for name, body in PACKAGE_FILES.items():
    path = os.path.join(SRC, "voicebot", name)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(body)
sys.path.insert(0, SRC)
os.makedirs(WORKING_DIR, exist_ok=True)

from voicebot import evaluate, tasks
from voicebot.engine import Call
from voicebot.lang import detect

RESULTS = {"mode": MODE, "gpu": None, "ollama_version": None, "brains": {}, "best_brain": None, "text_test": {},
           "asr": {"tried": [], "bench": {}}, "tts": {"tried": [], "samples": []}, "best_asr": None,
           "spoken_test": {}, "turn_seconds": {}, "live": {}, "versions": {}, "summary": [], "errors": []}
OLLAMA_LOG = "/tmp/ollama.log"  # what `ollama serve` prints (cell 4)


def save_results():
    """Write RESULTS to voicebot_results.json. Done after every stage, so a stop part-way loses nothing."""
    if os.path.isdir(WORKING_DIR):
        with open(os.path.join(WORKING_DIR, "voicebot_results.json"), "w", encoding="utf-8") as fh:
            json.dump(RESULTS, fh, ensure_ascii=False, indent=1)


def failed(stage, error):
    """Record a failed stage and go on (Kaggle's Run all stops at the first error that is not caught)."""
    message = f"{stage}: {error!r}" if isinstance(error, Exception) else f"{stage}: {error}"
    print("FAILED", message)
    RESULTS["errors"].append(message)
    save_results()


def sh(command, timeout=120):
    """What a shell command printed, or "" when the command is missing or fails."""
    try:
        return subprocess.run(command, shell=True, capture_output=True, text=True, timeout=timeout).stdout.strip()
    except Exception:
        return ""


def generate(body):
    request = urllib.request.Request(OLLAMA_URL + "/api/generate", json.dumps(body).encode(),
                                     {"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=600) as resp:
        return json.load(resp)


def in_memory():
    """The model as Ollama holds it now (from /api/ps: size, size_vram, ...), or None when it is not loaded."""
    with urllib.request.urlopen(OLLAMA_URL + "/api/ps", timeout=10) as resp:
        models = json.load(resp).get("models") or []
    return next((m for m in models if m.get("name", "").split(":")[0] == "wimb-shop"), None)


def processor():
    """Where the model runs now, as `ollama ps` says it: "100% GPU", "100% CPU", "25%/75% CPU/GPU", or None."""
    try:
        m = in_memory()
    except Exception:
        return None
    if not m or not m.get("size"):
        return None
    cpu = round(100 * (m["size"] - (m.get("size_vram") or 0)) / m["size"])
    return "100% GPU" if cpu <= 0 else "100% CPU" if cpu >= 100 else f"{cpu}%/{100 - cpu}% CPU/GPU"

In [ ]:
#@title 3. Install (about 3–5 minutes)
def pip(*packages, timeout=1800):
    done = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], capture_output=True, text=True,
                          timeout=timeout)
    if done.returncode != 0:
        raise RuntimeError((done.stdout + done.stderr)[-1500:])


def hf_token():
    """The Hugging Face token: Kaggle Secrets "HF_TOKEN" (Add-ons → Secrets), else the env. Never printed."""
    try:
        from kaggle_secrets import UserSecretsClient

        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        token = None
    return (token or os.environ.get("HF_TOKEN") or "").strip() or None


HF_TOKEN = hf_token()
print("Hugging Face token:", "found" if HF_TOKEN else "none (gated models are skipped)")
RESULTS["hf_token"] = bool(HF_TOKEN)
missing = [p for p in ("ffmpeg", "espeak-ng", "zstd") if not shutil.which(p)]
if missing:  # ffmpeg reads phone audio; espeak-ng: kokoro's Hindi pronunciation; zstd: Ollama's download
    sh("apt-get update -qq; DEBIAN_FRONTEND=noninteractive apt-get install -y -qq ffmpeg espeak-ng zstd", timeout=900)
for packages in (["faster-whisper", "soundfile", "librosa", "scipy"], ["kokoro-onnx"], ["datasets", "huggingface_hub"]):
    try:
        pip(*packages)
    except Exception as e:
        failed("pip install " + " ".join(packages), e)
try:  # Kokoro on the GPU (the CPU onnxruntime stays as the fallback)
    pip("onnxruntime-gpu")
except Exception as e:
    print("onnxruntime-gpu did not install (Kokoro runs on the CPU):", f"{e!r}"[:300])
if HF_TOKEN and "parler" in TTS:
    try:
        pip("parler-tts==0.2.3")
    except Exception as e:
        failed("pip install parler-tts==0.2.3", e)
if MODE == "live":
    try:
        pip("gradio==5.50.0")
    except Exception as e:
        failed("pip install gradio==5.50.0", e)
import importlib.metadata as md

for p in ("torch", "transformers", "faster-whisper", "ctranslate2", "kokoro-onnx", "onnxruntime", "onnxruntime-gpu",
          "parler-tts", "datasets", "gradio"):
    try:
        RESULTS["versions"][p] = md.version(p)
    except md.PackageNotFoundError:
        RESULTS["versions"][p] = None
print(RESULTS["versions"])
save_results()

In [ ]:
#@title 4. Brain: Ollama and the small models
OLLAMA_URL = "http://127.0.0.1:11434"


def ollama_version():
    try:
        with urllib.request.urlopen(OLLAMA_URL + "/api/version", timeout=5) as resp:
            return json.load(resp).get("version")
    except Exception:
        return None


if not ollama_version():
    if not shutil.which("ollama"):
        # zstd unpacks Ollama's download (.tar.zst). The download already holds the CUDA libraries; lspci is
        # left out on purpose: with it (and no nvidia-smi) install.sh would try to install NVIDIA drivers.
        if not shutil.which("zstd"):
            sh("apt-get update -qq; DEBIAN_FRONTEND=noninteractive apt-get install -y -qq zstd", timeout=600)
        install = subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True,
                                 capture_output=True, text=True, timeout=1200)
        print("\n".join((install.stdout + install.stderr).replace("\r", "\n").splitlines()[-15:]))
        if not shutil.which("ollama"):
            failed("install Ollama", (install.stdout + install.stderr)[-1500:])
            raise RuntimeError("Ollama did not install (see above)")
    print(sh("ollama --version 2>&1"))
    log = open(OLLAMA_LOG, "w")  # stays open: the server writes to it
    server = subprocess.Popen(["ollama", "serve"], env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"},
                              stdout=log, stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_version() or server.poll() is not None:
            break
        time.sleep(1)
if not ollama_version():
    tail = sh(f"tail -n 40 {OLLAMA_LOG}")
    print(tail)
    failed("start Ollama", tail[-1500:])
    raise RuntimeError("Ollama did not start (its log is above)")
RESULTS["ollama_version"] = ollama_version()
RESULTS["gpu"] = (sh("nvidia-smi --query-gpu=name --format=csv,noheader").splitlines() or [None])[0]
print("Ollama", RESULTS["ollama_version"], "is running. GPU:", RESULTS["gpu"])
save_results()

LLM_SECONDS = {}  # brain -> seconds of each model call


def pull(model):
    done = subprocess.run(["ollama", "pull", model], capture_output=True, text=True, timeout=1800)
    return done.returncode == 0, re.sub(r"\x1b\[[0-9;?]*[A-Za-z]", "", done.stderr + done.stdout)[-400:]


def make_llm(model):
    """llm(prompt, schema) -> the model's JSON text, through Ollama (JSON forced by the schema)."""
    LLM_SECONDS.setdefault(model, [])

    def llm(prompt, schema):
        started = time.perf_counter()
        data = generate({"model": model, "prompt": prompt, "format": schema, "stream": False, "keep_alive": "30m",
                         "options": {"temperature": 0, "num_predict": 300, "num_ctx": 4096}})
        LLM_SECONDS[model].append(round(time.perf_counter() - started, 3))
        return data.get("response", "")

    return llm


READY = []  # the brains that pulled and answered
for model in BRAINS if MODE == "eval" else BRAINS[:2]:
    started = time.time()
    ok, log = pull(model)
    entry = {"pulled": ok, "pull_seconds": round(time.time() - started), "log": None if ok else log}
    if ok:
        try:
            make_llm(model)("Reply with JSON: {\"ok\": true}", {"type": "object", "properties": {"ok": {"type": "boolean"}},
                                                                 "required": ["ok"]})
            READY.append(model)
        except Exception as e:
            entry["log"] = f"{e!r}"[:400]
    RESULTS["brains"][model] = entry
    print(model, "ready" if model in READY else f"NOT ready: {entry['log']}")
    if MODE == "live" and READY:
        break
save_results()

In [ ]:
#@title 5. Test calls, typed: rules only, then each brain
SCENARIOS = evaluate.scenarios()


def text_test(name, llm):
    started = time.time()
    results = [evaluate.play(s, llm=llm) for s in SCENARIOS]
    lines = evaluate.summary_lines(results, f"{name}: ")
    print("\n".join(lines))
    calls = LLM_SECONDS.get(name, [])
    RESULTS["text_test"][name] = {
        **evaluate.summary(results), "lines": lines, "minutes": round((time.time() - started) / 60, 1),
        "model_seconds_avg": round(sum(calls) / len(calls), 2) if calls else None,
        "failed": [{"id": r["id"], "wrong": r["wrong"], "turns": evaluate.transcript(r)} for r in results if r["wrong"]]}
    save_results()
    return results


if MODE == "eval":
    text_test("rules only", None)
    for model in READY:
        LLM_SECONDS[model] = []
        text_test(model, make_llm(model))
    ranked = sorted(READY, key=lambda m: (-RESULTS["text_test"][m]["all"]["passed"],
                                          RESULTS["text_test"][m]["model_seconds_avg"] or 99))
    RESULTS["best_brain"] = ranked[0] if ranked else None
    if ranked:  # for a live call, speed matters: the fastest brain within one test call of the best
        top = RESULTS["text_test"][ranked[0]]["all"]["passed"]
        quick = [m for m in ranked if RESULTS["text_test"][m]["all"]["passed"] >= top - 1
                 and (RESULTS["text_test"][m]["model_seconds_avg"] or 99) <= 3]
        RESULTS["live_brain"] = min(quick, key=lambda m: RESULTS["text_test"][m]["model_seconds_avg"]) if quick else None
else:
    RESULTS["best_brain"] = READY[0] if READY else None
print("Brain for the rest:", RESULTS["best_brain"] or "none (rules only)")
LLM = make_llm(RESULTS["best_brain"]) if RESULTS["best_brain"] else None
save_results()

In [ ]:
#@title 6. Ears and mouth: load speech to text and text to speech
from voicebot.audio import ASR_INFO, TTS_INFO, Ears, Mouth, load_first, phone_line, resample, to_int16

ASR_ORDER = (["indic-conformer"] if HF_TOKEN else []) + [a for a in ASR if a != "indic-conformer"]
EARS = {}  # every speech-to-text model that loads (eval) or the first (live)
for backend in ASR_ORDER:
    ears, tried = load_first("ears", [backend], HF_TOKEN)
    RESULTS["asr"]["tried"] += tried
    if ears is not None:
        EARS[backend] = ears
        if MODE == "live":
            break
MOUTHS = {}
for backend in TTS:
    mouth, tried = load_first("mouth", [backend], HF_TOKEN)
    RESULTS["tts"]["tried"] += tried
    if mouth is not None:
        MOUTHS[backend] = mouth
        if MODE == "live":
            break
print("Ears:", list(EARS) or "none", "| Mouths:", list(MOUTHS) or "none")
save_results()

In [ ]:
#@title 7. Speech to text on real speech, clean and as a phone line
import io
import unicodedata

import numpy as np
import soundfile as sf


def norm(t):
    t = unicodedata.normalize("NFC", str(t or "")).lower().replace("\u093c", "").replace("\u0901", "\u0902")
    return " ".join(re.sub(r"[^\w\s]", " ", t).split())


def edits(a, b):
    row = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        prev, row[0] = row[0], i
        for j, y in enumerate(b, 1):
            prev, row[j] = row[j], min(row[j] + 1, row[j - 1] + 1, prev + (x != y))
    return row[-1]


def error_rates(pairs):
    """(word error rate %, character error rate %) over (reference, heard) pairs."""
    w = sum(edits(norm(r).split(), norm(h).split()) for r, h in pairs)
    c = sum(edits(norm(r).replace(" ", ""), norm(h).replace(" ", "")) for r, h in pairs)
    nw = sum(len(norm(r).split()) for r, _ in pairs) or 1
    nc = sum(len(norm(r).replace(" ", "")) for r, _ in pairs) or 1
    return round(100 * w / nw, 1), round(100 * c / nc, 1)


def decode(audio):
    if isinstance(audio, dict) and audio.get("bytes"):
        data, rate = sf.read(io.BytesIO(audio["bytes"]), dtype="float32", always_2d=True)
    elif isinstance(audio, dict) and audio.get("array") is not None:
        data, rate = np.asarray(audio["array"], dtype=np.float32)[:, None], audio["sampling_rate"]
    else:
        data, rate = sf.read(audio["path"], dtype="float32", always_2d=True)
    return resample(data.mean(axis=1), rate, 16000)


def rows_from(ds, n, text_keys, group_keys=()):
    from datasets import Audio

    columns = list(getattr(ds, "column_names", None) or next(iter(ds.take(1))).keys())
    audio_key = next(k for k in columns if k in ("audio", "audio_filepath", "file") or "audio" in k)
    text_key = next(k for k in text_keys if k in columns)
    group_key = next((k for k in group_keys if k in columns), None)
    try:
        ds = ds.cast_column(audio_key, Audio(decode=False))
    except Exception:
        pass
    out = []
    for row in ds:
        out.append({"audio": decode(row[audio_key]), "text": row[text_key],
                    "group": str(row[group_key]) if group_key else None})
        if len(out) >= n:
            break
    return out, {"text": text_key, "group": group_key}


def fleurs(lang, n):
    from datasets import load_dataset

    code = {"hi": "hi_in", "mr": "mr_in"}[lang]
    errors = []
    for how in ("parquet", "script"):
        try:
            if how == "parquet":
                ds = load_dataset("parquet", split="train", streaming=True, data_files=(
                    f"hf://datasets/google/fleurs@refs/convert/parquet/{code}/test/*.parquet"))
            else:
                ds = load_dataset("google/fleurs", code, split="test", streaming=True, trust_remote_code=True)
            rows, keys = rows_from(ds, n, ("raw_transcription", "transcription"))
            return rows, f"google/fleurs {code} test ({how})"
        except Exception as e:
            errors.append(f"{how}: {e!r}"[:300])
    raise RuntimeError(" | ".join(errors))


def lahaja(n):
    from datasets import load_dataset

    ds = load_dataset("ai4bharat/Lahaja", split="test", streaming=True, token=HF_TOKEN)
    rows, keys = rows_from(ds, n, ("text", "transcription", "transcript", "sentence", "verbatim"),
                           ("native_state", "state", "native_district", "district", "region"))
    return rows, "ai4bharat/Lahaja test (Hindi from many districts)"


def bench(name, rows, lang):
    out = {}
    for backend, ears in EARS.items():
        for condition in ("clean", "phone"):
            pairs, seconds = [], []
            for i, row in enumerate(rows):
                audio = row["audio"] if condition == "clean" else phone_line(row["audio"], seed=i)
                heard, _, s = ears.hear(audio, hint=lang, allowed=[lang], detect=False)
                pairs.append((row["text"], heard))
                seconds.append(s)
            wer, cer = error_rates(pairs)
            key = f"{backend} / {condition}"
            out[key] = {"wer": wer, "cer": cer, "seconds_avg": round(sum(seconds) / len(seconds), 2),
                        "examples": [{"said": r, "heard": h} for r, h in pairs[:3]]}
            groups = sorted({r["group"] for r in rows if r["group"]})
            if groups and condition == "phone":
                out[key]["by_group"] = {g: error_rates([p for p, r in zip(pairs, rows) if r["group"] == g])[0]
                                        for g in groups}
            print(f"{name} | {key}: WER {wer}%  CER {cer}%  ({out[key]['seconds_avg']} s per clip)")
        if lang == "mr" and ears.backend != "indic-conformer":  # Whisper's Marathi is weak: is "hi" better?
            pairs = [(r["text"], ears.hear(phone_line(r["audio"], seed=i), hint="hi", allowed=["hi"],
                                           detect=False)[0]) for i, r in enumerate(rows)]
            out[f"{backend} / phone, decoded as Hindi"] = dict(zip(("wer", "cer"), error_rates(pairs)))
    RESULTS["asr"]["bench"][name] = out
    save_results()


if MODE == "eval" and EARS:
    sets = [("Hindi (FLEURS)", "hi", lambda: fleurs("hi", BENCH_ROWS)),
            ("Marathi (FLEURS)", "mr", lambda: fleurs("mr", BENCH_ROWS))]
    if HF_TOKEN:
        sets.append(("Hindi accents (Lahaja)", "hi", lambda: lahaja(2 * BENCH_ROWS)))
    for name, lang, load in sets:
        try:
            rows, source = load()
            print(f"{name}: {len(rows)} clips from {source}")
            RESULTS["asr"]["bench"][name + " source"] = source
            bench(name, rows, lang)
        except Exception as e:
            failed(f"speech-to-text test {name}", e)
    scores = {b: [v[f"{b} / phone"]["wer"] for k, v in RESULTS["asr"]["bench"].items()
                  if isinstance(v, dict) and f"{b} / phone" in v] for b in EARS}
    scores = {b: sum(s) / len(s) for b, s in scores.items() if s}
    RESULTS["best_asr"] = min(scores, key=scores.get) if scores else next(iter(EARS))
elif EARS:
    RESULTS["best_asr"] = next(iter(EARS))
print("Speech to text for the rest:", RESULTS["best_asr"])
save_results()

In [ ]:
#@title 8. Test calls, spoken over a phone line
from voicebot.speech_text import for_voice

PERSON_MOUTH = next((MOUTHS[b] for b in ("parler", "kokoro", "mms") if b in MOUTHS), None)  # the most varied voices
BOT_MOUTH = next(iter(MOUTHS.values()), None)
PERSON_MOUTHS = [MOUTHS[b] for b in ("parler", "kokoro", "mms") if b in MOUTHS]  # the test is run with each voice
VOICES = itertools.cycle(["person", "person2", "person3"])


def spoken_test(label, use_prompt, demo=False, person_mouth=None):
    person_mouth = person_mouth or PERSON_MOUTH
    ears = EARS[RESULTS["best_asr"]]
    rows = [s for s in SCENARIOS if "latin" not in s.get("tags", [])]  # speech to text writes Devanagari anyway
    heard_log, demos = [], {}

    def run(scenario):
        voice = next(VOICES)
        audio_parts = []

        def say_and_hear(line, call):
            lang = detect(line, default=call.lang)[0]
            try:
                audio, rate, t_say = person_mouth.say(for_voice(line, lang), lang, voice)
            except Exception as e:  # this line can't be spoken: the bot reads it as typed (counted)
                heard_log.append({"id": scenario["id"], "said": line, "heard": line, "seconds": 0, "unspoken": repr(e)[:200]})
                return line, 0.0
            audio = phone_line(resample(audio, rate, 16000), seed=len(heard_log))
            bot = [t.text for t in call.turns if t.who == "bot"]
            heard, _, seconds = ears.hear(audio, hint=call.lang, allowed=call.task.languages,
                                          prompt=(bot[-1][-200:] if use_prompt and bot else None))
            heard_log.append({"id": scenario["id"], "said": line, "heard": heard, "seconds": seconds})
            audio_parts.append(("person", audio))
            return heard, seconds

        def on_bot(reply):
            if demo and BOT_MOUTH is not None and reply.speech:
                audio, rate, _ = BOT_MOUTH.say(reply.speech, reply.lang, "bot")
                audio_parts.append(("bot", resample(audio, rate, 16000)))

        result = evaluate.play(scenario, llm=LLM, say_and_hear=say_and_hear, on_bot=on_bot if demo else None)
        if demo and scenario["task"] not in demos and not result["wrong"]:
            gap = np.zeros(int(0.35 * 16000), dtype=np.float32)
            demos[scenario["task"]] = np.concatenate([x for _, a in audio_parts for x in (a, gap)])
        return result

    started = time.time()
    results = []
    for s in rows:
        try:
            results.append(run(s))
        except Exception as e:  # one broken call must not stop the test: it counts as failed
            failed(f"spoken test {label}: {s['id']}", e)
            results.append({"id": s["id"], "task": s["task"], "tags": s.get("tags", []), "wrong": [f"crashed: {e!r}"[:200]],
                            "lines": [], "model_calls": 0, "model_errors": 0, "turns": []})
    lines = evaluate.summary_lines(results, f"Spoken ({label}): ")
    print("\n".join(lines))
    misheard = [h for h in heard_log if norm(h["said"]) != norm(h["heard"]) and not h.get("unspoken")]
    unspoken = sum(bool(h.get("unspoken")) for h in heard_log)
    RESULTS["spoken_test"][label] = {**evaluate.summary(results), "lines": lines,
                                     "minutes": round((time.time() - started) / 60, 1),
                                     "lines_heard_exactly": f"{len(heard_log) - len(misheard) - unspoken} of "
                                                            f"{len(heard_log) - unspoken}",
                                     "lines_not_spoken": unspoken,
                                     "misheard": misheard[:40],
                                     "failed": [{"id": r["id"], "wrong": r["wrong"], "turns": evaluate.transcript(r)}
                                                for r in results if r["wrong"]]}
    for task_id, audio in demos.items():
        path = os.path.join(WORKING_DIR, f"demo_call_{task_id}.wav")
        sf.write(path, audio, 16000)
        RESULTS["spoken_test"][label].setdefault("demo_calls", []).append(path)
    save_results()


if MODE == "eval" and EARS and PERSON_MOUTHS:
    # The callers' lines are spoken by each test voice, then sent through a phone line (8 kHz, μ-law, noise).
    # Eval 3–4 showed the ears mishearing the synthetic voice's short words ("राहुल" as "जाहुल" every time): two
    # voices tell whether the test voice or the ears is the problem. The hint (the bot's question) did not help.
    for i, mouth in enumerate(PERSON_MOUTHS):
        label = f"voice {mouth.backend}"
        try:
            spoken_test(label, False, demo=i == 0, person_mouth=mouth)
        except Exception as e:
            failed(f"spoken test ({label})", e)

In [ ]:
#@title 9. Voices: samples and speed
SAMPLES = {"hi": "ठीक है। तो मैंने नोट किया: 16A MCB है, दाम ₹250, डिलीवरी हो जाएगी। सही है?",
           "mr": "ठीक आहे. तर मी नोंद केलं: 16A MCB आहे, किंमत ₹220, डिलिव्हरी नाही. बरोबर आहे का?",
           "en": "Okay. So I have: 16A MCB is in stock, price ₹250, you deliver. Is that right?"}
if MODE == "eval":
    for backend, mouth in MOUTHS.items():
        for lang, line in SAMPLES.items():
            try:
                audio, rate, seconds = mouth.say(for_voice(line, lang), lang, "bot")
                path = os.path.join(WORKING_DIR, f"voice_{backend}_{lang}.wav")
                sf.write(path, audio, rate)
                RESULTS["tts"]["samples"].append({"backend": backend, "lang": lang, "seconds": seconds,
                                                  "audio_seconds": round(len(audio) / rate, 1), "file": path})
                print(f"{backend} {lang}: {seconds} s for {len(audio) / rate:.1f} s of speech -> {path}")
            except Exception as e:
                failed(f"voice {backend} {lang}", e)
    save_results()

In [ ]:
#@title 10. One turn, end to end, timed
if MODE == "eval" and EARS and BOT_MOUTH is not None:  # one turn, end to end, timed: hear → understand → speak
    ears = EARS[RESULTS["best_asr"]]
    times = {"hear": [], "understand": [], "speak": []}
    for i, line in enumerate(["हाँ है, ढाई सौ का पड़ेगा", "नहीं, डिलीवरी नहीं करते", "हो, आहे, दोनशे वीस रुपये"]):
        call = Call(tasks.load("shop_stock"), llm=LLM)
        call.start()
        audio, rate, _ = PERSON_MOUTH.say(line, detect(line)[0], "person")
        audio = phone_line(resample(audio, rate, 16000), seed=i)
        heard, _, t_hear = ears.hear(audio, hint="hi", allowed=["hi", "mr", "en"])
        started = time.perf_counter()
        reply = call.hear(heard)
        t_think = time.perf_counter() - started
        _, _, t_speak = BOT_MOUTH.say(reply.speech, reply.lang, "bot")
        for k, v in (("hear", t_hear), ("understand", t_think), ("speak", t_speak)):
            times[k].append(v)
    RESULTS["turn_seconds"] = {k: round(sum(v) / len(v), 2) for k, v in times.items()}
    RESULTS["turn_seconds"]["total"] = round(sum(RESULTS["turn_seconds"].values()), 2)
    print("One turn, average seconds:", RESULTS["turn_seconds"])
    save_results()

In [ ]:
#@title 11. Live: the web page (MODE = "live")
def notify(url):
    """Post the page's link to ntfy.sh/NOTIFY_TOPIC, where the GitHub job waits for it."""
    if not NOTIFY_TOPIC:
        return
    for _ in range(5):
        try:
            urllib.request.urlopen(urllib.request.Request(f"https://ntfy.sh/{NOTIFY_TOPIC}", data=url.encode(),
                                                          method="POST"), timeout=20).read()
            return
        except Exception as e:
            print("ntfy:", e)
            time.sleep(5)


if MODE == "live":
    from voicebot.assistant import ollama_chat
    from voicebot.server import make_app, open_tunnel, serve

    ears = EARS.get(RESULTS["best_asr"]) if EARS else None
    mouth = next(iter(MOUTHS.values()), None)
    save_dir = os.path.join(WORKING_DIR, "live")
    chat = ollama_chat(RESULTS["best_brain"]) if RESULTS["best_brain"] else None
    app = make_app(ears, mouth, llm=LLM, chat=chat, password=LIVE_PASSWORD or "wimb", save_dir=save_dir)
    serve(app, 7860)
    url = open_tunnel(7860)
    RESULTS["live"] = {"opened": time.strftime("%H:%M UTC"), "minutes": LIVE_MINUTES, "link": bool(url),
                       "brain": RESULTS["best_brain"], "ears": RESULTS["best_asr"],
                       "mouth": mouth.backend if mouth else None, "mouth_device": mouth.device if mouth else None}
    print("Page:", "ready" if url else "no public link (both tunnels failed)", RESULTS["live"])
    if url:
        notify(url)
    else:
        failed("live page", "no tunnel")
    save_results()
    end = time.time() + 60 * LIVE_MINUTES
    while time.time() < end and url:
        time.sleep(60)
        RESULTS["live"].update(saved=len(os.listdir(save_dir)) if os.path.isdir(save_dir) else 0)
        save_results()
    calls = sorted(os.listdir(save_dir)) if os.path.isdir(save_dir) else []
    RESULTS["live"].update(saved=len(calls), calls=len(calls))
    print("Page closed.", RESULTS["live"])
    save_results()

In [ ]:
#@title 12. Summary (saved to voicebot_results.json)
def summary_lines():
    lines = []
    for name, r in RESULTS["text_test"].items():
        a = r["all"]
        speed = f", {r['model_seconds_avg']} s per model call" if r.get("model_seconds_avg") else ""
        lines.append(f"Test calls, typed, {name}: {a['passed']}/{a['total']}{speed}")
    if RESULTS["best_brain"]:
        lines.append(f"Best brain: {RESULTS['best_brain']}; for live calls (fast, within one test call): "
                     f"{RESULTS.get('live_brain') or 'none fast enough'}")
    for name, r in RESULTS["asr"]["bench"].items():
        if isinstance(r, dict):
            lines.append(f"Speech to text, {name}: " + "; ".join(
                f"{k} WER {v['wer']}%" for k, v in r.items() if isinstance(v, dict) and "wer" in v))
    for label, r in RESULTS["spoken_test"].items():
        if isinstance(r, dict) and "all" in r:
            lines.append(f"Test calls, spoken over a phone line ({label}): {r['all']['passed']}/{r['all']['total']}; "
                         f"lines heard exactly: {r['lines_heard_exactly']}")
    for s in RESULTS["tts"]["samples"]:
        lines.append(f"Voice {s['backend']} {s['lang']}: {s['seconds']} s to make {s['audio_seconds']} s of speech")
    ok = lambda kind: [t for t in RESULTS[kind]["tried"] if t["ok"]]
    for kind, info in (("asr", ASR_INFO), ("tts", TTS_INFO)):
        for t in RESULTS[kind]["tried"]:
            i = info.get(t["backend"], {})
            flag = "" if i.get("business", True) else " (NOT for business use)"
            lines.append(f"{kind.upper()} {t['backend']} ({i.get('model')}, {i.get('licence')}{flag}): "
                         + (f"ok, on {t.get('device')}" if t["ok"] else t["why"]))
    if RESULTS.get("best_asr"):
        lines.append(f"Speech to text used for the spoken test and live calls: {RESULTS['best_asr']}")
    if MODE == "live":
        lines.append(f"Live page: {RESULTS['live'].get('calls', 0)} calls, {RESULTS['live'].get('saved', 0)} saved")
    return lines


RESULTS["summary"] = summary_lines()
print("\n".join(RESULTS["summary"]))
print("\nErrors:", RESULTS["errors"] or "none")
save_results()